1. 共用參數設定


In [ ]:
'''
Measure the elapsed time of every code cell, including cells that raise errors.
記錄每個 Code Cell 的執行時間，即使發生錯誤也顯示已耗費的時間。
'''
from time import perf_counter as _cell_perf_counter
from IPython import get_ipython as _cell_get_ipython

_CELL_TIMER_SHELL = _cell_get_ipython()
if _CELL_TIMER_SHELL is None:
    raise RuntimeError('請在 Jupyter Notebook 中執行本教材。')
_CELL_TIMER_STATE = {'started': _cell_perf_counter()}

def _cell_timer_begin(_info):
    _CELL_TIMER_STATE['started'] = _cell_perf_counter()

def _cell_timer_end(result):
    started = _CELL_TIMER_STATE.pop('started', None)
    if started is None:
        return
    minutes, seconds = divmod(_cell_perf_counter() - started, 60)
    elapsed = f'{int(minutes)} 分 {seconds:05.2f} 秒' if minutes else f'{seconds:.2f} 秒'
    status = '執行失敗' if result.error_before_exec or result.error_in_exec else '執行完成'
    print(f'⏱ 本格{status}，耗時：{elapsed}')

'''
Replace existing timers when the first cell is executed again.
重新執行第一格時，先取消舊計時器，避免重複顯示耗時。
'''
for _event_name, _callback in globals().get('_CELL_TIMER_CALLBACKS', ()):
    try:
        _CELL_TIMER_SHELL.events.unregister(_event_name, _callback)
    except ValueError:
        pass
_CELL_TIMER_CALLBACKS = (
    ('pre_run_cell', _cell_timer_begin),
    ('post_run_cell', _cell_timer_end),
)
for _event_name, _callback in _CELL_TIMER_CALLBACKS:
    _CELL_TIMER_SHELL.events.register(_event_name, _callback)

'''
Import standard-library tools for editable paths and local service settings.
匯入設定路徑與本機服務參數所需的 Python 內建工具。
'''
import os
from pathlib import Path
ROOT = Path('/kaggle/working')
COMFY = ROOT / 'ComfyUI-HyperFlowH3'

'''
Store large downloaded GGUF weights in scratch storage.
設定 GGUF 模型權重的暫存磁碟位置，減少工作空間占用。
'''
STORE = Path('/tmp/hyperflow_h3_models')
PORT = 8194  # 本機可用 Port
COMFY_REF = 'v0.45.0'

'''
Set the Hugging Face cache directory before any model downloads.
在下載模型之前指定 Hugging Face 快取目錄，避免大型檔案占滿工作空間。
'''
HF_CACHE_DIR = Path('/tmp/hyperflow_h3_hf')
os.environ['HF_HOME'] = str(HF_CACHE_DIR)
os.environ['HF_HUB_CACHE'] = str(HF_CACHE_DIR / 'hub')
os.environ['HF_XET_CHUNK_CACHE_SIZE_BYTES'] = '0'

'''
Select the HyperFlow GGUF quantization. Q8 is memory-intensive on T4.
選擇 HyperFlow 的 GGUF 精度；Q8 對 T4 記憶體要求較高。
'''
QUANT = 'Q8_0'  # 'Q3_K_M', 'Q4_K_S', 'Q4_K_M', 'Q5_K_M', 'Q6_K', 'Q8_0'
DEFAULT_QUANT = QUANT

'''
Set GPU 0 memory budget in GiB; 14.5 is aggressive on a 15 GiB T4.
設定 GPU 0 的記憶體預算（GiB）；15 GiB T4 設為 14.5 剩餘緩衝很小，OOM 可調低。
'''
GPU0_MODEL_GIB = 14.5  # 14.5、14.0、13.5
GPU1_MODEL_GIB = 14.5  # 14.5、14.0、13.5

'''
Select the text-encoder weight placement strategy.
設定文字編碼器權重管理：dynamic 分批調度較節省 GPU 記憶體；streaming 使用時整體載入；resident 盡量常駐 GPU 記憶體。
'''
TEXT_MODE = 'dynamic'  # 'dynamic', 'streaming', 'resident'
TEXT_GPU = 'cuda:1'  # 'cuda:1'（預設）、'cuda:0'（需留意 GPU 記憶體用量）
NODE_CACHE_MODE = 'classic'  # 'classic', 'none'

'''
Choose the HyperFlow text encoder; select a format actually supported by the installed loader.
選擇 HyperFlow 文字編碼器；需搭配實際支援的權重與 Loader。
'''
ENCODER_VARIANT = '8B_FP8'  # 可選 4B_FP8、4B_BF16、8B_NVFP4、8B_W4A8、8B_INT8、8B_FP8、8B_BF16、32B_NVFP4、32B_INT8、32B_BF16
DEFAULT_ENCODER_VARIANT = ENCODER_VARIANT
WEIGHTS_IN = 'disk'  # 'disk', 'auto', 'ram'

'''
Select the local directory used by R2V image uploads.
指定 R2V 參考圖片的儲存目錄；圖片檔案由該 Task Cell 管理。
'''
R2V_REF_DIR = ROOT / 'hyperflow_h3_r2v_references'
REMOTE_PORT = 8765  # 例如 8765、8766，範圍 1024～65535
REMOTE_SERVICE_ID = '06_HyperFlow_H3_Flash_影片生成_Kaggle'

'''
Place optional API job data inside the configured workspace.
將遠端工作與暫存資料存放在第一格設定的工作空間底下。
'''
REMOTE_ROOT = ROOT / 'remote_api_jobs' / REMOTE_SERVICE_ID
REMOTE_QUEUE_CAPACITY = 32  # 建議 12～32，依課堂人數調整
REMOTE_UPLOAD_CONCURRENCY = 2  # 建議 1～2

'''
Choose output retention and storage safety limits.
設定遠端成果保留期限、空間限制及主機磁碟最低剩餘量。
'''
REMOTE_RESULT_TTL_HOURS = 24
REMOTE_RESULT_LIMIT_GIB = 8
REMOTE_MIN_FREE_GIB = 2
REMOTE_STALL_HOURS = 2
GGUF_REPO = 'VideoRebirthAI/hyperflow-h3-flash'

'''
Validate user-editable local ports and remote queue limits.
確認教師設定的本機 Port 與遠端佇列限制有效，避免服務啟動後才失敗。
'''
assert 1024 <= REMOTE_PORT <= 65535, 'REMOTE_PORT 必須介於 1024～65535。'
assert 1 <= REMOTE_QUEUE_CAPACITY <= 1024, 'FIFO 佇列容量設定無效。'
assert 1 <= REMOTE_UPLOAD_CONCURRENCY <= 8, '上傳並發限制設定無效。'
assert REMOTE_RESULT_TTL_HOURS > 0 and REMOTE_RESULT_LIMIT_GIB > 0 and REMOTE_MIN_FREE_GIB >= 0
assert REMOTE_STALL_HOURS > 0
assert 1024 <= PORT <= 65535 and PORT != REMOTE_PORT
print('✓ 第 1 節完成：共用參數已設定，請繼續執行環境安裝。')


2. 安裝 HyperFlow H3 Flash 環境


In [ ]:
'''
Import the libraries needed for this section.
匯入本段程式所需的函式庫。
'''
import os
import subprocess
import sys
from pathlib import Path
import torch
assert torch.cuda.is_available() and torch.cuda.device_count() == 2, '請啟用 Kaggle T4 x2。'
for i in range(2):
    print(f'GPU {i}: {torch.cuda.get_device_name(i)} / {torch.cuda.get_device_properties(i).total_memory / 1024**3:.2f} GiB')

'''
Install the official HyperFlow loader and the 8B/4B projection runtime.
安裝官方 HyperFlow Loader 與 8B／4B 投影節點。
'''
repositories = {
    COMFY: 'https://github.com/Comfy-Org/ComfyUI.git',
    COMFY / 'custom_nodes/ComfyUI-GGUF': 'https://github.com/city96/ComfyUI-GGUF.git',
    COMFY / 'custom_nodes/ComfyUI-HyperFlow-Flash': 'https://github.com/Video-Rebirth/ComfyUI-HyperFlow-Flash.git',
    COMFY / 'custom_nodes/ComfyUI-ClipProj': 'https://github.com/nicolab28/ComfyUI-ClipProj.git',
}
H3_RESTART_REQUIRED = False
for directory, url in repositories.items():
    assert not directory.exists() or (directory / '.git').is_dir(), f'不是 Git 倉庫：{directory}'
    if (directory / '.git').is_dir():
        dirty = subprocess.check_output(['git', '-C', str(directory), 'status', '--porcelain'], text=True).strip()
        if not dirty and directory != COMFY:
            before = subprocess.check_output(['git', '-C', str(directory), 'rev-parse', 'HEAD'], text=True).strip()
            subprocess.run(['git', '-C', str(directory), 'pull', '--ff-only', '-q'], check=True)
            H3_RESTART_REQUIRED |= before != subprocess.check_output(['git', '-C', str(directory), 'rev-parse', 'HEAD'], text=True).strip()

        elif dirty:
            '''
            Preserve local changes for manual inspection.
            保留本機修改供使用者自行檢查。
            '''
            print(f'本機修改未覆寫：{directory.name}')

    else:
        directory.parent.mkdir(parents=True, exist_ok=True)

        '''
        Pin only the core ComfyUI checkout to a release tag.
        只有 ComfyUI 核心固定於正式版標籤。
        '''
        options = ['--branch', COMFY_REF] if directory == COMFY else []
        subprocess.run(['git', 'clone', '--depth', '1', *options, url, str(directory)], check=True)
        H3_RESTART_REQUIRED = True

'''
Fetch the selected release on every run.
每次執行都確認指定的 ComfyUI 正式版。
'''
subprocess.run(['git', '-C', str(COMFY), 'fetch', '--depth', '1', 'origin', 'tag', COMFY_REF], check=True)
previous_core = subprocess.check_output(['git', '-C', str(COMFY), 'rev-parse', 'HEAD'], text=True).strip()
subprocess.run(['git', '-C', str(COMFY), 'checkout', '-q', '--detach', COMFY_REF], check=True)

'''
Restart if an existing process is using an older core checkout.
如果執行中的 ComfyUI 核心版本過舊，標記為需要重新啟動。
'''
H3_RESTART_REQUIRED |= previous_core != subprocess.check_output(['git', '-C', str(COMFY), 'rev-parse', 'HEAD'], text=True).strip()
constraint = ROOT / 'hyperflow_torch_constraints.txt'
constraint.write_text(f'torch=={torch.__version__}\n', encoding='utf-8')

'''
Install only the dependencies declared by ComfyUI and HyperFlow.
僅安裝 ComfyUI 與 HyperFlow 官方宣告的相依套件。
'''
requirements = [
    COMFY / 'requirements.txt',
    COMFY / 'custom_nodes/ComfyUI-GGUF/requirements.txt',
    COMFY / 'custom_nodes/ComfyUI-HyperFlow-Flash/requirements.txt',
]
for requirement in requirements:
    '''
    Skip an optional file that is not present in this release.
    如果某版沒有相依套件檔案，就安全略過。
    '''
    if requirement.is_file():
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-c', str(constraint), '-r', str(requirement)], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-c', str(constraint),
                'huggingface_hub', 'hf_xet', 'requests', 'websocket-client', 'tqdm', 'ipywidgets>=8,<9'], check=True)

'''
Display installed source revisions for reproducibility.
顯示各程式庫的實際版本，方便問題追蹤。
'''
for directory in repositories:
    revision = subprocess.check_output(['git', '-C', str(directory), 'rev-parse', '--short', 'HEAD'], text=True).strip()
    print(f'{directory.name}: {revision}')

print('✓ 第 2 節完成：HyperFlow H3 Flash / ComfyUI v0.45.0')


3. 下載 HyperFlow 模型與輔助權重


In [ ]:
'''
Check disk capacity before downloading gigabyte-scale files.
下載大型權重前先檢查可用磁碟空間。
'''
import shutil
os.environ['HF_HOME'] = str(HF_CACHE_DIR)
os.environ['HF_HUB_CACHE'] = str(HF_CACHE_DIR / 'hub')

'''
Disable the optional extra Xet chunk cache.
停用不必要的 Xet 分塊快取以節省空間。
'''
os.environ['HF_XET_CHUNK_CACHE_SIZE_BYTES'] = '0'
from huggingface_hub import hf_hub_download

'''
Choose the generation task before model download; select the same task for generation.
在下載模型前先選擇任務；生成 Cell 會沿用此設定。 Options / 可選值: 't2v' 文字生成、'i2v' 起始圖片、'flf2v' 首尾圖片、'r2v' 多張參考圖片。
'''
TASK = globals().get('TASK', 't2v')  # 't2v', 'i2v', 'l2v', 'flf2v', 'r2v'
assert TASK in ('t2v', 'i2v', 'l2v', 'flf2v', 'r2v'), f'不支援的 TASK：{TASK}'

'''
Select the compressed FL2VA or the separate Ref2VA diffusion architecture.
一般影片使用 FL2VA，多圖參考 R2V 則使用獨立的 Ref2VA。
'''
MODEL_FAMILY = 'ref2va' if TASK == 'r2v' else 'fl2va'
task_models = globals().get('TASK_CONFIGS', {}).get(TASK, {})
ENCODER_SIZE = ENCODER_VARIANT.split('_', 1)[0]

'''
Map published GGUF sizes for the two independent diffusion architectures (decimal GB).
記錄 FL2VA 與 Ref2VA 兩套獨立模型的公開權重容量（十進位 GB）。
'''
MODEL_SIZES_GB = {
    'fl2va': {'Q3_K_M': 7.22, 'Q4_K_S': 8.01, 'Q4_K_M': 9.05,
              'Q5_K_M': 10.3, 'Q6_K': 11.6, 'Q8_0': 14.9},
    'ref2va': {'Q3_K_M': 7.4, 'Q4_K_S': 8.14, 'Q4_K_M': 9.22,
               'Q5_K_M': 10.45, 'Q6_K': 11.76, 'Q8_0': 15.16},
}
QUANT_SIZES_GB = MODEL_SIZES_GB[MODEL_FAMILY]
assert QUANT in QUANT_SIZES_GB, f'GPU 0 不支援 {QUANT}。'

'''
Obtain the specific compressed HyperFlow checkpoint, not the old 31B GGUF.
下載真正的 HyperFlow 壓縮模型，而不是先前 31B 的 GGUF。
'''
DIT_NAME = f'minimax_h3_{MODEL_FAMILY}_hyperflow_flash_14b-{QUANT}.gguf'
ENCODERS = {
    '32B_NVFP4': ('Comfy-Org/MiniMax-H3', 'qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors', 15.7),
    '32B_INT8': ('Comfy-Org/MiniMax-H3', 'qwen3vl_32b_minimax_h3_int8_convrot.safetensors', 27.1),
    '32B_BF16': ('Comfy-Org/MiniMax-H3', 'qwen3vl_32b_minimax_h3_bf16.safetensors', 51.5),
    '8B_NVFP4': ('Comfy-Org/Qwen3-VL', 'qwen3vl_8b_nvfp4.safetensors', 6.31),
    '8B_W4A8': ('Comfy-Org/Qwen3-VL', 'qwen3vl_8b_w4a8.safetensors', 6.31),
    '8B_INT8': ('Comfy-Org/Qwen3-VL', 'qwen3vl_8b_int8_convrot.safetensors', 9.35),
    '8B_FP8': ('Comfy-Org/Qwen3-VL', 'qwen3vl_8b_fp8_scaled.safetensors', 10.6),
    '8B_BF16': ('Comfy-Org/Qwen3-VL', 'qwen3vl_8b_bf16.safetensors', 17.5),
    '4B_FP8': ('Comfy-Org/Qwen3-VL', 'qwen3vl_4b_fp8_scaled.safetensors', 5.24),
    '4B_BF16': ('Comfy-Org/Qwen3-VL', 'qwen3vl_4b_bf16.safetensors', 8.88),
}

'''
Stop before downloading an unsupported GPU 1 model variant.
若 GPU 1 模型選項不存在，下載前就明確停止。
'''
assert ENCODER_VARIANT in ENCODERS, f'GPU 1 不支援 {ENCODER_VARIANT}；可選：{list(ENCODERS)}'
ENCODER_REPO, encoder_name, ENCODER_GB = ENCODERS[ENCODER_VARIANT]
ENCODER_FILE = f'text_encoders/{encoder_name}'

'''
Select the matching v3.1 projection only for smaller encoders.
只有小型 Encoder 需要對應的 v3.1 ClipProj 投影矩陣。
'''
PROJECTION_FILE = f'mmh3-{ENCODER_SIZE.lower()}-ClipProj-v3.1-mlp.safetensors' if ENCODER_SIZE != '32B' else None
TEXT_LOADER_NODE = 'ClipProjDeviceLoader' if ENCODER_SIZE == '32B' else 'ClipProjLoader'
print(f'GPU 0：HyperFlow {QUANT}（{QUANT_SIZES_GB[QUANT]} GB）；GPU 1：{ENCODER_VARIANT}（{ENCODER_GB} GB）；/tmp 剩餘 {shutil.disk_usage("/tmp").free / 1024**3:.1f} GiB')

'''
Download exactly one task-specific GGUF plus its encoder and two VAEs.
只下載所選任務的 FL2VA 或 Ref2VA GGUF，以及 Encoder 與兩個 VAE。
'''
models = [
    (GGUF_REPO, f'{MODEL_FAMILY}_gguf/{DIT_NAME}', 'diffusion_models'),
    (ENCODER_REPO, ENCODER_FILE, 'text_encoders'),
    ('Comfy-Org/MiniMax-H3', 'vae/minimax_h3_video_vae_fp16.safetensors', 'vae'),
    ('Comfy-Org/MiniMax-H3', 'vae/minimax_h3_audio_vae_fp32.safetensors', 'vae'),
]
if PROJECTION_FILE:
    models.append(('NicoLab28/ClipProj-MiniMax-H3', PROJECTION_FILE, 'clip_projections'))

STORE.mkdir(parents=True, exist_ok=True)

'''
Estimate the additional space needed for all selected model weights (decimal GB).
估算本次所選全部模型權重還需要多少暫存空間（十進位 GB）。
'''
model_sizes = [QUANT_SIZES_GB[QUANT], ENCODER_GB, 5.21, 0.605]
if PROJECTION_FILE:
    model_sizes.append(0.577 if ENCODER_SIZE == '8B' else 0.481)

missing_gb = sum(size for (_, filename, folder), size in zip(models, model_sizes)
                 if not (STORE / folder / filename).is_file())

'''
Read free space on the actual filesystem containing the model directory.
取得真正儲存模型權重的檔案系統剩餘空間。
'''
free_gb = shutil.disk_usage(STORE).free / 1e9
assert free_gb >= missing_gb * 1.05, (
    f'/tmp 空間不足：尚需約 {missing_gb:.1f} GB，剩餘 {free_gb:.1f} GB；'
    '請清除暫存模型，或改用較小的 GPU 0／GPU 1 權重。'
)
print(f'下載容量檢查：尚需約 {missing_gb:.1f} GB；剩餘 {free_gb:.1f} GB。')

'''
Track newly linked files so a running service can rescan dropdowns.
追蹤新增的模型連結，確保已啟動的服務更新選單。
'''
for repo, filename, folder in models:
    print('模型：', Path(filename).name, flush=True)
    source = Path(hf_hub_download(repo_id=repo, filename=filename, local_dir=STORE / folder))

    '''
    Make the model visible to ComfyUI without copying it.
    利用符號連結讓 ComfyUI 辨識模型，避免複製權重。
    '''
    target = COMFY / 'models' / folder / source.name
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.is_symlink() and (not target.exists() or not target.samefile(source)):
        target.unlink()
    if not target.exists():
        target.symlink_to(source.resolve())
        H3_RESTART_REQUIRED = True

    '''
    Reject unexpected same-name files to avoid silent model substitution.
    遇到同名但內容不同的權重檔案時，避免默默載入錯誤檔案。
    '''
    assert target.samefile(source), f'模型檔案衝突：{target}'

H3_MODEL_CHOICE = (MODEL_FAMILY, QUANT, ENCODER_VARIANT)
print(f'✓ 第 3 節完成：{TASK.upper()} / {MODEL_FAMILY} / GPU 0 {QUANT}；GPU 1 {ENCODER_VARIANT}（{TEXT_LOADER_NODE}）；Video/Audio VAE 已就緒。')


4. GPU 0／1 與 CPU Offload 設定


In [ ]:
'''
Require the teacher settings cell before applying ComfyUI GPU configuration.
套用 ComfyUI GPU 設定前，確認已執行教師共用設定 Cell。
'''
assert all(name in globals() for name in ('GPU0_MODEL_GIB', 'GPU1_MODEL_GIB', 'TEXT_MODE', 'TEXT_GPU', 'NODE_CACHE_MODE')), '請先執行「教師共用設定」Cell。'
gpu0_capacity = torch.cuda.get_device_properties(0).total_memory / 1024**3
gpu1_capacity = torch.cuda.get_device_properties(1).total_memory / 1024**3

'''
Reject physically impossible budgets and unsupported load modes.
避免設定不可能的 GPU 預算或不支援的載入模式。
'''
assert 0 < GPU0_MODEL_GIB < gpu0_capacity and 0 < GPU1_MODEL_GIB < gpu1_capacity
assert WEIGHTS_IN in ('auto', 'disk', 'ram')
assert TEXT_MODE in ('dynamic', 'streaming', 'resident')

'''
Reserve GPU 0 VRAM for activation tensors and decoder workloads.
保留 GPU 0 的部分 VRAM 給中間 Tensor 與解碼運算。
'''
GPU0_RESERVE_GIB = round(gpu0_capacity - GPU0_MODEL_GIB, 3)

assert NODE_CACHE_MODE in ('classic', 'none')
H3_DESIRED_CONFIG = (GPU0_RESERVE_GIB, WEIGHTS_IN, TEXT_GPU, TEXT_MODE, NODE_CACHE_MODE)
print(f'✓ 第 4 節完成：GPU 0 目標 {GPU0_MODEL_GIB:.1f} GiB、保留 {GPU0_RESERVE_GIB:.1f} GiB；GPU 1 使用 {TEXT_MODE}，參考值 {GPU1_MODEL_GIB:.1f} GiB（非硬上限）；offload={WEIGHTS_IN}')


5. 啟動 ComfyUI 與節點相容性檢查


In [ ]:
'''
Import the libraries needed for this section.
匯入本段程式所需的函式庫。
'''
import requests
import time
LOG = ROOT / 'hyperflow_h3_comfyui.log'
BASE = f'http://127.0.0.1:{PORT}'

'''
Test the ComfyUI API without exposing any public tunnel.
不開放公開網址，直接測試本機 ComfyUI API。
'''
def ready():
    '''
    Handle normal startup connection errors without crashing.
    安全處理服務啟動期間尚未開放連線的情況。
    '''
    try:
        return requests.get(f'{BASE}/system_stats', timeout=3).ok

    except requests.RequestException:
        return False

'''
Read a node's enumerated dropdown options from the live API.
從服務目前註冊的節點規格讀取下拉式選單。
'''
def choices(node_name, input_name):
    '''
    Inspect the requested node's input declaration.
    讀取指定節點的輸入規格。
    '''
    groups = NODE_INFO[node_name].get('input', {})
    for kind in ('required', 'optional'):
        spec = groups.get(kind, {}).get(input_name)

        '''
        Return the dropdown list only when its schema is enumerated.
        只有明確的選單才回傳可選值。
        '''
        if isinstance(spec, (list, tuple)) and spec and isinstance(spec[0], list):
            return spec[0]
        if spec is not None:
            return None
    raise RuntimeError(f'{node_name}.{input_name} 欄位不存在。')

need_restart = globals().get('H3_RESTART_REQUIRED', False) or globals().get('H3_ACTIVE_CONFIG') != H3_DESIRED_CONFIG
if ready() and not need_restart:
    need_restart = globals().get('H3_ACTIVE_MODEL') != H3_MODEL_CHOICE

'''
Shut down only a child process owned by this notebook kernel.
只允許關閉目前 Notebook Kernel 自己管理的服務行程。
'''
if ready() and need_restart:
    '''
    Avoid killing a model service that belongs to a different session.
    避免終止其他 Notebook 或 Session 的模型服務。
    '''
    assert 'H3_PROC' in globals() and H3_PROC.poll() is None, f'埠 {PORT} 已被其他行程使用；請重新啟動 Kaggle Session。'
    H3_PROC.terminate()
    try:
        H3_PROC.wait(timeout=20)

    except subprocess.TimeoutExpired:
        H3_PROC.kill()

        '''
        Wait for the kernel to release process resources.
        等待系統完全回收該行程資源。
        '''
        H3_PROC.wait(timeout=10)

if not ready():
    command = [sys.executable, 'main.py', '--listen', '127.0.0.1', '--port', str(PORT),
               '--enable-dynamic-vram', '--fast-disk', '--reserve-vram', str(GPU0_RESERVE_GIB),
               '--cache-classic' if NODE_CACHE_MODE == 'classic' else '--cache-none',
               '--disable-pinned-memory']

    '''
    Store complete startup warnings in a diagnostic file.
    將完整的啟動警告保存於診斷檔案。
    '''
    with LOG.open('w', encoding='utf-8') as stream:
        H3_PROC = subprocess.Popen(command, cwd=COMFY, stdout=stream, stderr=subprocess.STDOUT)
    for attempt in range(180):
        if H3_PROC.poll() is not None:
            raise RuntimeError(LOG.read_text(errors='replace')[-6500:])
        if ready():
            break
        time.sleep(2)
    assert ready(), LOG.read_text(errors='replace')[-6500:]

H3_ACTIVE_CONFIG = H3_DESIRED_CONFIG

'''
Retrieve registered node types and parameter schemas.
取得已註冊的節點類型與參數規格。
'''
NODE_INFO = requests.get(f'{BASE}/object_info', timeout=90).json()
REQUIRED_NODES = (TEXT_LOADER_NODE, 'HyperFlowLoader', 'HyperFlowSigmas',
                  'MiniMaxH3ImageToVideo', 'MiniMaxH3ReferenceToVideo', 'VAELoader', 'BasicGuider', 'KSamplerSelect',
                  'RandomNoise', 'SamplerCustomAdvanced', 'VAEDecode', 'VAEDecodeAudio',
                  'CreateVideo', 'SaveVideo', 'LoadImage')

missing = [name for name in REQUIRED_NODES if name not in NODE_INFO]

'''
Show actionable errors for incompatible or incomplete plugin installs.
外掛版本不相容時顯示可追蹤的具體缺失。
'''
assert not missing, f'ComfyUI 缺少節點 {missing}。請檢查 {LOG}。'
assert DIT_NAME in (choices('HyperFlowLoader', 'unet_name') or []), f'HyperFlow Loader 未辨識模型：{DIT_NAME}'
assert Path(ENCODER_FILE).name in (choices(TEXT_LOADER_NODE, 'clip_name') or []), '找不到 Qwen3-VL Encoder。'

'''
Validate the projection only when selecting a smaller encoder.
只有選擇小型 Encoder 時才確認 ClipProj 投影矩陣。
'''
if PROJECTION_FILE:
    assert PROJECTION_FILE in (choices(TEXT_LOADER_NODE, 'projection') or []), '找不到 ClipProj v3.1 投影矩陣。'

assert TEXT_GPU in (choices(TEXT_LOADER_NODE, 'device') or []), f'{TEXT_LOADER_NODE} 不支援 {TEXT_GPU}。'
assert TEXT_MODE in (choices(TEXT_LOADER_NODE, 'mode') or []), 'Encoder Loader 不支援指定的 Offload 模式。'

'''
Require the native MiniMax text-encoder architecture when using 32B.
選擇原生 32B 時確認節點有 MiniMax 的文字編碼器型別。
'''
if ENCODER_SIZE == '32B':
    '''
    Avoid loading the 32B weights under an unrelated encoder type.
    避免將 32B 權重當成其他架構載入。
    '''
    assert 'minimax' in (choices(TEXT_LOADER_NODE, 'type') or []), '缺少 MiniMax 原生 Encoder 型別。'

for name in ('minimax_h3_video_vae_fp16.safetensors', 'minimax_h3_audio_vae_fp32.safetensors'):
    assert name in (choices('VAELoader', 'vae_name') or []), f'找不到 VAE：{name}'

assert 'steps' in NODE_INFO['HyperFlowSigmas']['input'].get('optional', {}), 'HyperFlow Sigmas 缺少 steps 欄位。'

'''
Verify the canonical Euler sampler remains available.
確認官方 HyperFlow 所需的 Euler Sampler 已存在。
'''
assert 'euler' in (choices('KSamplerSelect', 'sampler_name') or []), 'Sampler 不支援 euler。'
H3_ACTIVE_MODEL = H3_MODEL_CHOICE
H3_RESTART_REQUIRED = False

'''
Confirm the server is ready for generation.
確認 HyperFlow 服務已可接受生成工作。
'''
print(f'✓ 第 5 節完成：GPU 0 {QUANT} / GPU 1 {ENCODER_VARIANT} / Disk Offload。')


6. T2V：Prompt 與生成參數


In [ ]:
'''
Describe the text-to-video scene and synchronized sound.
設定文生影片的場景與同步音效描述。
'''
T2V_PROMPT = '''
A cinematic close-up of a red panda in a misty forest. The red panda slowly turns its head.
Natural fur details, gentle camera movement, soft morning light. Sound: birds chirping and light wind.
'''.strip()
T2V_WIDTH, T2V_HEIGHT = 384, 224
T2V_SECONDS = 10

'''
Select the official HyperFlow 4-, 6-, or 8-step schedule.
選擇官方 HyperFlow 的 4、6 或 8 steps 排程。
'''
T2V_STEPS = 6
T2V_SEED = 42
T2V_OUTPUT = 'hyperflow_h3_flash_t2v.mp4'

'''
Preserve configurations for every task independently.
分開保存各項任務的設定，避免不同 Prompt 互相覆寫。
'''
TASK_CONFIGS = globals().get('TASK_CONFIGS', {})
TASK_CONFIGS['t2v'] = dict(prompt=T2V_PROMPT, width=T2V_WIDTH, height=T2V_HEIGHT,
                           seconds=T2V_SECONDS, steps=T2V_STEPS, seed=T2V_SEED,
                           encoder_variant=ENCODER_VARIANT, quant=QUANT,
                           output_name=T2V_OUTPUT, first_image='', last_image='')

'''
Select the current generation task and its model settings.
選擇目前生成模式及此模式的模型設定；尚未執行生成。
'''
TASK = 't2v'

print(f'✓ 第 6 節完成：T2V | {T2V_WIDTH}×{T2V_HEIGHT} | {T2V_SECONDS} 秒 | {T2V_STEPS} steps | {QUANT} | GPU 1 {ENCODER_VARIANT} | {T2V_OUTPUT}')


7. I2V：Prompt、圖片與生成參數


In [ ]:
'''
Describe how the uploaded first frame should animate.
描述上傳的第一張圖片要如何轉化為動態影片。
'''
I2V_PROMPT = '''
Animate the subject in the reference image with subtle natural motion and smooth cinematic camera movement.
Keep the person's or object's identity, clothing and environment consistent. Include realistic ambient sound.
'''.strip()
I2V_FIRST_IMAGE = str(ROOT / 'first_frame.png')
I2V_WIDTH, I2V_HEIGHT = 384, 224

'''
Configure output duration for image-to-video.
設定圖生影片的目標秒數。
'''
I2V_SECONDS = 10
I2V_STEPS = 6
I2V_SEED = 42

'''
Set an independent image-to-video output filename.
指定圖生影片自己的輸出檔名。
'''
I2V_OUTPUT = 'hyperflow_h3_flash_i2v.mp4'
TASK_CONFIGS = globals().get('TASK_CONFIGS', {})
TASK_CONFIGS['i2v'] = dict(prompt=I2V_PROMPT, width=I2V_WIDTH, height=I2V_HEIGHT,
                           seconds=I2V_SECONDS, steps=I2V_STEPS, seed=I2V_SEED,
                           encoder_variant=ENCODER_VARIANT, quant=QUANT,
                           output_name=I2V_OUTPUT, first_image=I2V_FIRST_IMAGE, last_image='')

'''
Confirm the configuration cell has run without attempting to open the image yet.
顯示設定完成；實際圖片檔案會在生成時再驗證。
'''
TASK = 'i2v'

print(f'✓ 第 7 節完成：I2V | {I2V_WIDTH}×{I2V_HEIGHT} | {I2V_SECONDS} 秒 | {QUANT} | GPU 1 {ENCODER_VARIANT} | 圖片：{I2V_FIRST_IMAGE}')


8. L2VA：最後影格生成影片


In [ ]:
'''
Describe how the uploaded last frame should animate.
描述上傳的最後一張圖片要如何轉化為動態影片。
'''
L2V_PROMPT = '''
Animate the subject in the reference image with subtle natural motion and smooth cinematic camera movement.
Keep the person's or object's identity, clothing and environment consistent. Include realistic ambient sound.
'''.strip()
L2V_LAST_IMAGE = str(ROOT / 'last_frame.png')
L2V_WIDTH, L2V_HEIGHT = 384, 224

'''
Configure output duration for image-to-video.
設定最後影格生成影片的目標秒數。
'''
L2V_SECONDS = 10
L2V_STEPS = 6
L2V_SEED = 42

'''
Set an independent image-to-video output filename.
指定圖生影片自己的輸出檔名。
'''
L2V_OUTPUT = 'hyperflow_h3_flash_l2v.mp4'
TASK_CONFIGS = globals().get('TASK_CONFIGS', {})
TASK_CONFIGS['l2v'] = dict(prompt=L2V_PROMPT, width=L2V_WIDTH, height=L2V_HEIGHT,
                           seconds=L2V_SECONDS, steps=L2V_STEPS, seed=L2V_SEED,
                           encoder_variant=ENCODER_VARIANT, quant=QUANT,
                           output_name=L2V_OUTPUT, first_image='', last_image=L2V_LAST_IMAGE)

'''
Confirm the configuration cell has run without attempting to open the image yet.
顯示設定完成；實際圖片檔案會在生成時再驗證。
'''
TASK = 'l2v'

print(f'✓ 第 8 節完成：L2V | {L2V_WIDTH}×{L2V_HEIGHT} | {L2V_SECONDS} 秒 | {QUANT} | GPU 1 {ENCODER_VARIANT} | 圖片：{L2V_LAST_IMAGE}')


9. FLF2V：Prompt、首尾圖片與生成參數


In [ ]:
'''
Describe a video that connects its first and last reference frames.
描述從第一張圖片過渡到最後一張圖片的影片內容。
'''
FLF2V_PROMPT = '''
Create a smooth cinematic transition from the first reference frame to the last reference frame.
Maintain consistent subject details and lighting, with natural intermediate motion and synchronized ambient sound.
'''.strip()
FLF2V_FIRST_IMAGE = str(ROOT / 'first_frame.png')
FLF2V_LAST_IMAGE = str(ROOT / 'last_frame.png')

'''
Set output dimensions for first-and-last-frame generation.
設定首尾影格模式的輸出解析度。
'''
FLF2V_WIDTH, FLF2V_HEIGHT = 384, 224
FLF2V_SECONDS = 10
FLF2V_STEPS = 6

'''
Set this task’s seed; use -1 to randomize each generation.
設定此任務的種子；-1 代表每次生成都重新抽取。
'''
FLF2V_SEED = 42
FLF2V_OUTPUT = 'hyperflow_h3_flash_flf2v.mp4'
TASK_CONFIGS = globals().get('TASK_CONFIGS', {})

'''
Register both conditioning images and all parameters.
登記兩張參考圖片與全部生成參數。
'''
TASK_CONFIGS['flf2v'] = dict(prompt=FLF2V_PROMPT, width=FLF2V_WIDTH, height=FLF2V_HEIGHT,
                             seconds=FLF2V_SECONDS, steps=FLF2V_STEPS, seed=FLF2V_SEED,
                             encoder_variant=ENCODER_VARIANT, quant=QUANT,
                             output_name=FLF2V_OUTPUT, first_image=FLF2V_FIRST_IMAGE,
                             last_image=FLF2V_LAST_IMAGE)

'''
Select the current generation task and its model settings.
選擇目前生成模式及此模式的模型設定；尚未執行生成。
'''
TASK = 'flf2v'

print(f'✓ 第 9 節完成：FLF2V | {FLF2V_WIDTH}×{FLF2V_HEIGHT} | {FLF2V_SECONDS} 秒 | {QUANT} | GPU 1 {ENCODER_VARIANT} | 參考圖 2 張')


10. R2V：Prompt、參考圖片與生成參數


In [ ]:
'''
Describe the primary picture; the prompt stays valid when selecting only one image.
描述第一張參考圖；即使只選一張圖片，Prompt 仍能正常使用。
'''
R2V_PROMPT = '''
<Picture 1> defines the main subject's appearance.
Create a cinematic scene with natural motion and ambient sound.
Keep the main subject consistent, and use additional pictures as visual references when supplied.
'''.strip()
import ipywidgets as widgets
from IPython.display import display
from PIL import Image
import io
import tempfile
import shutil
from html import escape
R2V_FORMATS = {'.png': 'PNG', '.jpg': 'JPEG', '.jpeg': 'JPEG', '.webp': 'WEBP'}

'''
Preserve the most recently confirmed batch across cell reruns in this Kaggle session.
在同一個 Kaggle Session 中重跑 Cell 時，沿用最後一次成功選取的圖片組合。
'''
R2V_REF_DIR.mkdir(parents=True, exist_ok=True)
R2V_ACTIVE_FILES = sorted(p for p in R2V_REF_DIR.iterdir()
                          if p.is_file() and p.suffix.lower() in R2V_FORMATS
                          and len(p.stem) == 3 and p.stem.isdigit())

assert len(R2V_ACTIVE_FILES) <= 9 and all(
    path.stem == f'{index:03d}' for index, path in enumerate(R2V_ACTIVE_FILES, 1)
), 'R2V 圖片資料夾編號不連續；請重新選取整組圖片。'

'''
Keep the public reference list synchronized with the files actually on disk.
讓公開的參考圖片清單與磁碟上的實際檔案保持一致。
'''
R2V_REF_IMAGES = [str(path) for path in R2V_ACTIVE_FILES]
R2V_REF_VIDEOS = []  # Example: ['/kaggle/working/clip.mp4']
R2V_REF_AUDIOS = []  # Example: ['/kaggle/working/voice.wav']

'''
Use soundtrack of a reference video if the uploaded video has audio.
若參考影片含音軌，選擇是否同時使用該音軌作為聲音條件。
'''
R2V_VIDEO_AUDIO = False  # True, False
R2V_REF_IMAGE_SIZE = 'match'
R2V_WIDTH, R2V_HEIGHT = 384, 224

'''
Set the approximate video duration in seconds.
設定目標影片秒數，實際影格數會依 H3 規則對齊。
'''
R2V_SECONDS = 10
R2V_STEPS = 6

'''
Encoder sizes and precisions; B means one billion parameters. Prefer native 32B for picture references; it needs GPU 1 offload on T4.
Encoder 大小與精度；B 代表十億個參數，32B=320 億、8B=80 億、4B=40 億。 Setting / 設定值  | Model / 模型                 | Precision / 權重精度 32B_NVFP4       | 原生 Qwen3-VL-32B            | NVFP4（4-bit 浮點） 32B_INT8        | 原生 Qwen3-VL-32B            | INT8 ConvRot（8-bit 整數） 32B_BF16        | 原生 Qwen3-VL-32B            | BF16（16-bit 浮點） 8B_NVFP4        | Qwen3-VL-8B + ClipProj      | NVFP4（4-bit 浮點） 8B_W4A8         | Qwen3-VL-8B + ClipProj      | W4A8（4-bit 權重／8-bit activation） 8B_INT8         | Qwen3-VL-8B + ClipProj      | INT8 ConvRot（8-bit 整數） 8B_FP8          | Qwen3-VL-8B + ClipProj      | FP8（8-bit 浮點） 8B_BF16         | Qwen3-VL-8B + ClipProj      | BF16（16-bit 浮點） 4B_FP8          | Qwen3-VL-4B + ClipProj      | FP8（8-bit 浮點） 4B_BF16         | Qwen3-VL-4B + ClipProj      | BF16（16-bit 浮點） 多圖參考預設使用原生 32B；在 T4 上須透過 GPU 1 卸載管理。 8B／4B + ClipProj 可節省資源，但多圖參考的影像條件品質尚需實測。
'''
R2V_SEED = 42
R2V_OUTPUT = 'hyperflow_h3_flash_r2v.mp4'

'''
Retain the other three task configurations in the same notebook session.
保留同一個 Notebook Session 內其他三種生成模式的參數。
'''
TASK_CONFIGS = globals().get('TASK_CONFIGS', {})
TASK_CONFIGS['r2v'] = dict(prompt=R2V_PROMPT, width=R2V_WIDTH, height=R2V_HEIGHT,
                           seconds=R2V_SECONDS, steps=R2V_STEPS, seed=R2V_SEED,
                           encoder_variant=ENCODER_VARIANT, quant=QUANT,
                           output_name=R2V_OUTPUT, first_image='', last_image='',
                           ref_images=R2V_REF_IMAGES, ref_image_size=R2V_REF_IMAGE_SIZE,
                           ref_videos=R2V_REF_VIDEOS, ref_audios=R2V_REF_AUDIOS,
                           video_audio=R2V_VIDEO_AUDIO)

R2V_UPLOAD_STATUS = widgets.HTML()

'''
Keep the active file picker replaceable so the next selection cannot append old uploads.
讓選檔元件可重新建立，確保下次多選不會累積上一批檔案。
'''
R2V_UPLOAD_PANEL = widgets.VBox()
R2V_UPLOAD_HELP = widgets.HTML(value=(
    '<b>R2V 參考圖片（1～9 張）</b>：一次選取所有圖片；每次重新選取會完整取代上一批，'
    '從 001 重新編號，並刪除不再使用的舊圖片。'
))

'''
Apply one complete browser selection as a new image set, never merging it with an earlier batch.
將瀏覽器本次選取視為完整的新圖片組合，絕不與上一批合併。
'''
def on_r2v_upload(change):
    '''
    Update the global image list when a valid new batch is committed.
    新圖片批次確認有效後，更新全域參考圖片清單。
    '''
    global R2V_REF_IMAGES
    uploaded = change['new']
    if not uploaded:
        return
    staging = None
    message = ''

    '''
    Try to stage and atomically replace the entire selected batch.
    嘗試建立暫存圖片組合，再以整批替換方式更新正式資料夾。
    '''
    try:
        assert 1 <= len(uploaded) <= 9, '請一次選取 1～9 張圖片。'
        staging = Path(tempfile.mkdtemp(prefix='.hyperflow_r2v_new_', dir=ROOT))
        mapping = []

        '''
        Keep the browser's selected file order for <Picture 1> onward.
        按照瀏覽器回傳的選取順序，對應 <Picture 1> 等圖片標籤。
        '''
        for index, item in enumerate(uploaded, 1):
            original = item['name'].replace('\\', '/').split('/')[-1]
            extension = Path(original).suffix.lower()
            assert extension in R2V_FORMATS, f'不支援的圖片格式：{original}'
            content = bytes(item['content'])

            '''
            Reject an empty upload before opening the image decoder.
            在開啟圖片解碼器前先拒絕空白檔案。
            '''
            assert content, f'圖片內容為空：{original}'
            with Image.open(io.BytesIO(content)) as picture:
                assert picture.format == R2V_FORMATS[extension], f'圖片內容格式不符：{original}'
                picture.verify()
            filename = f'{index:03d}{extension}'
            (staging / filename).write_bytes(content)
            mapping.append(f'&lt;Picture {index}&gt; = {escape(filename)} ← {escape(original)}')
        backup = ROOT / f'.hyperflow_r2v_old_{__import__("uuid").uuid4().hex}'
        R2V_REF_DIR.rename(backup)
        try:
            '''
            Publish the verified batch as the sole active image directory.
            將已驗證的新批次整批設為唯一有效的圖片目錄。
            '''
            staging.rename(R2V_REF_DIR)

        except Exception:
            backup.rename(R2V_REF_DIR)
            raise
        shutil.rmtree(backup, ignore_errors=True)

        '''
        Read only the currently numbered files in exact Picture order.
        只讀取現在有效的圖片，並依 Picture 順序排列。
        '''
        R2V_REF_IMAGES = [str(path) for path in sorted(R2V_REF_DIR.iterdir()) if path.is_file()]
        TASK_CONFIGS['r2v']['ref_images'] = list(R2V_REF_IMAGES)
        message = '<b>已套用最新 ' + str(len(uploaded)) + ' 張圖片：</b><br>' + '<br>'.join(mapping)

    except Exception as error:
        message = '<b>未更新參考圖片：</b>' + escape(str(error))

    finally:
        '''
        Always clean up any unused staging directory and reset the browser picker.
        無論成功或失敗，均清除暫存目錄並重新建立瀏覽器選檔元件。
        '''
        if staging is not None and staging.exists():
            shutil.rmtree(staging, ignore_errors=True)
        R2V_UPLOAD_STATUS.value = message
        R2V_UPLOAD_PANEL.children = (new_r2v_uploader(), R2V_UPLOAD_STATUS)

'''
Create a new browser file picker and subscribe to its batch-upload event.
建立新的瀏覽器選檔元件，並監聽每批圖片上傳完成事件。
'''
def new_r2v_uploader():
    '''
    Offer one browser picker for multiple image files.
    提供可一次選取多張圖片的瀏覽器元件。
    '''
    uploader = widgets.FileUpload(accept='.png,.jpg,.jpeg,.webp', multiple=True,
                                   description='多選參考圖片')
    uploader.observe(on_r2v_upload, names='value')
    return uploader

R2V_UPLOAD_STATUS.value = ('目前圖片：' + (', '.join(Path(path).name for path in R2V_REF_IMAGES)
                                          if R2V_REF_IMAGES else '尚未選擇'))

R2V_UPLOAD_PANEL.children = (new_r2v_uploader(), R2V_UPLOAD_STATUS)

'''
Render the upload widget directly in this R2V settings cell.
直接在 R2V 參數 Cell 顯示圖片上傳元件。
'''
display(widgets.VBox([R2V_UPLOAD_HELP, R2V_UPLOAD_PANEL]))
'''
Select the current generation task and its model settings.
選擇目前生成模式及此模式的模型設定；尚未執行生成。
'''
TASK = 'r2v'

print(f'✓ 第 10 節完成：R2V | {R2V_WIDTH}×{R2V_HEIGHT} | {len(R2V_REF_IMAGES)} 張參考圖 | {QUANT} | GPU 1 {ENCODER_VARIANT}。素材設定完成後請執行第 11 節 Generate。')


11. 生成影片（T2V／I2V／L2VA／FLF2V／R2V）


In [ ]:
'''
Serialize notebook generation with optional remote FIFO GPU jobs.
讓 Notebook 手動生成與選用的遠端 FIFO 共用 GPU 鎖，避免同時推論。
不必先啟動 ngrok 或任何 HTTP API。
'''
import threading
GENERATION_LOCK = globals().setdefault('GENERATION_LOCK', threading.Lock())
with GENERATION_LOCK:
    import json
    import shutil
    import uuid
    import math
    import websocket
    from tqdm.auto import tqdm
    TASK = globals().get('TASK', 't2v')
    assert TASK in globals().get('TASK_CONFIGS', {}), f'請先執行 {TASK.upper()} 的參數設定 Cell。'
    config = TASK_CONFIGS[TASK]

    '''
    Share sampling, decoding and video output logic across four task modes.
    四種生成模式共用採樣、解碼及影片輸出流程。
    '''
    MODE = TASK
    PROMPT = config['prompt']
    WIDTH, HEIGHT = config['width'], config['height']
    SECONDS, STEPS = config['seconds'], config['steps']
    SEED, OUTPUT_NAME = config['seed'], config['output_name']

    '''
    Resolve task-specific reference images without mixing modes.
    依據所選任務使用對應參考圖片，避免不同模式混用。
    '''
    FIRST_IMAGE, LAST_IMAGE = config['first_image'], config['last_image']
    REF_IMAGES = list(config.get('ref_images', []))
    REF_VIDEOS = list(config.get('ref_videos', []))
    REF_AUDIOS = list(config.get('ref_audios', []))
    VIDEO_AUDIO = bool(config.get('video_audio', False))
    REF_IMAGE_SIZE = config.get('ref_image_size', 'match')
    ENCODER_VARIANT = config['encoder_variant']

    '''
    Derive the encoder architecture needed by the selected loader.
    取得 Loader 所需的文字編碼器架構大小。
    '''
    ENCODER_SIZE = ENCODER_VARIANT.split('_', 1)[0]
    QUANT = config['quant']
    assert QUANT in QUANT_SIZES_GB and ENCODER_VARIANT in ENCODERS, f'不支援的 GPU 模型組合：{QUANT}/{ENCODER_VARIANT}'
    assert STEPS in (4, 6, 8), 'HyperFlow STEPS 僅支援 4、6、8。'

    '''
    Ensure edited GPU 0 memory budgets have been applied to the running service.
    確認調整過的 GPU 0 記憶體預算已套用到目前服務。
    '''
    assert globals().get('H3_ACTIVE_CONFIG') == H3_DESIRED_CONFIG, 'GPU 0 預算已變更，請重新執行第 4 節啟動 ComfyUI。'
    assert globals().get('H3_MODEL_CHOICE') == (MODEL_FAMILY, QUANT, ENCODER_VARIANT) and globals().get('H3_ACTIVE_MODEL') == (MODEL_FAMILY, QUANT, ENCODER_VARIANT), (
        f'目前選擇 GPU 0 {QUANT}／GPU 1 {ENCODER_VARIANT}，但尚未完成下載或服務預檢。請重跑第 2 節與第 4 節。'
    )
    assert MODEL_FAMILY == ('ref2va' if TASK == 'r2v' else 'fl2va'), '目前 GPU 0 模型架構與任務不一致，請重跑第 2、4 節。'
    assert DIT_NAME == f'minimax_h3_{MODEL_FAMILY}_hyperflow_flash_14b-{QUANT}.gguf', 'GPU 0 模型檔名不一致，請重跑第 2 節。'

    '''
    Reject a stale GPU 1 encoder that belongs to an earlier selection.
    避免切換 GPU 1 模型後仍使用前一次載入的 Encoder。
    '''
    assert Path(ENCODER_FILE).name == ENCODERS[ENCODER_VARIANT][1], 'GPU 1 模型檔名不一致，請重跑第 2 節。'
    LAST_OUTPUT = None
    assert ready(), 'ComfyUI 已中斷，請先重新執行第 3 節。'
    assert MODE in ('t2v', 'i2v', 'l2v', 'flf2v', 'r2v') and PROMPT.strip() and STEPS > 0 and math.isfinite(SECONDS) and SECONDS > 0, (
        '請確認生成模式、Prompt、STEPS 與 SECONDS 設定正確；SECONDS 必須為正數。'
    )

    '''
    Require positive dimensions aligned to 32, without an artificial resolution limit.
    寬高需為正數且符合 32 的倍數，不另外限制解析度或長寬比。
    '''
    assert WIDTH > 0 and HEIGHT > 0 and WIDTH % 32 == 0 and HEIGHT % 32 == 0, (
        'WIDTH 與 HEIGHT 必須為正數且是 32 的倍數。'
    )
    FRAMES = max(22, 17 * math.ceil((SECONDS * 24 - 5) / 17) + 5)
    assert FRAMES <= 3600, f'影格數 {FRAMES} 超出官方 H3 節點上限 3600，請縮短影片秒數。'
    import secrets
    RUN_SEED = secrets.randbelow(2**32) if SEED == -1 else int(SEED)

    '''
    Require a supported seed before building the inference workflow.
    在建立推論工作流程前，先檢查種子範圍。
    '''
    assert 0 <= RUN_SEED < 2**32, 'SEED 必須介於 0 至 4294967295，或設為 -1。'
    print(f'本次生成 seed：{RUN_SEED}')
    output_stem = Path(OUTPUT_NAME).stem
    run_name = f'{output_stem}_{QUANT}_{uuid.uuid4().hex[:8]}'
    text_inputs = {'clip_name': Path(ENCODER_FILE).name,
                   'type': 'minimax' if ENCODER_SIZE == '32B' else 'auto',
                   'device': TEXT_GPU, 'mode': TEXT_MODE}

    '''
    Add the learned projection only when a 4B/8B encoder is selected.
    只有使用 4B／8B Encoder 時才接入學習式投影矩陣。
    '''
    if PROJECTION_FILE:
        text_inputs['projection'] = PROJECTION_FILE
    graph = {
        '1': {'class_type': TEXT_LOADER_NODE, 'inputs': text_inputs},
        '2': {'class_type': 'HyperFlowLoader', 'inputs': {'unet_name': DIT_NAME, 'weights_in': WEIGHTS_IN}},
        '4': {'class_type': 'VAELoader', 'inputs': {'vae_name': 'minimax_h3_video_vae_fp16.safetensors'}},
        '5': {'class_type': 'VAELoader', 'inputs': {'vae_name': 'minimax_h3_audio_vae_fp32.safetensors'}},
        '6': {'class_type': ('MiniMaxH3ReferenceToVideo' if MODE == 'r2v' else 'MiniMaxH3ImageToVideo'), 'inputs': {
            'clip': ['1', 0], 'vae': ['4', 0], 'prompt': PROMPT,
            'width': WIDTH, 'height': HEIGHT, 'length': FRAMES}},
        '7': {'class_type': 'BasicGuider', 'inputs': {
            'model': ['2', 0], 'conditioning': ['6', 0]}},
        '8': {'class_type': 'KSamplerSelect', 'inputs': {'sampler_name': 'euler'}},
        '9': {'class_type': 'HyperFlowSigmas', 'inputs': {'model': ['2', 0], 'steps': STEPS}},
        '10': {'class_type': 'RandomNoise', 'inputs': {'noise_seed': RUN_SEED}},
        '11': {'class_type': 'SamplerCustomAdvanced', 'inputs': {
            'noise': ['10', 0], 'guider': ['7', 0],
            'sampler': ['8', 0], 'sigmas': ['9', 0], 'latent_image': ['6', 1]}},
        '12': {'class_type': 'VAEDecode', 'inputs': {'samples': ['11', 0], 'vae': ['4', 0]}},
        '13': {'class_type': 'VAEDecodeAudio', 'inputs': {'samples': ['11', 0], 'vae': ['5', 0]}},
        '14': {'class_type': 'CreateVideo', 'inputs': {
            'images': ['12', 0], 'audio': ['13', 0], 'fps': 24.0}},
        '15': {'class_type': 'SaveVideo', 'inputs': {
            'video': ['14', 0], 'filename_prefix': run_name, 'format': 'mp4'}},
    }
    if MODE == 'r2v':
        assert isinstance(REF_IMAGES, (list, tuple)) and 0 <= len(REF_IMAGES) <= 9, 'R2V_REF_IMAGES 最多 9 張；也可以改用影片或音訊。'
        assert all(Path(path).is_file() for path in REF_IMAGES), 'R2V 有不存在的圖片；請重新選擇。'

        '''
        Restrict the reference resize mode to the two official choices.
        參考圖尺寸選項只能填寫官方支援的 match 或 max。
        '''
        assert REF_IMAGE_SIZE in ('match', 'max'), 'R2V_REF_IMAGE_SIZE 只能是 match 或 max。'
        import re
        mentioned = {int(number) for number in re.findall(r'<Picture\s+(\d+)>', PROMPT, re.IGNORECASE)}
        assert mentioned <= set(range(1, len(REF_IMAGES) + 1)), 'Prompt 的 <Picture N> 必須對應已提供圖片。'
        graph['6']['inputs']['audio_vae'] = ['5', 0]

        '''
        Apply the documented image-sizing method for every reference.
        對所有參考圖套用官方定義的縮放模式。
        '''
        graph['6']['inputs']['ref_image_size'] = REF_IMAGE_SIZE
        for index, image_path in enumerate(REF_IMAGES):
            source = Path(image_path)
            assert source.is_file() and source.stat().st_size > 0 and source.suffix.lower() in ('.png', '.jpg', '.jpeg', '.webp'), f'無效的 R2V 參考圖片：{source}'
            node_id = str(20 + index)
            destination = COMFY / 'input' / f'r2v_{index + 1}_{uuid.uuid4().hex[:8]}{source.suffix.lower()}'

            '''
            Create the input directory if it does not exist.
            視需要建立 ComfyUI input 資料夾。
            '''
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, destination)
            graph[node_id] = {'class_type': 'LoadImage', 'inputs': {'image': destination.name}}
            graph['6']['inputs'][f'ref_images.ref_image_{index}'] = [node_id, 0]
    if MODE == 'r2v':
        assert len(REF_IMAGES) <= 9 and len(REF_VIDEOS) <= 3 and len(REF_AUDIOS) <= 3, 'R2V 最多 9 張圖片、3 段影片、3 段獨立音訊。'
        assert REF_IMAGES or REF_VIDEOS, 'R2V 至少需要一張參考圖片或一段參考影片；音訊可作為附加條件。'
        assert len(REF_IMAGES) + len(REF_VIDEOS) + len(REF_AUDIOS) <= 12, 'R2V 參考素材合計最多 12 個檔案。'
        tags = {'Picture': len(REF_IMAGES), 'Video': len(REF_VIDEOS),
                'Audio': len(REF_AUDIOS) + (len(REF_VIDEOS) if VIDEO_AUDIO else 0)}
        for label, count in tags.items():
            numbers = [int(n) for n in re.findall(r'<%s\s+(\d+)>' % label, PROMPT, re.I)]
            assert all(1 <= n <= count for n in numbers), f'<{label} N> 編號超出實際上傳數量：{numbers}。'
        for index, filename in enumerate(REF_VIDEOS):
            original = Path(filename)
            assert original.is_file() and original.suffix.lower() in ('.mp4', '.mov', '.webm'), f'參考影片不存在或格式不正確：{original}'
            destination = COMFY / 'input' / f'h3_ref_video_{uuid.uuid4().hex[:12]}{original.suffix.lower()}'
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(original, destination)
            node_id = str(120 + index * 2)
            parts_id = str(121 + index * 2)
            graph[node_id] = {'class_type': 'LoadVideo', 'inputs': {'file': destination.name}}
            graph[parts_id] = {'class_type': 'GetVideoComponents', 'inputs': {'video': [node_id, 0]}}
            graph['6']['inputs'][f'ref_videos.ref_video_{index}'] = [parts_id, 0]
            if VIDEO_AUDIO:
                graph['6']['inputs'][f'ref_video_audios.ref_video_audio_{index}'] = [parts_id, 1]
        for index, filename in enumerate(REF_AUDIOS):
            original = Path(filename)
            assert original.is_file() and original.suffix.lower() in ('.wav', '.mp3', '.flac', '.ogg', '.m4a'), f'參考音訊不存在或格式不正確：{original}'
            destination = COMFY / 'input' / f'h3_ref_audio_{uuid.uuid4().hex[:12]}{original.suffix.lower()}'
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(original, destination)
            node_id = str(140 + index)
            graph[node_id] = {'class_type': 'LoadAudio', 'inputs': {'audio': destination.name}}
            graph['6']['inputs'][f'ref_audios.ref_audio_{index}'] = [node_id, 0]

    '''
    Attach a verified local image to an H3 image-conditioning socket.
    將確認存在的本機圖片接到 H3 圖片條件輸入。
    '''
    for mode, image_path, node_id, input_name in [
        ('i2v', FIRST_IMAGE, '16', 'first_frame'),
        ('flf2v', FIRST_IMAGE, '16', 'first_frame'),
        ('flf2v', LAST_IMAGE, '17', 'last_frame'),
        ('l2v', LAST_IMAGE, '18', 'last_frame'),
    ]:

        '''
        Only attach images required by the active mode.
        只接入目前模式需要的圖片。
        '''
        if MODE == mode:
            source = Path(image_path)

            '''
            Stop before generation if a reference file is missing.
            若圖片不存在，在生成前立即停止。
            '''
            assert source.is_file(), f'找不到參考圖片：{source}'
            destination = COMFY / 'input' / f'{node_id}_{uuid.uuid4().hex[:8]}{source.suffix.lower()}'
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, destination)
            graph[node_id] = {'class_type': 'LoadImage', 'inputs': {'image': destination.name}}

            '''
            Connect the loaded image to H3 conditioning.
            將已載入圖片連到 H3 條件輸入。
            '''
            graph['6']['inputs'][input_name] = [node_id, 0]
    NODE_INFO = requests.get(f'{BASE}/object_info', timeout=90).json()
    assert TEXT_GPU in (choices(TEXT_LOADER_NODE, 'device') or []), 'Encoder Loader 不支援 GPU 1。'
    assert TEXT_MODE in (choices(TEXT_LOADER_NODE, 'mode') or []), 'Encoder Loader 不支援指定 Offload 模式。'

    '''
    Verify that the official HyperFlow Euler sampler is available.
    確認官方 HyperFlow Euler Sampler 可用。
    '''
    assert 'euler' in (choices('KSamplerSelect', 'sampler_name') or []), 'HyperFlow 必須使用 Euler Sampler。'
    assert 'HyperFlowSigmas' in NODE_INFO, '缺少 HyperFlow Sigma 節點。'
    if MODE in ('i2v', 'l2v', 'flf2v'):
        frame_inputs = NODE_INFO['MiniMaxH3ImageToVideo'].get('input', {}).get('optional', {})
        expected = ('first_frame', 'last_frame') if MODE == 'flf2v' else (('last_frame',) if MODE == 'l2v' else ('first_frame',))
        assert all(field in frame_inputs for field in expected), f'MiniMaxH3ImageToVideo 缺少 {expected} 輸入。'
    if MODE == 'r2v':
        required_nodes = {'LoadVideo', 'GetVideoComponents'} if REF_VIDEOS else set()
        if REF_AUDIOS:
            required_nodes.add('LoadAudio')
        missing_nodes = required_nodes - set(NODE_INFO)
        assert not missing_nodes, f'ComfyUI 缺少參考媒體節點：{sorted(missing_nodes)}，請更新 ComfyUI。'

    '''
    Validate that all graph dependency links target existing nodes.
    驗證所有 Workflow 連線的來源節點確實存在。
    '''
    for node_id, node in graph.items():
        '''
        Examine each declared input for possible node-reference links.
        檢查節點輸入中是否包含連到其它節點的參照。
        '''
        for value in node['inputs'].values():
            if isinstance(value, list) and len(value) == 2:
                assert str(value[0]) in graph, f'節點 {node_id} 連結到不存在的節點 {value[0]}。'
    for node_id, node in graph.items():
        schema = NODE_INFO[node['class_type']].get('input', {})
        accepted = set().union(*(set(schema.get(group, {})) for group in ('required', 'optional', 'hidden')))
        if node['class_type'] == 'MiniMaxH3ReferenceToVideo':
            assert 'ref_images' in accepted, 'MiniMaxH3ReferenceToVideo 不支援 ref_images；請更新 ComfyUI。'
            for family, paths in (('ref_videos', REF_VIDEOS), ('ref_audios', REF_AUDIOS), ('ref_video_audios', REF_VIDEOS if VIDEO_AUDIO else [])):
                assert not paths or family in accepted, f'目前 ComfyUI Ref2VA 不支援 {family}。'
            accepted.update(f'ref_images.ref_image_{index}' for index in range(len(REF_IMAGES)))
            accepted.update(f'ref_videos.ref_video_{index}' for index in range(len(REF_VIDEOS)))
            accepted.update(f'ref_video_audios.ref_video_audio_{index}' for index in range(len(REF_VIDEOS)) if VIDEO_AUDIO)
            accepted.update(f'ref_audios.ref_audio_{index}' for index in range(len(REF_AUDIOS)))
        unknown = set(node['inputs']) - accepted

        '''
        Fail early if a workflow input is no longer supported.
        當 Workflow 參數不再被節點支援時，立即停止並指出名稱。
        '''
        assert not unknown, f"節點 {node_id} ({node['class_type']}) 不支援的輸入：{sorted(unknown)}"
    print(f'{MODE.upper()} | {WIDTH}×{HEIGHT} | {FRAMES} frames | {STEPS} steps | GPU 0 {QUANT} | GPU 1 {ENCODER_VARIANT} | HyperFlow', flush=True)
    client_id = uuid.uuid4().hex
    ws = None

    '''
    Connect before queueing to avoid missing early progress messages.
    在送出任務前先連線，避免漏掉初期進度事件。
    '''
    try:
        ws = websocket.create_connection(f'ws://127.0.0.1:{PORT}/ws?clientId={client_id}', timeout=5)
        ws.settimeout(2)

    except (OSError, websocket.WebSocketException):
        ws = None

        '''
        Explain why no step percentage will be shown.
        說明為什麼目前不會出現步數百分比。
        '''
        print('即時步數事件不可用；將顯示階段與經過時間。', flush=True)
    response = requests.post(f'{BASE}/prompt', json={'prompt': graph, 'client_id': client_id}, timeout=120)
    if not response.ok:
        if ws is not None:
            ws.close()
        raise RuntimeError(response.text[:5000])
    prompt_id = response.json()['prompt_id']
    start = time.monotonic()

    '''
    Create a single notebook progress bar with real step events only.
    建立單一 Notebook 進度條，僅使用真實採樣步數更新。
    '''
    pbar = tqdm(total=STEPS, desc='排隊／載入模型', unit='step', leave=True)
    item = None
    last_history_check = 0.0
    phases = {'1': '文字編碼器', '2': '載入 HyperFlow GGUF', '6': '建立影片條件',
              '11': '採樣中', '12': '影片解碼', '13': '音訊解碼',
              '14': '合成影片', '15': '儲存影片'}

    '''
    Wait until ComfyUI reports a final history record for this task.
    持續等待 ComfyUI 回傳此任務的最終執行紀錄。
    '''
    try:
        while True:
            '''
            Measure real wall-clock time instead of estimating model progress.
            計算真實經過時間，不將時間誤當模型百分比。
            '''
            elapsed = time.monotonic() - start
            message = None
            if ws is not None:
                try:
                    message = ws.recv()

                except websocket.WebSocketTimeoutException:
                    pass

                except (OSError, websocket.WebSocketException):
                    '''
                    Release the unusable client connection.
                    關閉失效的 WebSocket 連線。
                    '''
                    ws.close()
                    ws = None
            if isinstance(message, str):
                event = json.loads(message)
                data = event.get('data') or {}

                '''
                Reject progress from another client or generation.
                忽略非本次任務的進度，避免步數混亂。
                '''
                if data.get('prompt_id') == prompt_id:
                    if event.get('type') == 'executing' and data.get('node'):
                        pbar.set_description(phases.get(str(data['node']), '執行中'), refresh=False)
                    if event.get('type') in ('progress', 'progress_state'):
                        progress = data
                        if event.get('type') == 'progress_state':
                            progress = data.get('nodes', {}).get('11', {})

                        '''
                        Ignore progress emitted by loading or decoding nodes.
                        忽略非採樣節點的進度，以免誤顯示百分比。
                        '''
                        if str(progress.get('node', progress.get('node_id'))) == '11':
                            value, maximum = progress.get('value'), progress.get('max')
                            if isinstance(value, (int, float)) and isinstance(maximum, (int, float)) and maximum > 0:
                                pbar.total = int(maximum)
                                pbar.n = max(0, min(int(value), pbar.total))
                                pbar.set_description('採樣中', refresh=False)

            '''
            Check the persistent history regardless of WebSocket availability.
            不論 WebSocket 是否正常，都定期檢查 ComfyUI 任務紀錄。
            '''
            if elapsed - last_history_check >= 4:
                last_history_check = elapsed
                if not ready():
                    '''
                    Preserve recent server logs for actionable diagnostics.
                    保留最近服務紀錄供後續除錯。
                    '''
                    raise RuntimeError('ComfyUI 服務中斷：\n' + LOG.read_text(errors='replace')[-4500:])
                item = requests.get(f'{BASE}/history/{prompt_id}', timeout=60).json().get(prompt_id)

                '''
                Stop polling once the job has a final record.
                任務產生最終紀錄後停止輪詢。
                '''
                if item:
                    errors = [m for m in item.get('status', {}).get('messages', []) if m[0] == 'execution_error']
                    if errors:
                        raise RuntimeError(str(errors[-1][1])[:5000])
                    break
            pbar.set_postfix_str(f'{elapsed / 60:.1f} 分鐘', refresh=False)
            pbar.refresh()
            if ws is None:
                time.sleep(1)
        pbar.n = pbar.total

        '''
        Label the status as completed.
        將顯示狀態設為完成。
        '''
        pbar.set_description('完成', refresh=False)
        pbar.set_postfix_str(f'{(time.monotonic() - start) / 60:.1f} 分鐘', refresh=False)
        pbar.refresh()

    finally:
        pbar.close()

        '''
        Close the WebSocket when it exists.
        若 WebSocket 仍存在，關閉它以釋放資源。
        '''
        if ws is not None:
            ws.close()
    files = [p for p in (COMFY / 'output').rglob('*') if p.is_file() and p.name.startswith(run_name)]
    files = [p for p in files if p.suffix.lower() in ('.mp4', '.webm', '.mov')]
    assert files, f'工作流程完成但未找到影片：{item.get("outputs", {})}'
    source_video = max(files, key=lambda path: path.stat().st_mtime)

    '''
    Save the user-visible output in Kaggle working storage.
    將成果儲存至 Kaggle working 目錄。
    '''
    OUTPUT_DIR = ROOT / 'hyperflow_h3_outputs'
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    output_base = OUTPUT_DIR / f'{Path(OUTPUT_NAME).stem}_{QUANT}{source_video.suffix}'
    output = output_base

    '''
    Number duplicate filenames deterministically.
    依序為重複檔名編號。
    '''
    for index in range(2, 1000):
        if not output.exists():
            '''
            Preserve the current unique output path.
            保留目前可用的輸出檔名。
            '''
            break
        output = output_base.with_name(f'{output_base.stem}_{index:03d}{output_base.suffix}')
    shutil.copy2(source_video, output)
    print(f'完成：{output}｜耗時 {(time.monotonic() - start) / 60:.1f} 分鐘')
    LAST_OUTPUT = output
    print('✓ 第 11 節完成：影片已生成；請執行第 12 節預覽與下載。')


12. 影片預覽、下載與 GPU 使用量


In [ ]:
'''
Display the most recently generated video and its download link.
顯示最近一次成功生成影片的預覽與下載連結。
'''
from IPython.display import Video, FileLink, display
assert globals().get('LAST_OUTPUT') is not None and LAST_OUTPUT.is_file(), '請先執行第 9 節生成影片。'
display(Video(str(LAST_OUTPUT), embed=True, width=480))

'''
Provide a clickable Kaggle file download link.
顯示 Kaggle 影片檔案下載連結。
'''
display(FileLink(str(LAST_OUTPUT)))
status = subprocess.run(
    ['nvidia-smi', '--query-gpu=index,name,memory.used,memory.total',
     '--format=csv,noheader,nounits'], capture_output=True, text=True, check=True
)
print('GPU 記憶體使用量（MiB）：')

'''
Display both devices in a readable form.
逐行顯示兩張 GPU 的使用狀態。
'''
print(status.stdout.strip())
for line in status.stdout.splitlines():
    fields = [part.strip() for part in line.split(',')]
    if len(fields) == 4 and fields[0] in ('0', '1') and fields[2].isdigit():
        used_gib = int(fields[2]) / 1024
        limit_gib = GPU0_MODEL_GIB if fields[0] == '0' else GPU1_MODEL_GIB
        if used_gib > limit_gib:
            print(f'提醒：GPU {fields[0]} 使用 {used_gib:.2f} GiB，超過目標／監控值 {limit_gib:.2f} GiB；非硬上限。')

print(f'✓ 第 12 節完成：影片預覽與下載連結已顯示；{LAST_OUTPUT.name}')


13. 擷取最後影格（選用）


In [ ]:
'''
Import the libraries needed for this section.
匯入本段程式所需的函式庫。
'''
import json
from IPython.display import FileLink, display
from PIL import Image
import shutil
assert shutil.which('ffmpeg') and shutil.which('ffprobe'), '找不到 ffmpeg 或 ffprobe；請先安裝 FFmpeg。'
assert globals().get('LAST_OUTPUT') is not None and LAST_OUTPUT.is_file(), '請先執行第 9 節生成影片，並確認第 10 節可正常預覽。'

'''
Build a PNG path next to the generated video.
在生成影片旁建立最後影格 PNG 檔案路徑。
'''
LAST_FRAME_PATH = LAST_OUTPUT.with_name(LAST_OUTPUT.stem + '_last_frame.png')
probe = subprocess.run(
    ['ffprobe', '-v', 'error', '-count_frames', '-select_streams', 'v:0',
     '-show_entries', 'stream=nb_read_frames', '-of', 'json', str(LAST_OUTPUT)],
    capture_output=True, text=True, check=True,
)
last_index = int(json.loads(probe.stdout)['streams'][0]['nb_read_frames']) - 1

'''
Save the exact last frame as a PNG for the next I2V clip.
將最後一個影格儲存成 PNG，供下一段 I2V 使用。
'''
subprocess.run(
    ['ffmpeg', '-y', '-loglevel', 'error', '-i', str(LAST_OUTPUT),
     '-vf', rf'select=eq(n\,{last_index})', '-frames:v', '1', str(LAST_FRAME_PATH)],
    check=True,
)
display(Image.open(LAST_FRAME_PATH))
display(FileLink(str(LAST_FRAME_PATH)))
print(f"下一段：請在第 7 節 I2V 設定 I2V_FIRST_IMAGE = '{LAST_FRAME_PATH}'")

'''
Confirm completion of the extracted-frame export.
確認最後影格已成功輸出。
'''
print(f'✓ 第 13 節完成：已擷取最後影格；{LAST_FRAME_PATH.name}')


14. （選用）啟動 ngrok 與 FIFO 遠端 API


In [ ]:
'''
Require the first configuration cell before enabling the optional remote API.
選用遠端 API 啟動前，必須先執行第一格共用參數設定。
'''
assert all(name in globals() for name in ('REMOTE_PORT', 'REMOTE_SERVICE_ID', 'REMOTE_ROOT')), '請先執行第 1 節共用參數設定。'
import zlib
import math
import re

import queue
import copy
import secrets
import io
import json
import os
import shutil
import subprocess
import tarfile
import tempfile
import threading
import time
import urllib.request
import uuid
import zipfile
from email import policy
from email.parser import BytesParser
from getpass import getpass
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
from urllib.parse import urlsplit, parse_qs

REMOTE_KIND = 'video'
REMOTE_FIELDS = ['task', 'prompt', 'seed', 'width', 'height', 'seconds', 'steps', 'ref_image_size', 'quant', 'video_audio']
REMOTE_IS_LTX = False
REMOTE_SCRIPT = zlib.decompress(b'x\xda\xd5}ks\x14W\x96\xe0w~EFGLTU[\x94\x91\xc0/\xa6\xe9\r\x0c\x02k\xda\x06\x16\x81{g\x19B\x91\xaaJI\xd9\xd4\xab+\xab\x10\x8cV\x11\x12 $\x1eB\xb2\xcd\xc3\x80x\x99\x87\xd5<\x04\xd8\x06\x84\x00\xf1a\xfe\xc9\x8e\xb2\x1e\x9f\xfc\x17\xf6\xbc\xee\xcd\x9bYYB\xb4{vc{b\xb0*+\xeb\xdes\xcf=\xf7\xbc\xcf\xb9\x89Db]O\xbeT,W\xac\xca\x90c\xe5\xdc\xfe\xb2]v\x1d\xcf*8N\xd6\xc9Z\x03\xc52|\xe1z\x96\xe7d*n\xb1\x90^\xe7\x9f}\xecO\xdc\xad\xcd=\xac-<\xab\xcf\x9f\xf1_\xcf\xd4N\x8d5\xe7\xc6\xeaWN\xf8\x93o\xe0\xa3\xbf\xf4\xe0?\xc7\x8e\xadK\xc0\xc8.\x8f\xfc\x17\xafXP\x7f{C\xd5\x8a\x9bS\x9f\xaaU7\xab\xfe\xce\xdb\x95!\xf5\xf7\xb0\xd3\xef\x153\x87\x9c\xca\xba\x81r1oU\xfe\x9a\xcd\xa7\xedj\xa5h\xc9\xf7\xf8`\x1dM\xb1\xdfs\x08\xf2\x8a\xed\x1d\xb2\x9c#\xa5\x9c\x9bq+\xb9\xa3\x00o\x0e \x86\x15\xb8\x05\xfa>_\xcc:\xb9\xf5\xd9\xe2p!W\xb4\xb3\xc1rj?\xbd\xad\x9f\x9f\xaf\xcd\xdf\xf2\xaf\x9fYY<\xd3x}\xaa\xfe\xe0f\xfd\xf1X\xed\xec\xa4\xbfp\x05\x16e\xed\xdb\xda\xfb\xa7__\x9f\xad\xff\xb0\xb4\xf2\xf6\x9a\xb5s\xcf~k\x83\xc5o\xd6\x1e\xdd\x86\xa7\xb5\xeb\xcfk?\x9e\x81%[\xbbK8\xa6g}h\xf93\x8f\x9b\xe3\x8b\xfe\xd8\xeb\xcdV\xa2\xd2u8\xd1a%\\\xfe\xcf@n\x80\xff(\xc3\x7f\x14\x9ep\nk\x8b5\x98+\xf6\xdb9/\x99J\x0f:\x95d\x02\x9f\xe2\x9b8@\x8aW\xbb\xd7\x01\xd8q9zy\xb8\xee\x84g\x95\xca\xcea\xb7X\xf5`\xe5\xce\x11\'S\xc5\xaf2\xc5\xc2\x80;X-\xdb\x08\x95\x95qr\xb9\xf4\xba\xc6\xc2\x98?s\x11wl|q\xe5\xd5+\xff\xcce\xff\xc5O\xfe\x8d\x17\x8d[g\x9b\xe3\xe7`\xb9\x8d\xf9G\xb0nk\x1b\xbc\xad\xa0\xb3=\xcf\x01\x9c\x13\x90\x80\xcd\x18(\xfb\xb6\xed\xde\xb5\xa3gg/@;2\x9a\xea\xb0\x06\x12\x8d\x07g\xfc\x89)\x1e\xd8\x1a\xc1w\xd2\xd5R\xc9)\'S\xa3\x16R\xca\xcc\xf1\xda\x85\xc5\xf0\\\t\xb5D\xaf\x98;\xec\xc4\xae\xd2-d\x9d\x92\x03\xff\x14\x80\x90\x9cJ\xc5-\x0cz@\x913\x17\xfd\xe5K\xe6\x9a`\x86\xfa\xf4|\x1d`\xa0y\xd4B\x18!\x80h\x13\xe6\x03\xf8\xe1 \xcf\xdd;d\x97aV;\x0fdT\x18\xec\xb0\xb2N\xa6\x98\x85\xbf,\xbb\x90\xb5\x0e\xbbY\xa7h\x15\xab\x95R\xb5b\xe5\x8a\x83n\xc6\xb23\xe5\xa2\xe7\xc1!\xa9\x96\x99\x00\x91\xca\x10\xa2\xabW\xeb\xf3\x0b\xf5\xf37jS\xb3HZp(&\x9e"\x99\x9d\xbbU\x9b\xbf\xfd\x9fc\xe3\x8d\x1f\x81r^\xfb3\xa7\xfd7O\xeb\xa7&\x1b\xaf\x17\xfd\xc9\xa5\xda\xb3q8P\n\xd8\xafvo\xef\x16P\xc3\x94^\x82\x13Q\x82\xf5W\x8a\xe5\x80\xbcC\x98\x92\xbd^y\x83\xa4]\xbf\xba\xe0\x9f\x9af\xbc\x18\x08\xb7\x1a\xb7o!u\xef\xa1\xd1\xd4\xa4{\xf6\xee\xfej\xcf>\x98\x96Qu \xc1\x93%\x0e\x86A\xd0\xb3e\xec\xc2a\xdb3\xf7\xe2\xee%\\gx/xy\xfe\x93%\xff\xb1\xde\x8b?\xf7l\xdf\xf7E\x87\xf5Ew\xcf\xce/\xcc\t\x87\xddle(q\xb0C?\x18r\xdc\xc1\xa1\x16\x08p\x99\xeb\xbd\x92\x93q\x07`\x1f\xb2\x8a\xc8q\x9f\xd4\xf6\xc1\x08\xd5B\xa5\rH\xf5\x1f\xbfE\x02\x9c\x9a\xe4-\xa9=\xbak\xd0Io7\xd0\xc6\xf6\xde\x0e\xabw_\xf7\x9e^\x038\xe0\x1a\xc5B\xd63\xc1\xf3*N\xc9S\xd0m-\x95\xe0\x00\xb6\xc2\xe7\x013%\xd8\x84z\x06\xdc\x9cS\xb0\xf3N;\xe0\xe6\x17\xfcG\xb3\x00\x9c\xd0\xc5\xfd\xf3\xfe\xect\x00\\\xf7\xf6\x0ek\xf7\xfe}{\xf6\xef\xeb\xdb\xb5\xf5\xab\xee\x10|N\xd6\x04\x8e\xa7\xeb\xc3\xa9\x14\x88\xfax\x85\x00,;\x03N\xd9)d\x1c`\xb0\xf6 \xb0\xffa\xb72\x04\xbf\xb6\xf2\xee\x11D\xa5\xd0\xf5\xca\xf2\xb5\xdawWLh\x99\xc6\xfc\'\xd3\xb5\xc9Sp\xd8\x1ac\xc7\xfd\xb9\x8b@\xd1\xc00\x9b\xe3o\xfd\x89\xe9\x95\xc5i\x7f\xf6,\x1f\x82\xda\x8b\x17\xf0\xb2Z\xc8\x8e\x9e\xbd\xbd\xfb\xfaz\xbe\xda\xba\xb3\xbb\xc3\xfar\xab\xfa\xdbX\xce\x80[\xf6*}\x04\x91\xb9\xaa\x9c\x1d<5\xd8b\xb1 \xb8\xcf\x17\xbd\n\xac(\x03\\\x02\x9e\xd0\xaf\xcay\xd8\x80\x92\x9b\xa9T\xcb\x8az\x91\\P\xba\xed\xed\xfa:\xbd\x0e\xfe\x01\x9e}_\xb8\xe3\xdc\x98\xbf|veq\xac\xf6\xf0\x16\x1c`\xff\xf4\r\xff\xce\x15d\xe7\xcbg\x91u\xd1\xfa\xea\xcfN\xf8\xb3Sj){\xbbw0\xf0H,9\xd7\xab$\x19V\xe6\x90\x80\\\x06\xd7\x03\xfex\xe0`Jxy/\x81a\x15Ih\xd89c\x0f\x88\xd3xL\xcc\x15\xf8\xd7\xce\x15\x0b\x8eeW\xb3nQq;\x00\x07\xcf6qRD:\xb3\x91\xa9If{\xcd\x1b?7\xe6O\x9b\xd0}\xdd\xb3\xbd{w[\xe8xB\x05\x1d\xbe\xbfu\xff\xf6\x9e\xf6\xef\x13(\xfa}\x1a\x9b\x7f\x01?\xe8/\x16s\xa1\x1f\xd0\xe0\xfc\x13\xf8\xc5\x0e\x10\x1d\x8eB\xc1\xb6\xa1b\x11\x0et\xb0\xf2\xb2\xe3\xb9\xff\xae\x08\xceJ\x82J\x90\x19\xb2`\x93\xf2\xf6\x91Tt\xb7\x02bc"\xb4\xf8\xed\xda\xd4E|\xddbv\xd3\xb8|\xa1>{\x92\xc9\xafe\xb7\xfaz{\xfeg@o\x91\xcd\xea\x03H\x1c\x14\xbf4lH\x00\xa3\x0e\xd0\t\xe2\xc89b\xc3\x06V\x9c#\x15\x0b\xa0\x07\x88\xcb\xd6a\xd0\x9el\x90N\xac\xb6\xa0\xe2$\xa2+$\x8ca\x17\x97\xaf\xa3\n\xf5\xf0\x96\xc9\x98\x99\xfcxx\x94\x92\xb5\x8b\x93\xfe\xa3K\xf5\x17\xf3(+.\xcf\xd7OM\xc1o\xd4*\xbawm\x03\t\xb1\xb7\xef\xeb\xad{{\xb6\xee2Y\xa8\xc0\xd2\'\xb0\xa8C\xb2\xdd)\xbb"W\x15\xb4v93\xe4V\x1c>\x16\xa2\xf1\xf5\x1f\r\xb3xT\x97\x9c\xb2\xa2;\xebK\xfahi\x9d/\x02#+C\xfe\x9d\x1f\xfd\'3QH\x05\xdb\x11\xc0\xd3\x1epk@|\x1f\xe0\xba3u`C\x0b+\xd5\x90|q\x14\x14\x88\x1d\xb9\xe2\xb0\xb5s\xe7\xfe\x1d\xd6_\xab\xb08\xf7\xdfm}\x92I\xc5C,x@\xb3\xb0D8\xfbCN\xe6\x90\xa7$a\xed\xd1\x1d\xc6v\xa0\xdbE\x86lN\xce\xf8g/\xd6n\xbe\x06j\x01\x0e\xb6\xf2\xea.k}\xac\x1e\xd6\xee\xffP\xbbq\x17\xf8Bc\xf2\x17\xb5\xb6\xff\xbe?\x8c{\x02*`K\x7f\xc1\x13^-x\xa0\xfb\x80\xda\nk\x08\x01\x8d|\xd9\xb3\xfa\x1d\x00\x1e\x8ew\xa5\xe2\x80\xa8E\xe2\x1ft\n\x8e"\x14V"@\x82\x83:U;\x7f\xbe6u\x1f8j\xed\xfc\xe3\xda\xcc/\xb0\x00\x13`\x90\x11\xf5\xf9\xa7\x11\xa5\x8d\xe1\x03%\x81\xfe\xa0-\xe8\xed\xdb\xf99\xf1\x96(\x05\xc1[\xf2\xa8\x17U8s\x1e\xa2I\xc6\x02s\xbe___\x19\xa1!G?\x1c\x89\x8c3\x9a\x08\x8b\xea\xe2\x00H\x18\x17\x18\\\x80l\xcf\x1d\xcc\xdb\x96\x07\xdb\x93\xad\xe6\x00\x05\xc8\xba\x81\xc2\x8eO\x888Y\xf8\xbev\xf1\xa5\xf1>B\xb0\t\xd4\xa6\x8f\xe1\xff?\xb5@X[\xbd4B\xed\xdc\xb7\x86\xc6$Kf\xa1\r\x8bIn\xea\xb0>\xee\xb0>\x05\x854\x11\x8c\xc5_\xc3\\\xbc\xb8`\xdc@\x05\xed\x86\r\x83\x1dq\xb2.n\x19k\xfcy\'_,\x1f\xb5\xfa\xabY`\x13\x9e5d\xc3Q\xeaw\x1cP9\x80P]T\xbc\x8a\xb4\xdar\xb5P\xc0=\x04P\x0e\xbb\x19\x90\xf0@A\x8d\xfb\xd3\x8d\xfbok\x17~a\x15[Fl\xcc\x7f_;\xf9\xbc\xf9\xe0|\xf3\xe6l}\xe1\x12\xea\xe1\xa4\x0c\xf8SOXg\xab\xcdM\x03\xb1FV\x17\xd1\xbe\xbf\xd8\xd8\xb7u\xdb\xbe\x9e\xaf\xbbE\x9dM\xa4\xac-[,x\xba\xbd\xbb\xb7go\xf7vy\x0c\x18\xe0I\xf5\\\x8d\x85\xd3\xb5\xab\xbf\x00\x91\x83\xa6\xde\x9c\x9c\xae]|\xc2\xcaz\xfd\xe1Ck\x93\x05\x06\x90\x7f\xe1\x86\x7f\xe6\x82\xb5\xad\x98\x1f8\xba\xbf\'@\xce\x1e\xb04P\xff\x0e\x88\x14\x08\x1cL:Z}f\x08\xf8y!L\xe5\xc3\xa0 *\xbbK\xd4\xa0\xb23\x88G\x14\xf4W\xd8u\xd0\xc1f\xae\x86\x08y\xf9,\xc0\xe5\xbf\x9dh\xde\x9cB#b\xe1,\x9c\x00\xb6\xb8@\xc2)\x88\x84\x1c\x1b\xf3\xaf\xfc\x93\xa7\xdf\x89$\xd4\xa6\xbf\xec\xdb\xf6\xc5\xee\x9em\xdd\x8c\xa3$?\xda\xb1\xf5\xab\x9e/\xff\xb5\x83\xcfGG\xf4L\xa4\x08\xdc\xb68\xa7!\xdeg\xb8\x0e+\xb9\xce\x82\xff\r$x\x8bA\x8c\xd7\xbe\x9b\x14\x82\x90\x03\xf5\xeb\xeb\x19\x96\x00-\x07\x0bY\xd2\x9b\x93\xfe\x93+\xb5\xb9\xfb&V@\xe41\xad\xc0\xee\x02\x93\x02d\xf0\xa66^|\x83\xdb\xd9\x85\xdb\x89\xba\x81\xdaZ\xda\xccT\x88\xd6q\xf3\x86\x8b\xe5C\x03xDP=\xa5\x85+9\xa1\xf4UOdl\x88-\x0f\x93\x82\xee)J\xb7\xfe\xac\x86\xd1\xa4\x0e+\xea\xb4X\x8bm\x1e\x7f\xc3\xca\xa2\xb9~\x14#\x0fO\x03\xc0\x91m4QJ8F\xe9\xdcu\xd8NX\xee\x00\x9b\xa3\xf0\x90\x0ci\xcb\x01\xad\x02\x8dk\xfc\x16\xcf;\x0f/\x88\x15\xf6M\xa2\t\xf0 \x9a\xeb\xe24\xb3r}\x06\x14\xba\x80\x19\x04h"x\xbe\xb6sn\xd6\xaeD\x8c\x9f\xac;0P\xf5\xc8\xec0\xe5(".t\x00\xecA\xdb-x\xec`1\x0e\x02"\x15\xc8\x1f\x94\xe3\xcb\xc7X\xb4D\xe0\xac\xff\xb4\xec/\xdd\xd3\x9a4l\xb4?\x01\xe2\xe74\x80\xed\x8f\x9d\xd1\xd6\x02\x89\xae\x08\xe6\xb6\xf7(\xdb`\x0b\xd0Z\xde-\x80Js\xa4ohc\xdf\x88\x89\xd3\xd1\xbe!d\x8b\xb8Y}\x03\xa0U\x0f\xf5un\xea_/d\x98\x1e\x1c\xac\x0e$4\xe3P"\x107\xb1=\xea\xc2h\x13\xf9g\xa3\x16\x0b\x14\xc5T\xadh\xaa2dW\x80\x83\x82j;\xe8!\xf3\xb4\x0b\x96c\x97\x81\x95\x96\x03\x05=\xbd\x8e\x97\xce<B\x06`HPC\x7f5-\xf2\xe2\xd44k\xeb\x8c"\xc4I7\xcf\x12A\xcb\x1e\xbb2\x94T\xa7jG\xcf\x97\xdd\xa94\x925bII\xbe\x03\x91Cw\xf0@\xe7AAB\xe7\xdf\x85\x84\xad\x87\x8b.\xe8\xf1C\xc5ar$\x08.@\xd1\xad\xe6DM4W^\xcdd\x1c\xcf\x1b\xa8\xe6P\x90X\xf6@\x05\xb54k\xc0vs@Z\n\x1d\xac9\x8afp\xe7i\xed\xc2%f\x99\x81\xe1r\xf35 \xa7q\xffN\xf3\xd6\xe3\xfa\x9d\xa5\xc6\x94\xf8\x19\x14:\xc8\xdab\x13\x12t\x97]`]\x88>\xce\xb6\x12\xeb\x88G\x80E#\xc8\xb9b\x06\x84\xb7\xc83\x0b]\x81\xc0\xedsJ\xd0\xa9c\x8f\x8c\x1a\x94\x14\x12\x1ax\x9e\x01\xc4\xbf\xdd\xa8\x9d\x9bo<\x10\xde\x04\x00\xd5\x1e\xdd\xf6\x17\x17\x9b\xe3gV\xde\xccEv\xa6\x0cz\xf5\xd1$\x9e\\\xc5\xdeAF\xad,>\xaa]|\xc1\xf8\x85\xf1\xa3bj\xe3*\x87\xd4\x90N\xa8\x16v\x88\xd7\xa4#\xf0\x0e\x90\xe1\xdeA\xa7\xb5T\xf4\xdc\n\xea\xc6Y\xad\x9f\xf3\x814=8\xc0\x15\x94\xafd\xdct\x1a\xf83\xa7\xd9\x99P\xfb\xfe\xb1?{\xaf6w\xaava*\x86\x91\x91F\x12q\x06\xe6b|\x82,r\xd8\x11\x93\xf6*e\xb7\x94\xe4G\xac\xb8\xfc\x11\xce"~Boi\xda\xf5\x06\xe0hW\x9c\xa4x,\xe4E\xfe\x80\xaf*y\x83\xfe8\xde\xa7vK\xe2\xd1Q\xbe\xaa\x9f\xb3M\xc2>\xce___U\x8fY.\xd7\x8f-\xc17\xec3\xd1\xb2d\xaf\xf3\xd7\xaa\x0b\x1c0\xc0\xa7\x9bw\n\x1e\xf9Das\x06\x0b\xac&m\xec\xea\xd0\xfe\x05\x1b\x99g\xc5\x15\r\xb1\x8c\xce\x89*\xedZ\xce\xcd\xbb\xe8\xbcy\xfc\xb0\xf9\xe0{45\xd4\x84+\x8b\xe7\xeb\x0f\xef\x81\n\n\xe3\x90+ql\x1a\x9e\xa2h\\\x9c\xf6g\xee\xf9w.6/\xcf\xfaS\xcf\x1b?\xde\xae]\x9f\x05\x06\n\xd2\xb1y\xe1\x05\x0cT{|>\xb2/\xe4\x84\xd2(\x15_\x94\xfa\xc8_\xfe\x13N\x03\xfc!\xf4\x8a~\xa8\xf1\xcb/#\xfa\xe4\x95\x08\x9a\x00j\xa0\x8f\x10\xc8&\xe6\xb6\xd9\xb9L5\x87\xc4k\x83\t\ttl\x15\xaa\xf9~8\xfb\xc5\x01P\xe6\x80M\x90\xf0Mv~R\xb0>\xb0>J\xa5\xd75\xe6\xa7@\x8f\x13D\xc0\x1b\x8d{3\xfe)\x14\xa4p\xd0\xe1\xf4\x13F\xa6\xe4\xf5__\x9f\xd2>\x97\xbd \x14\xd0\xac\x07q\x90\xec\x82}\xe8\xfc\xc4\xfa=\xd3R\xc6qsIEH\xf0\xb0k\x93\xb5\x1e\xa6\xb2>\x84wR4\xabv \x95\x90\xab\xe3)+\xd8\xb4\xcd_\x01\x11~\x05&7\xc0Q\x80\xc3\x86\xd6n\xb5\\F%q\xe3\xc7\x1b6\xac\'\xe8I\xdb\x07\x95]\xf6\xb59\xfe\xcc_\x98\x12E\xdf\xf8=\x9c\xe8\xe6\xab\xebJ\xff\x1d\xf3\xef\\\xa11\x14\x06\xd0\x99xo\x06\x16\x18\xd9HY\xd7\x1f\xb6\xd0\xdbh\xbdh<X#\xfc\xe5\xa8\xd5x>\x81nH1.\xd4\\ P\x81`\xe8w\xcci\xea\x8b\x0b\xf5\x1b\x8f\x98a\xf2\xc9\x0ex\xccNf+\xb8M\x03@\xacC\xb0\xa1\xeb\xfb\xdd\n;\xf8\xc8\x015<\x04\xfa\xef\xfaN\xe4\x93e8\x0e\x0e\xb0\x19Trg\xee\x03c\x10\xf7+|\x8b"\x1f\xb8\xe5\xab\xa5\xfa\x833\xc0\xd3Pb\xc9H\xcd+\xf3\xc08\xd9\x03\x18\t\xa3xN\xa6\x0c\xb6G\xd8\x8f\x87\x9e!\x9a\x1d\x049\xc9\x0cu\xb2\x8a\x87\x9d\xf2p\xd9%\x16N{U\xac8\xfd\xc5\xe2!\xe5\xac\x05\x1e\xf7xF\x8b\x91\xdaS@\xcc\x15\xd6,xrV6\xe1P5\xee\x9d\xf4\x1f?\x00)!\xbfG";w\xc3\x07\x15\x85\x96\xa3\xbd*\xfbw\xf5\xa1o\x12\x88K\x00M\x97\xe1\xc8\xa0\x84\x1fNv\xfd\xfe\xf7\x1b\xbbR\xa8\xb6\xf1+[\x10\t\xa4\xb3\xb9\x85J\x12\x9fEX\x08\xc8Im3\xd3\xf2\xc4J\xee\xaf\xba\xb9\xacZ\x92[P~#\xa5\xba\x02\xa6\xe7\xe6\x05\xad${\xfc\x17wA\xd8\xb0g\x1dH\n\xed\n\xb0\xa3\x99\xb9\xd32\xeb\x8fO\xf8sQ\xc5s\x03R\x92^\xd0\x1f,\x02\x1f\xf83}\xe4\x93\xbd\xf2\n6\xee5\x9aq\x93?[\x9b\xba>\xdb\xf4\xd9\xc7\x9ft}\xf6\x11J\xe1\xa9\x8b\x80\x198\xf9\xb0\xc6\x80rzA\x01 \xa0m\xd0\x12I\x9e\xe2\xca@\xed\xb1\x10K\xa0\x05\xe0\xdee\x80\x1d\xf6\xa3r\x00\xd2*[\xcd\xa0\xf7\x05\x88\xad\x0c\xc7\x85\xe4\xb8\xff\xf8^\xf3\xcaUq\xeb+71\xcex\xf1\xe5\xca\xf2\xdb\x95\x97\xa8\x14\x81\x90\xac\xcf,\x0b\x15\xd1\xd6\xd6\x9f\xcd\xd6\xaeky[*#\xc6\x07\x12\xa6\x0eA\xc0\xa0E\xaf\x96<\xaa\xdc]\x9f#\xba\x01\xc6j\xc1\x05Z\xd6\xa6\x97\xb8\xb1K\xa0\x8e\xbbG\x94\xaf\xcb\xb1\xaa\x9ea+\x00}\xdd|\x89\xdebr\xb6\xb0\xce\xc4;\x034\x85\xda\xd2\x1ddWzH\x89\x11\x9c\x9a\xae?\xd7\xce\x15q_\xc3\x11\xca\x03U\x91\xf2f\xf8\xbdSi\xfc\x82\xe1\xfc\xca>\x04\x1a\x8b\r\xa6\t\xa2\xd1\x0c\x13\xa1\xb1\xca\x87R\x05),\x8c\xf1T\x0b\x99!\xbb0Hgs\xfag@)\x13s\xfd\xea"(\xe4H\xfa/o \xef6N\xc8\xca\xdbk\xf5\x0b\x97\xd9\xbf\n\xab\xa9\xdd\xd2B\x1e\xa6\xecc5\x12\xf8\xce\x88\x01\xf4h\x9f\xe8\xd2}#\x18\xe8L\xe3?\x9b\xc0\x8a\x1cr\x8e\x1c\xd8\xfc\xe9A\xe5(\xd9&\xaeA\xc7\xe4\xa9\x1b\xbb>\x17\xd7\x1b\xfa?\xe18\xe4\xed\\\x0e>l\xcb\xb9%\x10\xda\x7fQz4\x8a\x02\x84\x1c\x8e$\xc0I?\x13\xdd\x97I\x11Dg\xe3\xdex\xed\xf4\x05dh7\xfe\xd6\xbc|\x1b\xcf\xef\x93\x19\xd0c\xa3J2:2\xfb\xdc\x02@\xef\xc1RF\x12\x19\x98\x8ac\x07\x9b\xdbi\xce\x1d$\xfd\x8c\xff%*`Q\xc0\xfb\xca\xe6 S-\xec\x00\x04\x93\r\xa0T&\x1bFu\x13\xad\xc3d\x1d\xd46a\xa0}\xdd\xffc_\x1fh\xdf\xe8\x88E\xd9"OP\xa3\x1a\x15\x05;\xcb!\xd1\x1c(\xd1\xa8a\xc0\xd9\xf9\x8b\xf8\xf7\x03\x8el[\x9b>\xff\xf0\xd3\xcf\xb5\xf5A\x11m\xb6\xe3\x14w\x16/\xf2\xa6\xcf\xc1\\\xfdTcQXu\xed\xdc]\xb0,\xfcG\x8b\xf5\xb7\xdfb$\xc3\xc0\xa7\xe6\xd2\x03\xa8\xb5\xfdK\xf7\xb6}=\xbbw\x11\x8e6\xafc\xe5\x00\xbe\xc4\xff\xfe\xc9qJ\x1c\xa0@#:\x0c\xa9\x84\xf1\x14x\xbc\xf3\x14\xf1\xc4\xedH\xd3\xef\xfd\'\x93@\x9f \xf3Q\xbf6\x00\xf0O\xdf\x04\xd84\xc0\xe2\xa1\xd5QV\x84\xcf\x04\xc3\xd8f\x8a\xef\t\x08\x89\x83x\xc6\xc2\x0b0\x99@\x1bW\x1e\xf9M\xb7\xee\xe9\xb1\x06\xcbvi\x88\xa4O$\xb6\xc4\x9a7za_\xddm\xf1\xec\xd1\xcfY\x10\x0bs\xe0\xa0\xf4\x93)\xdc\x0b\xe5JP(\xe69\x806yA\x9d@\rH\xa6\xc0\xb5\xfb\x84\xee\x888\xbe\xdc\xbd\x15\xc9m\x17\xd0\x08\xea\xda\xb4T\xf8\xceX\xf8(S\\\xa2\xabu\x84\xc0U\xc8\x88L\x98C\x8c$\xaa\x05\xa7\xa2\x0e\x85\xb2\xb2\xe1\rq\x82\xc0\xf0\xf0\xfc\xcf\xa4\x02\xf6\xf6\xf5\xec\x1aU\x13m\x8a\x99\xe8\xeb\xad\xdd\xb1S\x1c\xb6\x1d5\x83i\xbas\x1c\x05\xbf\x1c(u~\x9c\xf6\xec\x01\xa7\x02\xdau\xb1\xec%\xf44\x1f\xfd\xf6i(P#\xd3l\xec\x8a\x9f\xe6\xe3\xd6i\x92\t\xd1\xe2\xbe\xd8\xb8W\x85s\xf6\x15\xbfF\x98\x89\x07p8;\xec\xae\xd1\xbf\xe8\xc1\x80\x8bz;\x15\x02Ss\x06bG\xf0\xe4\x00l{\x87\xb5\x01\x8dr\x00\x92\x1elR\x0f$Z\xbdY\xcc\xa7\x80\xadHXy\xb3%ag\x15U\xde,\xea:\xdabNa\x90^\x11uQ\xad\xf5\x93\x18\x94~n{nf\'p\xf3(R\x83\t)\x18A\xd0u)\xe80v\xec\xe2Q\x03)D\xdf|L\xdf\xe8\x99>\x8d\x99\xe9O\xbdh\xaf:e\x0e\x16Fv\xd0\xe3\xef\xf4.:U\xf8\x14\xec\xd2g\xab\x1179\xd0\xbd\xc8\x801@s|{3[\x9fz\xe8\xce\r1c\xef%EfW\xd1\xf5\x9c\xc8\xb8\x05|\xd6G\xd1\xe9\xcdZ\xaf\n\x06\x8b9\xc7\tY\xf7\xb6\xaaW)\xe6\xb7f\x0f\xdb@O\xd9v\xc8\xa6\xf1\x9966(\xc0\x07y{\xf0\xe1\'\xfc,x_\x10G_~\xaaW\xca\x18\xc1g\x9f\xa9g\xa8~\x15T\xb0YmYg\xb0e\x9d]\xf1\x07n;&\x8e8\xb1\xdb\xc53t\xb6!\xe2`\xe4\x8d\xab\x8d\xbcU\xa2\xa9k\x1f\xfe\xa3\xc8\xf0q\x0ci[\xd9\x81\xf5\xf2)l\x83i\x89c\xd3\x1c\x9aF8\xb6K\xcf6\xaag\x03D5]\x9b\xd2\x1b\x829?\x8a\xdd\xe7\xc3\xab\xcfHl\x8f\x07\xd7\xe7\\\xe9\x98}\xac\x81\xc2\xb7J\x07\xc3/\x8be\x10\xb0\xc4\xd5J\x9b\xe84\x88\xa2\xd0[\xd51\xc5\xac\x93u36\x1a\x17\xc0\xac\xba\xbe\xdej\xe9\xb3\t\x92\x93e\x18\xbcEX\xb6\x00\xe9\x12\xb4P\xd6\x86\x11\xa3\xe6`\xb4\xa8\x0e2\x16\x0b\xaf\xda\x0f\xd7V^=g\xd1F\xde\x89{\xacC\x18\xa36\xa6&u\x18\x9b\xc3\xd5\xe1@u\x94mF\xf4\tRz\xb5\xf7\x9e\x94o#\x01\x0ex!\xa2\x0e-\xa7\xca0\x06\xaa(\x87\x00\xd6\x01\xecG\xd2<\xac\x12hu\xa2\\p\xa0\x93\x17\xd2\x18\x9b\xa8}\x7f\x0eS\xdbffW\x96\xafZ\x9d\xbf\xbe\xb9\xfe\x99\xe5\xbf~f\x06\xdd\x1b/\x1e\xfb\xcb\xdfD\x95\x0b1\xa0\\\x0f=\xdexf\x93AnD\x87\x95\xc4l\x82\x0e\xabR\x05BM\xb1\xb3\x8aL-\xd8M\xe3\xbd\x14>\xfa\x0cv\x12P\xdbg\xa4V\x88?\x80 \xf9\xf5\xf5UP\xd1\xfd\x99\xc7\xa0T\xd4\xce\xbfD\'0\x19\xec\xa8\xf1\xaa\xe4\x87D\x18]\xe2\x8d\xce\xbb\x9e\x87\xbbWDo+0C\x8f\xa2\xdezsu<\x16U0N\xfd:*\xd8\x02\xc3\xa9\xa8\xec\x15F\x1a\xfb\x9b\x19\x1f\xfe\xd4\x13m\xcc\xb0\xa5Y;\xf3m\xfd\xd9\x05tK=\xfa\x1e\xccR\x00\r\xe3n?<\x00\x83\xbfv\xea%\x18\x16\xfe\xcc\xc5\x95W\xb7uNI\x1b\\\x82\xde\x9f$\xf5\x1bw+\x95v\xbd>$\xff$\xe7C\xe03T\x1e\r\xec1\xe2,\xd4h\xd5\xd4z\n@\x9b\x8e\xf7q\xec%\x0eO\xe8u\x14\x0fO$5\x83\xf3(U\xac\xb32\\\x0cT\xc2\xccP\x114v\xa5\xaa\x86\xc9\x1a\xa6j\xde\x9c\xc0\x04\x9b\xe3o\xfc[\x0f\xfc\xc7\x0fX\x11\x0c\xe2\xcb\xa1\xdc\x8d6\x88\x88\xe4m\x90?\x95\x933(K\xe3H"\x15\xa5\x19~\x91\xe7E\xf7[t\x96D\xf4@\xe1\xd7\xbc\xd9\xec\x7fC\x02\x10\xd5\\\xd2\xeed\xe9\x9d\xeb\xfbm\xcf\xa4\x1b\xa0\xa7,\xbd\xcf\x08\xa8\xdd\\\xf4\x9fLKv\x9d\x15\xa4\r\xbd\x98o\\\xbe\xc1\x9eb\xf3\xf4\xc3W\xcd\x9b\'\xfd\xa5\x19\x15R\x98\xf2\x97\xaf[\x9dV\xe3\xd9\x8b\xfa\xc2%q\xe0\x99\xa0\x8aK\xa8\xecD\xf7\xce\xce"\xc1hF`\xed\x91\xa4\xa7\x9c\xdd\x0fJ\x97Z\x0c2\x8b\xf5\xe8a\xa1\xd05/M\xed\x1c\xa5y\x08\xe0+\x8b\x8f\xfc\x89\xf9\xe6\xf1y\xeb\x0fj\xa0]\x7f\xb4j\xf3\x97\xebO\xefDA\xca\x83\xa4D\x06\x9aE\x05\x1d\x9d\x0c\x8cB\xa6Sqg\xc2\xf4e\'=\xe0bNS.YN\xa8Q\xff\xcd\xfb \xf9o\xd9\x0fR\x7f\x84\xad\x14\xdd\r_\xec\xd9\xb9k\xf7\xde\xeem[{\xbbS\xa3\xe1\x85JT\x05\xe5\x002;\xb5\r|`e\x0b)\xc2\x84\xd1\xe7B\xb1\xa2\xd9!\xaf\x92\xe3(j\x95\xe8\xa2\x98\xfd&tZ\x8c\x8diC\x8d\xc1r\xff\x80\x1e\xafJ\xb2\x8c\xfe\x84dgG\x0b7\xfb\xc0\xeaL!i\x06\xc4\x10\xc2\xa6\x04\xba9\x85\xee\xc5O\xccs\xf5\xd4\x11\x12\xddV,\x14\x94\xf7U<I\xb6\x96\'U/\xc8\xdf\t\x08\xd3\x10m\xda\x8e\x8cx\x92\x0c\x99\xd4\x1c\xbb\x8dB\x8a8\x88!\xc1\xa2H K\x0c\xd5\xa1\x83\x07\x94\xd4\x86\xbf\xb4\xed@\xc6\xa4R:";\x17\x08\xe1b\xa6\x8aX\xc4,V\xdc\xb0\xf5J\xae:\x95\xa1"\xa7\x9e;\x87\x9dr\xb0\xbb\x8e\x82\x7f\xbavj\x0c\xf8[\xc0e8q\x82x\nzq\x96\xff\x86\xee\xb1\xc5\x85\xda\xf9\xe5@\xa2\xbe\x1b\xfcH\xee\x17\xae!\xccJ\xc2+\xd9\x03,\xd1)\xab\x94*\xca\x05\xa3s\x95\x05\x9d\xa4\xb0\no`OR jU6R+7\x88\x82\x8d(A\xe7\xd6\x91\x0e\xc6X\x9fb\xff\x0e\x9c/rL\x9b\x84\'\xea\x829\x00\x91\x10&E1\x17(I(\x9a\xcf\x8b\x08\xbeL\xb1t\x94\xf8^%F\xe2\x91\xd4\xbb3\xd9\xb8\xfd&"\xef\xfc\x89)\t\x04\xb24\xa4\x93\x84Q\x0f\x1d\x0f\x8b\x83\xc5+V\xcb\x19G9\xf7\x825\xa5\xe2A\x0f\x8b\xf0\x0e\x0b\x13\xa6\x8e\xa2$7S\xac\x94\xb0V\x89\xb6^\x00yD"\xff\xe7\xd8x\xfdoK\xf5\xcbo0\x81\x82RO\xcc\x9c\'^H\xe0\xea\x8b\x03Hx\x01/\xc3\x10\xcd\x94\xba\xc9\x0fA\r\xaa$\xd1WID\xa5\xa3K\xea\xdb*\xc8\xcf#i0\xc807\x9eeZ\xba\x04\xf6!\xf0\x8b\xf4_J\xea\xbf\x0e\xff1\xec\xf4\x97\x12\x94__?q\x0b\xd0\x8aG7\x9a\x0c\x89~\\\x1e|4\xd1\x0e\x8bL\xb6\xda\xad\x8bA\x1b\xabg{P\xec\x11I\x076\x10\xf8x\xc6T\x03\x85\x81\x90O\x14O\x1c\x07oH\xc6\xb5\xc3\x18\xce\xd5\xe7\xa2\x8c\x00-#\xd9\xb5\x01\xb8#Qt\x9b\x1d\xef\xc5,w\xf1\xecj\xa0\xaa\x05J^\x8cz\xa5\x83-g\xff\xf3\x1a\x80f\xb74\xb0\t\xe4y\xe1q\xeaW\x17\x9agO\xb4[F\xd6\xc1\xe08\x07\x98\xb7X\xdbv\x7f\xb5\xe3_\xad\x0f\xc5\x80I\xc0_\x03\xa8\xb0\xf7\x8d\xd0\xd2P\x00\xb4s\x01\x8f\xc4\xd2A\xbb\x9dc\x03Mb\x1f\x08d\xd6-\xc3\x81\xc0\xc44\xb0\x15\xe0\xc0f\x8b\x0e\x8b;\x8a\xde\x1bG\xf6\x9e8\x82\xd9\xc9\x16Yj\xe3\xe7\xc9\xda\xc5\xcb\xfe\x9d\xe55\xac6]\xb21\xb2\x97\xce\x1f\x82\xb9\x93\xfc\xc1\xdb\xb2\xaf\\\x05\xd3\x8b&\xed+\x1e\xa2\x8fm6t\x1bp\x97\x88\x80\xe2\x03\xab3\xce\x8bYw@\x18\x10\xa5\xcd\xa2\x05\x1e\xac\x84\x99g\xc0\xf9\'~\xf4\x17^\x82q\xa5\x99\x12\x06\x12&\xee\xbesI\\\x87\x94Ff\xd7\x95\xe4M\xe80\xd7\xd9\x96\x03q\xe6\x1a\x9f\x93\x00\xf2\xb0{\x14\xfdm=\xacF\x92\x8d\xa8~\xdd\x1c\x9bi\x8e\x9f\x13\'h\xf0\x12\x9f\x1b\xd3\x98h\x075\xcb-9C\x07\xc5Ko\x9a\xd3z\xd0\x88s@y0\xcc\x9d\xc4S2:\x1a\xbfNL\xde\xfcw\xa7\\\x145\x17\xbd\xf4\x83e\xcc\xdc\xa4\xaa,\xef\x9f9\xeb9\xa2\\\x82\xf4C+S~C\xb4\xaf\x85\x1e\x89\xfe==b*\xa3v\xbbA\xb4[dbz\xfc\xda\xc3\x13+o\xc0x\xba\xaaT3R5}\x10(\x86>\xbc:v"R}\xc0\xc8\xbfO\x07\x12\x9e\x8f\xe6(+*\x82PVVB\xb9$\x01\x95\xe6\x9d\xacksM\t\xa7\xe6\xb3J\xab\xd5g\xab\x94\xb33\xceP1\x07\xbc\tDN\xf3o\x97\x1a\x8fN\xa9\xdd\x1cW\xf97\xe3l\x9d\x02?\xe2\xf2\x13\xd3B \x9a\x0e\xb8g\xbc\x07\xc0\x90:qV3A\xa6\xbe\xe0\xa4\x7f\xfabc\xe8\x0b\xce\xee\xe7/\xb4\xb9\x18\x18\xd7\x1a\xea\x8dVm\xe1\x99\x06\x9d>E\n\x0c\xe2\r\xb4^\x94\xc9\x01\x042Ec\xf2g\xff\xc97\xcc\x870\xd2\x1c\xf6\'\x80\x0c\xc6\xcc\xa5\x85gfE\x03\x10\x82 \x0c,\xfd7s\xf5cK\xcd\xcb\'\xfc\xd37Y5\x8d\xce\xdf\xaaxG1\xf1A\x1c\n:\xbb\x04@\xd9\x81_n\xd6\xae\xcd\xfa\xb3S\x8d\xf9)\xc1Jg\x97\xe5\x8f\x05Q?\x89\xa3\xd8\x83\x1c\'\x93S\x00\xa7+\x02\x00\x0c\xfb\xb5x\xaf"\x80\x84\x03^\x89\xad\xe2@\x8b\x02\xf7\x81\x95\x8c\xae\x00(\xc2\xac\xb7 G\xfa\x06\xb0\x8e\x94zHG\xb1\x83\xe9\x94\xac=\x802\xedV\x9c\xbc\x97Lm\x0e\xa40\xd1.\x82\x7f\x80,51\xd2Z\xed\xb3\x7f\xf2\x02\xd3\xcc\xfa\'5\xbai\xa2\xa5\x0eF\xb5!\xfci\'"\xb6\x80\xff0$zx\x99\x99\x94\x98?\x8c\xd0x\xa3h\x051\xddK\x8a\x06[\'\x8b\xa7\xfdc?\xc31iN\xce\xa0R#\xbf\x1c\x8d\xf3X\xd8\xd9\xd6\xf2\x99*\xb9y$\x9c\xaa\x04\x1e\x17\xf1\xe1y\xf7\x82$$`ee\xdb*\xb9%\xabdg\x0e\x11\xa3`E\x9d\xddzZ\xcb\xa5\x10\xab$\x86\x98,[\x11,\xe8\x90p\xa4\x83\xd2\x1cz\x13\xe9\xf7\xd5\xd9U4y\xa5\xb0\xb4\xea\xf1\xb2\xe7\xc1\xb6\x15\xcb\xee\xa0\x8b\xb5B\xa20\xab\x9f\xa6\xa2[\xa0^\x8c\xa8\xa4\xfaq\xac\xda\x89\xeeR\xd42\xf3\xc5\xc3J\xdb\xcc\xb3\xb6i\x1eI\xd3\x9f\x15(\xcd\x92%veDM1\xfa>\xba\xd2\xd0\xc6>]\x86\x14\xa7(uv\x81\xa6\xd4\x06\xf8\xd1\xdf\xaa\xa6\xc4j\x04j\xb2\x88N\x10\xaf\xc5v\x06j,&N\x05\xaf\xc1\xbc\x18\x13\x0c\xde\xeb\x8c\x7foMb=\xc6KN\x8e\xf0x\xa9\x1e\x1eZ\x01\x127\xf6N\xa7BC\x03\x99\x97@z\x03\x9e\xa2\x81B\xe5\x817\xc4d\xcb\x0c\xb1R\x97Ob\xda\xd8[S\xea*\xa0H\xec\xaa\xd1\xc2\xfcms\x88O\xbek&\xa9KKG\x1e\xb4\x9b\xb53\xe2\x99\xf8\x92\n\xc3#\x05w\x16\x86\x1d=P\xef\xca\xc5\xea \xaby\x19*\xd7\xa1/\x03\xf5.\x96W\xb0\xec2y\x85\xb6\x95\x1a\xe3?\xc1\xb7|\xb0\xc26\xe6{\xb1\x08\x91\x14\xff\xb7X\xc4\xb0\xcd\xac!_\xdaH\xff\x1d\x00\x9d\x87\xfe(\x0e\xb2\x85\x9a\xdfd\x9b,\x83\x11\xf0_\xc82d\x83\xff\xffc\x19\x9b\x0c\xcb\xf7\xbd\xd8@\\\xe8O\x05\xe0\xde\xc9\x07b\xcf\x8eqjb\xceK\x8bn\xbc\xb5RAs\xdc\xb6\x0e;ew\xc0\xa5\x12A\xcc\x10\x17\xa7+e\xf2\x7f\xb1Q\\zf\xcc[\x8c\x87\xf4:Lda_\x91\xf2\xb7r\xba\xb8(\x82\xe7\xee\xa2o\t\x86\x90\xcf\xa4\xeb\xb1Y\xa7\xf4bU\xeeg\xfa\xc1:,\r*-\xb0O\x9d\x98\x03\x84\x84\xa4$]\x87\xca\x8e\x13\x9d\x1fS\x026\x15\x1aS\x8eiBt\xb3\xa4\xce\xca~\xef\x1f\x04\xb5\xcc\xf8\xfe\'\x94\xe8m\xb7\xbe\x9e\x8by\xf9\xd3\xd6\x97\x0fF\xa2~\xbb1\xa7\xc9\xe6=\x90b\xed2\'Mj\xdf\xaf\x9d!\x9d\'\xaf\x99\x13&8Q\xf0Q\x12k\xc9\'\xca\x8ax\xdb\xe8\x93az\xe0@\x9b\xdb\xd9\xc3\x92\xafY\x08\n\x06\x98\x10\xa8\x88\x88=JF\xcdL>d\x0f3\x15p=9\x03\x15\xf1\xbb\xc1\x0b\xa0\x0c*o\xdd?\xc4\x87\xd8[)\x96\x94\xab\xd3\xa8\x12\x80\xe5\xda\x86\x0eI\xd0\x83\x85/\xaeF\xc3\x93rFLt\xcd\xd40\xa7\x14\x88X\xd5ib\xd7\x89\xe9\x9f\xfd\xf1\xb9\xda\xa3\x1f\xde\xd3m\x88|\xb3vj\x19G\x9ez\xf2~>=\xedVa\x8f\x8eD\xa9\xb4\x9b\x87vD;\x8b\x82\xd5`(\x80\xa3\xb9->]>p\xbf\xd5\x0b6"g\xf2\x1f\xee\xff\xaa\xd8\xe5A\xa7\xa2|~dus\xa6\x1f\x971\xb7u}\xc9r\xd7\xe6$\xfa\x07\xfa\xbdB\xc1\x82v\xbe/\xcaIe\xd7W\x8b\xc7\x8bR\xae\xc5\xb9NN/.yh\x8e\xfd\xd4\xb8uv\xe5\xedB\xed\xfc\xcb\xff:_\x97\x02T2R%\x8a\xa8\xd3\xfe\xe2\x08\x8a\xd5\x1d\xce\xec\xe3\x04E|\xdd\xcc\xe8\xfb\x7f\xea\xe02\x03i\xa1(\x08\x1c\x1d\x10=\xa6\xdc\n\xaf\xee\xc5O\xa6\xa3\x0e\xc3e,\xac\xa2bj\xcd\x0e\xaa@X\xc5\xcb\xdb\xbd\xce\xfa2\x1b\xb89\xe70\xf6\x16\xc8\x96\x8b%\xac#\xc4T\x85\x8c\x8dN7\xae|\xc7\xe4\n\x8f\xa2!\x98\n\x02$\nCV\xd0\xca\xcf\xdbG\x99\xb4\x1c\xa9(\x03%\xb7\\\xa9\x96([\x1e\xe3\x8dKw\xb0\x16\x93\xb2\x02\x98%\xaf,\x9e\xa9\x9d9\x85=/..\xe8\x02D.\xee\xf2\x97\xcfr\xed\xb0\x14\xc3Q\xb6\xbb\xbfp\xa59v\x93\nR\xa71\x87z\xe1\xa5\x92\xd4\x98\x84\xd9\xd7\xb3k\x07\xb6\x86\x90\x82\x08\x8f\xcah\x817|\xbe\xb5\xb7{\xf4\xc3b?\x06u\xfa\xdc\xc2\x00\xaa5\x157\xef\xc0Q\xd8\xf2\xd9\x86T\x1a\xfbF%S1\xd5]\xb8\xc0\x84\'\xf5\x80\x9cx\x10\xaa\xb2\x94\xc8\xa8\xca\xa6e-}e\xf9Z}\xe2\xc7\xfa\x85\xe7\xc8\xa4uO\x05\x0eY\\9\xc1)\n\\\x19 \x03\x87\xeb\x1bT\x17$\xc9G&\x85Xr\x1e\x921Y\xa7\x92\xc0\x9c\xc2\xfd8p\x10]A\x91\xfc\\\x1dj\xe2\xd9\xdeUl\xaa\xd6R\x1c\x18\xa0\x06V,H\xd9\xcb\x18T\xd1\x86\x9b1\xf8\xd3\x8bX\x1d\x1c\xca)2\x97\xa1\xcb\xd2VY\x07%]\xafa\x15\x1c\xc5\xb4v\x0bxzR\xb5(\xd4\x15\x8frH\xbeM&q7f/Z\x92\x9a\x87\xb2\xd7>l\xbb9\xbb?\xa7\x8b\xe8[\xb2\x87\xf9\'\x92*\x88M\xb7\x8c\xf62\xb2PI\x8a\x0c-3&\xab2\x94Ji,7\x98K\xca>\xd8\xcc\x0bM\xdc\xd28\x80\xd2E\xaa\xfdp\x1e\x87Bm+zCm\x0fh\x8d%\x14\x0b\x18\xdfh\xb3B\xb3\xcf\x01\xee(p\x9f\xb8\x82\xf7\x96\xbcN\\\xaf>zX\x0f\xfdz\xc9\x7f\xf2M\xcb\xc0:\xb8\x9f\x08\x17\x80\xea\xa64\xa4\xe8\xb2\xef\x18\xb4RKJ\xb98\x08\xc0r\xafU\xc7c>\xaaZa\xcd\xcd\x8b\xcb\x9f\x13`XM\xe2\\\xad\xcb\xc7\xb8\xaf\x1a\x96%\xf29$u\x8f\t\xaay\xef\xa2\xffdY\xbcj!\xe5_i\xa6d\x92\xc6UR\xa66\xc7\xf8\x07\x89/(\x8a\n\x16\x182P\xa4\xf8A\x17\xb4\xa8f\x0e\x14 \xa5\xb0\xbe\xcb\xe5\xa1\xa4XP\xb7\x16.G\xc5<\x19.83r&\x98\xab\xf8\xb7\x1e\x88\xb6\xa1\x0b\xd7BV>b4\xa8\xba\xd0{v\xa0Mv\xf4A\xeeA\xc0*\x16\xb5[\xe3\x07jA\xfc,\xba~V\xd2qI\xbc\xa14k\x87U\xcaU=\x96\x80\xc1\xe6R\xcd\x04\xa0v\xc7\x97;\xba\xbe6m\x08\xd8?\xe5\xfd\xe4\xb3 \x91\xf2\x87\x0f\xb9\xe2\x18}\xf5\xc6\x9e\x89\x91q\xec\x84T\xa0\x18\xb5\xc9Q\x1c8GJL9X\xe2o\x9aV\x11c(\x94G\xce[\xcd\x0e\xf0d\xd2|\xaf#\xfcb.x-<x*\x15\x9b \x88\xf9C\xc0\xe4\xf2`C\xb8H*\xe4\xee!\np\xf1\xb8c\xc6\nk\xbc-F\x84 \xea\xde\xb1\xda\xf59L\x00{{\\\x95\x18\n\x93l\x8e]^Y\xba\xaab\\\x81\xd9\x80\xa5[\x94B\x116\x1bb\x12\x01\xc1\xde\xceQ\xd3\xb7\x10\xdd\xe0\x11\xd4\xdf(l\x929\x11KD\x96p\x83\x11\xf5\xea\xa8\xa5OX\xa2\xb5\x12\x9cR9s9\xd8\x1f\xedE\x07\xad\x021 I\x17\x1e\xa5\x9ci\xa5v=\x87\xcaDW,0\x1f\x90c\xcem\x19\xb4\x97l\xe2\x85\x7f\xecr\xa4\x85\x15\xfb\x8c\x82\xa6\x92\xe4I\x0b\xb2\x90\xda\x87\xc5\x94\x19\xdcGS\x92\xee\x18\xf2\x9b\xc68:G\x91R\x8c\xfeXD&\x98\xcc\x95R6p\xe0d\x0b|l\xe1\x89\xd2v6\x9b4\\3\xfcS1\x1c5,\x11\xe0\xd6\xd3,\xfa\xb4\xa7\xcc}\xc6\xf8}\xe8\xe7\xb8\x91\ne\xbcu\x82\xb2\xfb\xdf6\x1f\x9cW)\xc6h"b\x02L2\xf4\xdb\xd4(W\xb5\xa2\xc2v\xf1Ik;\x17C\xdb\xb0\x89\xc4\xa4\x90GU\xd0e\x8e\x82^Y8\xe4)\x83K[\xf9\xb2\xaf\xac\x81p>V\xd0\xfa\x03\xd4\xe2\xfa\x0b\xcciZY>Y[\x9a5\xb9{`\xcc+G\x8e\xd6x\xf9\x90\x15\xc4\xa6\xd0A\xab\xd0Y\xe8>b\xe71\x81\x99RB\xb2\x0e\x18\x07\xdc\x05\x91\xcd\xc02\xd6\x9e{tlq\xb0\xf5\x81\xa1E\x8b\x90\xdcL.\xff\x14\xaf\xec"5\xd5x\xc49\x99\xfe\xd9\t\x7f\xf6\x01+\xf5\xfe\xc4s\x7f\xe1\xb8f\xe5\x80rP\x1f\xe3\x9c\xb4\x87\xed\\\x95\xe0\xc6)\x03\xc5>M\xcf\x83\x15D\x8d\x01r\xe4T\x86\x8b\xeb\x1df*\xa2\xbfcu\xa2\xdal\xde\x8a\x0c\x9b*XFo\x98"T\xb3\xd6\xbcz\xd9\x9f\xa0\x88\xe5\xe2\x04\xa8\xeaXS<\xf3\xb8q\xfc\xcd\xca\xab\xdb\x8d[\x817\x9a\x17\xc1\x9b\xd2\xce6\xc1\x04\x92 k\x9b`\xef \x90R:tL\x0f\xa9\xdbM\x97\xb1\xa6\xe8@\xda\xc9\xe2\xe6\x89\x11T\x9c\xdcQv\xaf\xe0\x1e8*\xf5Iv;\xc6\xc5"n\x16\x04\xf7\xd9,l\x84\x99\x98i\xd2\x13*\x0f\xe4l\x91\xea\xd8\x89\x17\xd8L\xe9\xec\xe3\xc6\xfd;\xe6"\xe3\xe0S\x0e\x98J\x99\x17u`\xc3\xc1\x94\xa6<\xca\xaf\xa2\x19,\xed\xbe\xb0b\xa1Qo\xa91FM\x06\x9aG\'\x01\xd3\xa9\xee\xac\xc3t\x1ac\xaa\x08\xe9SN\x8f\xd63<e\xb4h%#\xd8Q59k\x15\xd2?\xe6\xf1\x8c?v&8\x85\xe1\xee\xa7\xef}\xd4Dmr\xbdhU\xbf\x02BJ\xfa\x91<\xecL\xc6)U\xf4I\xa4fA\xa6\x9a\xc4\x8c_\xdc\x01\x13\xe3\x8d\xf9\xa7\xd2\x17t\xe2.\x8b\xc3\xa0{\x99\t\x82\xcc`\xeaE|\xc8\x0c\xa7\xc0\xc1V\x95(\xbc\x8e\x9eB&W\xcd:\x9a\x05w\x04\xda\x1fe\xe2\x1d\xc5Z\xdd\xf5\xba^b\xc8\xcd\x02\xdf\x8bV^\x82\xca\x03\x9a\x1c&p\xb0J75Y\xff\xf9U\xfd\xd9\xd3\xe6\x95\xa7\x8dK3\xb2\x06J\x95\xae\xff2\x8d\xab\xa2\x1di\x91\xe4\nM\x9c2\x9cJW\x0b\x00H\xf2\xf7I\xfc\xc4\x8b\xa5\xc5\x0c\x96\x8b\xd5\x12-\x85#\xf3\xf4\x99\x15^\xb5\n\x94h\x86\xd2\x97`\xb0\x13\xa9\xa8V\xd3}\xa4\x84\xab\x0c\x19]\x98\xec\xa0SnX]W:=\xa5|\xc2\xcbn\x99\xfc5\xd9b\x05\xc15\xf6\x13q\xc1VJh\x14\xf1\xda2\x1e\x9e^h^<\x83\x15\xf0A\xc6\x0f*\xc3@\xad\x97o\xc4o4\xf0\x87\xd6\x8d%\xf9\xde\xbe\xacps[\xcf\x8eV]B\n\xfc\x90\xed\x85\xbdo\xeb\xc5\x7f\x1e\xa0\x02\xf7<\xea\x9ef\xd2\xd5GP\x9f<V\xe9L7m0\xd0\xean e\xb4\x19\x8dAqo\x15qt\xac\xb2h\xc3m\x10\xfc\x9a\xab:b$\xbc\x9a\x90\xf4C\x90\x9b\xb9\xa3\x1d\\\xf3C\xa4\x14\xe9\xfdi\xa6\xa7X\x91>\x9fF\xa8Q}gF\\C\xbf\x8eMQ\xc1F\xa1\x9b\xe3X0\xe5\xe6\x13H\x1a\xc60.T\xab4\xcdu\xa4\xca*@\xc4\x08\xff\xccH\x0c\x89"\\q\x00\xdd"V\xe7n)\xb2g6\x0c\xdfV=\xd5T9\xe8\xe0\x11\x92\xba\xec\xcc\x94\xf6\ts7j\x8fnk\x02\x0f\xa53)%\x1f\xe1<w\xd7\x9f\xb9\xb4\xf2\xeaU\xed\xc4Lm\xee~\xfd\xc6\xddpev\x0b}\xc8\xe2\xd3\xd5\x12\xaaf\xc9w\xa4\xb0\x05\x91b\xca\xdf\xa12\x84H&R*\xf5\x8e\xc1\xdbF\xeaW\x1b\\\xa8em\x83\xbf+8\xbf\x96\x89"\xa4\xf5\xaey\xdb\x066W\x9bK\xc8<\xcaFw\xb8\xd8(*\xf0\xa9\x96\x9d|\xf106})\xc3\x9f\xc8\x1e)\xfafs\xc3\x0eM\xab\xd8\x95\xdd\xd3\x96a\xed\xd42f7\x91\x86\xc6u`\xf5SS\x81H\xff\xf1U\xf3\xf2\x1d\x0c\x91\x9f\x7f\x89L\xb2]\xfd~\xb5p\xa8P\x1c.\x88\x08\t\xeb\x9c)0,\x14."\xf0\xdbn\x8e\xda\x8c\x89.\x16QIH\xc8[\xd8\x8c\x8d\xba\x90I.\xbd\x94\x06^x\x1e\xd5)P\t:9\rk0Y\xa1\xf8s\x8c\xf8\x17\x16=\x9e\x9d\xc45\xc72}\x83\x05\xc8\xa2\xe0\xb8\xff\xaeE\xf5J\x8e\xb4\x8a\x86\xd1\x94e\xe6\xea3\xbf5l \x19/5\xfa;\xd5\xb5\x12\x04[L\xa7t\x14\x8d\x88\nrZ\x8aA\x8fVq\xc1\xc3\x08\xaaj\x16Fn,\x10i\xd8\xf4f\xec\x12\xacSG\xfbX\xf5\x0c\xb5\xf2\x9d\x9a\xe4\xa63\xcd\xc9\x99h\xd3\x15\xea\xbe\x174\xc2\xff_\xd6\x08\xd5\xa3\x8f\xfe\xc7\xa5\x11\xaeC\xa7g\xaa_\x91\xf4<\x82\'\\|\xcd-\xcb\xe0s\xa8]\xa4|\x8ei\x17\t\xdfh\x7f\x1fp\xe8\x81\\\xd5\x1bR!*\n\xe4\x83\xde=X\x08\xf2\xe8\xff\xec\xf4\xf7\x12;\xc4\x9c\x17\xd4\xf3\\\xec\xa0\xe2V\x8er8\x11H$\xd4\x98\x16[[\xdd\t\xfa\xd0QtM\xa7\xb7\x18\x83\xf9\x0b?\xd4\xa6\x9e\xd7\x1f<n\x80\xda<u\xb7\xfe\x83n\xde\xc4\xd3pRD4:\xc8 \xee+\xdb\x99C\xc6X\xe2\xc0ss\x08\x95\xe7\xe0y$\xf3\x82<v\x9a\xa4\xa5\x8b\x1e\xec\xda\xd8\x19\x98\xb1\xf1\xf6M\xe3\xe5\x1d\x13"r\x14\xd7\xc6~\x94\xaeao\x9e6o\xbc\xe5\xa4C\xe9\x86\x14\xea\xc4?\xecE\x9b\xe4Q\xe4H\xa8\x050Wu\xa8#Q\x11\xc0\xc3\xaa4Ua\xca\xa7\r\xf4\xca\xc12@c\xc1^z\x9c[\x88\xc446\x0e\xc7B\x9a]\x90\xeb\x87\xad3\x1dl\xa9\xbd\x9e\xa9\x9d;\xe5O]\xab\xcd\xddh\x8e\xfd\xe4/\xdd[Y:\xc39\xaf\xdc\xac\xa5|ts\x9b\xd2\xb0b\xb4\xbaJq\x18\r\nwp\x05\xeb\xc7\xb1\xf3R\x107v\x1b\xa4T\xe0\x98a\r/@\x99\t\xc2\xca\xe2O\x80\xa0\xe8a&,\xe9{;\xd2\x19\n\xd5\xf6\x05\xb6+\x90\xff\xb0\xb7\xf9\xc3\x0f;\xbb>Io\x80\xff\xeb\xdc<\xb2g\xf7\xde}\xa3\x1f\x0e{\xff\x8d\t\xa1\'\xbbeD\x93\xc4\xa8\x11\x0b\xfa(\xc2\x8b\xbf\xc4\x1eb\xc8\xbf]\x10$U\xea\x1bgg=\xec\xa6$\x8d\x12\xe9X\xa3\x0b/\xe7p\x07N,k\xc2\xb2\'\xaa\xca\x1a\xac\xe6pgd\xe1\xd4,\x0emg\xa0\xe4G\xa7\xfc\xe5\t8\xde\xcd\x8b\xdf\xa1{n\xe1\n \x9f\x15y6+u\xf3P\xee\xbeZ?3V\x9b8\xd3\x8a\x884\xb5\x1a"\xd8\x93]\x00\xbbs\x049\xb2\x95\xdc\xdd\xdb].\x17\xcb\x1d\x06\x9a4\x86\xbb\xe9\x1d\x0c\xc6\xb6nk\xc5-T\xcdnWAO\xa3\x1cr)\xbd\xadpB\x8dB(Q\xd7\xa9\xbb\x11\xfc\x8b\xcb\xa2MdB\xe3\xbe\x91\xdc\xe9H\x9f\x01\t\xe9Q\xe7,sY\x91*\xd9\xf2!\xa1 \x96\xb6\xa0=\xe1\xc4\x81s\x9d\xab\xe5+6\xeaq\x05\xd0\xff\xec\x12(U\xeb\x11#V\xd6\xf5J9[p\x0fB\x02\xce!\x13\x15k\xafX5}\xfe%\xb2\x93\x8b\x0b+\x8bc\x0c\xb0?u\x89\xd9l\xfd\xf9w\xcd\xf1s\xbc?\xb1\xe4\xc7\x874b\xfd\xe4\x10\x8e\xe1\xa1\xa3(\xe0\x90\x83b\xa74\xbc\x1b\x80\x03\xef\xb9\x1cv\xdc\xc2\xf6\x9c\xca\xad\x7f\xff>\x16\xbf\x1f\x83\xc39\xd6\\z-\xd7g\x80\xac\x99;\x0e\'\xb6>\xb3\xccm\x1f\xeb\x97\x97\xfd\xa9\x93\xd2R\xd0\x9c\x91y}\x821\xce\xef\xaac3\xcd\x8c\x07\xd4u0\xa2xI\xcd+\xe7j\x0b\xcf\xd0\x9a\x0c\xaf-\x8e_\xf7:b\xc7iF\'G\x1d\xa6<\x8c.<\xce\xc3\xd2\xd6\xb9\xea@\r\x931\xd5\x02\xd3YY\xba\xc3\xb9VA\x1b\xe5P\x97i\xa0\xa3R\xb1\xe09f\xb4\xb6T\xf4\x8cp\xad\xf4X\xe9\xa0[}\xb6\x8c\x04=W\xc4w\x92\xd0G\x18\x9e\x05\xc798\xcd\x9d]\x1bt\xb37*bv\x0b\xdc\xf2P-K\x97\xe0\x13\x0f\xe1\xae\xa9\xc0Ss\x0e\x17\xe4kr\xef\xb7\xb1-\xbd\xd1\xe0Mj.\x88\xb2\x01\x93\xe8O\xbfz\xdd\xbf\xf9T2%\xb8\x17\xea\xfd;\xfe\x93s\xc0\xfb\xd1 %\xf2\xaa\xfdp\xcd\xf0+\xa32\xa2\x90\x90.\x1e\x8a\x1e\xc6\x1c\xdeW`c\xcbk2\x14\x02\x06) \x97m\xd7S\xed\xe9(\xdf\xa3\xbc\xde\x031j9x\xf0\x85\xea\t&f(\xc2\xfa/\xcd5/\x9eb\x9d*\xa8\xc9\xd5c\xc7\xd8\xca\xc3\x1ekl\x15"\xf9\xb6\xb9Y\x802i\xc2.\x12=`\xc7FQ\xd2\xe4\x99\xda\xf9\xe5\x10\xaf_\xcdO\x08\xec-\x83XH\xa6\xdaT\xc3\na%<\xeed\xc9\xb1\x92,\xb2\x83\x9cg\x16\xc0\x06\x14H\x15\xcfZ\xdf\x95\x9863\xdc\x9f\'\xe5"\x0bs*D\xb2c\xed\xc5\xb6\xe3y\x87XjRo\x19\xd6%\x1d\xd8\xfc\xd1\x86\r\x1b\x0ej*\xcb;T|nT\xf1\xf7lGi\xd1"\x9a\xa5\x1f\xb1\xd6w\xe4\x82\x07P\x14\xd3\xeb\xa4j\x97\x0e\x12\xfc>\xc8\x8ax2)\x9eb\xd6\x83\x88\x1e\x03\xe2z\xf1\xc2\x88\\\xf2\xec\xac\xf0h\x889\xcbA]\xb5\x83\xe7F\xddF\x84Y\x1axl\x14A\x89\xaf\x904\x8e\xac\xf5\x97b\x7fz\x9d(\xa5\xd4\rT\x8a\xc7\x97\xeeD a\x95U\xc1@\xb9\x1f0?\xe2.\x9d/\x02\r\x15\x0bnF\xa5YH\x86\x95m!\x15\xe7\x8c\xae\x91\xe6\xa9\xe3\x8a7x3\xc7\x0c\x95\x84\x81\xee\xe2\xcfiV\xc4\xbf\x83\xa6\x91\xfa\xa8\xa1\\U}\xfe\xc1|\xc6p\xaf\xd1L\x97\xa5\x80F\x17\xce\xb5\x85.\xfbJV\x8a\x15;\xb7\x85\x14a\xca]\xcalI\xd4\xce}\xdb\xbcr\xfa\xd7\xd73\x9c\x90\xc3\xc2\x0b\x18\x13\xa8\xb3\x95-\xd4\xb8(\x81\xc5\xf8\xf6a\xc7\xe4\xa2\x9aNmk\x80\x12\xa5a\xa3\xa9\x82\xb2\xecd\x8ae.;7\xda0\x1bZ\x04i\x10d\r0-\xd4\xe6\xc6\xea\xcf\xc4\x14\xa8\xff2\xd6<{\x02\xcd\x1fP,/\xbe\x96,\x1aCW\xd0,\x86\x9b\x19\x06\xea$w2(;%D|\x16\x15\xc6b\x8e\x1a\x14+\xb8H\xd0\xe3\xf6\x07\'\xb4\xecp\xd8m\x80x\xb4t9Dge\xf0\n\xcb\x9c\xe6\xcdW\xf5\x97\xe3(U\x88V\xb1\xf8\x10v\xe1\xc6\xdd\xc6\xdf~0\xe1V\xc0Q\x10U\xe6\xed\xa3\xe5\x02\xa8\xa0\xac\xa9\x0bD\xb0\xae\xdd\xaa\x94\xab\x95!D\rEX<\x92\xa3\xdc\x1c\x9e\xb2r\xd5QR9m,r\x85z8\xf7\x97\r<8\x83\xd2\xf4`\xea\tS\x82\xdcCF"\x115\x81\x9f\xbe\xc5\xeaq&\x8c\xcb\xc7V^Qpcy\x1e\xec\x88@\xf3":\x19\x026\xc7\x01?l%\x95\x88\xa4\xc1\xa0w\x14\xfb4%\x98N"\xdd\xe7\xf0[\xecg\x96\x10\xc2\xe5\xd4\x04\n\xe3\x87\xfa\x12r\x9b\xaa\x04S\xeb\xca\xe2#\xfc\x1d\xb5\x7fJH\xe4\x92\xae\xf4\xa2\xa7\xd8\xba)\xc1aL\xf544\xd0&\xfa\xd5\xec\x94\xce\x84\xa0_}DOO\xfc\x84\xd1\x05~*}\x8b\xfel\xbb\x15\xb9c!\xf0+H\xdc\xb5=\r\xeb\xcbg\xd2\xebjg\xc7\xebKgY\xbb\rT|\x10C\xc7~\xd6w\xa2P\x126\xd2\xb3\xf4\xd7\x0e\xd1E\xab\xb1A\xbd\x0caW\xf1\x98 \xb12xrn:8\x1dX:\x908\xac\xf3"H\xac\x01\x883\x84\x80\xe2\xfby\xeaW\x7f\x01"\xe0\xbb\x96\xb0\xbd65\x1fb\t\x80\r}\xc6\x96\x85\x7f\x85\x14\xbe!L\xd1\xc5\x83\xddF\xfc}\x05\xc2\x0fm\x01\xe2S\xc3v.\xb7\x1e\xe4\x16v>p\xf3\x1c\xd3\xa6K\xa3\x06,L\x18\xcc\xb3\x9e\xc1yt\x01\xb1\xeah\x14w[&"5\x155\xb1\x1f\x9fL\xf2G\x00\xb7~\xe1\xb9\\\x93\x12\xd5\x0f\xa3\xe0\x89Z\x1c\xc3\x8918\x8cL\xba\x9dT\xcf8.\xb1\xaf\xe0\xb0\xcb\xa1C\xcd\x80\xb6\xbdm\x9c\xde\xcc+\xe5<\x05\t\xcc\xd0\xd5B\xe2O\xd5\xe3b\xe4\x98\x93\\\x0c\xf3!0C\xe3\xa0S\x90\x98\xbax\xeb\n\x00\xf1\x86\xcc\x90\xad\x8cUS\xd0\xcf\x86\xc12k\x08v\xb12t4\x1a\x005\x90 \xdd\xe5Iy\xe1\xa6\x9d\x0c\xfc\xea\x00\xaf\xa6L\xc5E\x0e\xb9\x89\'\x13\xb2\xe5(\xa3Mr\x86\xe0\x10\x96J\xd1\x1b|\xcc_\x07\xd02lL\xdah]L\x9e\xe2n\xf7\xcc\x96\xdf\x15\xbf\x0c\x0e\xe4jo\x85\xc9%b\xfd\x17)\r\x14\x13^\xcbv\xc1s[a\xa5Cz\xeen\xed\xfcs\xe6y"\xc2\xa7.\x9al\x9aSA\xa3\xe0\xb6\x03&\xa0\x8eat\xcbf\x0e\'S\xe1U\x88\xa5\x1cc\x1f\xefc\x94k3y\x8d\xcb\xdf\n\xda\x92\xebT\xe4\xfaO\xd5\xd3\x8b\x04\x06oz9/a@\xdc\x7f\xe3\xa6\x85\x16L\xf0m3\xaci\xcc\xdd\x80\xd3\xce\xe2\x89\xa9\xcb\x7f\xfa\x90\xc9\xafv|\xa2\xf6\xf38\xf3\x05\x0e\xc2\xb39\xd2\x0eCkZ\x84v\x01(\r\x04u?\xcd\xf3\xd5%\x11\x94$\x1cb\xb1\xa1\x80\xbaAox\x91\xdd\xcb\xd7\xc0}\xe5F+\x16\x00\x9a\xe3\xaf\x05\xa4\x92\xedy\xb1[\xf7~N\x8e5P/\xf61G\xdb2\xe3\x80\xce\x913\xd4!&c\xcck.8h\xc4\x83\xa6\x06J@\xd6\xf5\x84{\x98E\x00A\x03\x06\xf6\x80\\i\x8e\x7f\xc7\xe7\r\xd1q\xf5zsl\x0c\xb5\x10BA\xe3\xf5}P\x8fL\xd7`\xfc\x99|\xaf-4m1\xb4\x1d\xc9O\xb2\x8aQ\xa6\x01&\xd3\x90hh*\xecVm\xb5\xd2V\x9b?j\xb5\xbd\xeb\xfd\xed.\x838P-\x03\xc8e\xf1\xb0\xe9\xc4H\x13\xcb\xba\xac+:\x06\xbbw\xf4=\x1e\xda&z_\xc8W\x15&\x94E\xf3/\xbd\xbbwqG\n\xf1\x99c\xdb\x10t\xfbh\x9e\x07f\xa8\xfd\xcf|K\xae3\xac\xea\xa6\xd0-\xd8\x0f\xfaS\xb9U\xac\xd0\x88\xacG\xca\t\xc7\xb4 \x84\x9e\x8bF0\xd9\xf1\xe6l\xe3\xde\x1bIk=\xb6\xb4\xb2t\x16^Xy3\xcd/\xac-\xcdF\x18b\x07\xfabS\xef\x90<{\xec2\xfa\x1e\xb4p\x0fH\x81NB$\x7f\x86n\xbf\x88y\xb7U\x1a\xc6\xcd\xc5gk\x0b\xf9x\xd2\xc85=\x05jju u\xaa-\x8fP\xb2\x8fRdE_\xf7\xe1yU2gu\x06\xc6!\xe7\xa8\x17\xedV\x13\xf0+\x86V\xa1\x1c\x9bh\x8a\xd1=>\xd9\x98\x7f\xc4\xa9\x18\x1ci\xc5\x86\x12\x0bWZ+\xc5\xe2\x80Db\x80\xc51\xda(w\x04\x9fp\xda\xf7\xc8\xe8\xbb\x16HI\xa2\x9a\xb0\xe4\x06\x9f"\x9d\x129\xd0\x98\xb0\xd1F\x01\xf0\xdf\xbe\xa9_\xb8\xdb\xbc\x16\xba\xfdQ\x9f\x8c\xc0\xfb\xcfF\xcf\x8b\x17+K\xc7\xde\xb5\x1a\xa0(\x04\x9f\x17\x128\x11(OK\x7f\\#\xbf\xe5x\x0b\xe8c\x99!*^G\xfd8\xe4q$\xa7\x08\xde/\xdd\x92\xb1E\x9bF\x91\x96\xd6\x84\x89\xe0\xf7\xe1nr\xef\x02\x07\x96fl\x12E\xfehY\t\rC\x82dw\xb0~\xcaT\x8a\x93.\xed\xa6\xa0l5u7\x83\xb84PE\x81\xcd\x1b\xe2[\xaa0\x15\x08\x81`Gd\xe0<N\xc7\x8ee\xde\xd4 iw\xca\x9e\xc5\xbb4._\xc5\xeb19z\x10g!\xac\x05\\t\x8b`\x88\xa1\x0f\xdd e\x97NR\x92m`B\x02f\xb5!F\x0e06\xa8\x8eA\xef\x06Vg\x97\x1d\xba\xbcd\x0b\xdf\xc5\xba6\xca\xc0\x86@:\x7f\x88lz\xe5\xfc\xd7\x02Y\x95n(\xe3\x14\x9e\xe5\xe3TJT&\xb1\x96\x81\xfc\xa0\xda\xbc\xa7\xfbF\x82\xdb\x1fM\x93\xe37\x91\x0b%M\xa9\xe3\x9a\xe0&\xe1\xf4w\x1fvf{\x7fb\xd9\x86Qh7\xa3\x0c\x18\xba\x86\x9b\x11"\xbd\x81\xb8\xcc\x92\xd3X\xc8\xe5\xc8<0\x9e\\\xc0\x04\xc7$2\xc3\xb2\x02[\x18\xf3E\xe6\xe6\x99\xeb\xa1\xec$\xffhpE\xf5{\xd1\x8a\xe2S[\xe8\x90\xbc\xdfR\xbb\x8fT0\x07X\x1bk\xbc\xb1x\x04\xd6SB\x95\x1e\x1c\xd9;\x06{8)\xd8\xe2\x8a\xae\xf8\x05s\xd2\x83DD\xb8\x98\x86\x7f\x04\x92\xd6\x9fX\x94\xee\x97\xb3*\xc7\xea\xefZt[\xb6\x11\xd9\xfa6;\xbf\xda\xd0Z\xc6I*\x11\xf0C\xc5\xdb\x12^p\xb9X\xb8\xedW\xcb\xa6\xeb\x9b\xa6\x1f\xdd1X\xe3\x95\x13|6T\xb6n;\xd2_\x0b\x8c\x91}\x0f\x98\xa3g\x16\x8dP\x9bp\x9d/\xb9\xd6\xf1{\x06\x0bH\xf3\x81=\x99w)I\xb0\xff\xa8n\x9a\x0c\x120\x8b\x8d\xca\x8dD\xf1XDhih,\xdc\x94\x86xe\xc4Dp\x91\xdd\xdf\xcd2\x81"\x90+j\xcfN +:\xac\xd6\x87$>SL2\xe8\xf5\xfb\rdb\xb8\xed\xc5\x93,W\xa0\x92?;r\x1b\xddj\x14C]yM\xdf\xddM\xcc\xa0\xc5KcI\x98`\x04q\xf1M\xf42\xfb\xbf\x07l\xc9\xfc\xce\xdbG\xdc|\x15]\xd5a\xfc\xd0\xd7(D\xc2\x8f\xa9\xfd\xf2\xdf\x87\']\x81@\x8dM\x80\xb3\x12\x1f\xc5\x1c+\x89\xa9\x05\x1d\x01\xc5\xf4\x08"\xc8\xb1\xb8R\x9dH\xc5aA\xea!f\xabP\'R\xbc\xb1o\xec5F!NNs\xb4\xa1~\xe1A\xf3\xda\x0f\x7f/\xbab\xd3\xe6\x93n\xa1\x82\x01\xdc\xa2]\x916\xe7\xa6\xce\xcf\x98\x8d{K!\xfd\x8f\xd6\x86U\x88\xee] Q\xbaW1\x97\x13\xa5F]v\xa0\x92\x1c\x85\xf4\xa4a\xa4;0\x80\xa8\xd6\x96\x9d\xbe\x1dl\xb5\xd1\xa5\xc4),\xc9\xd1\\\x9e=\xcb\xc4\x87\xf8=q\x1fvBD:\xabC@\xa0\xab\xd3\xe5ZVF\xda\x0f/a\x0b\xdd\x1e&HK\xfd6\x84\xed/\xe9\x9a[\x8c:\xa9\xdb\x9cu\x8c\x8b\x88\xb2\xc3*`Gb\x10t\xe8g\x91>\x87\xe2?vVG\xd9\xca\xf2\xb5\xdawW$\xe2e\xc4\xba\x02V\xb78\xadZ\x02c\xb8\xaevn~\xe5\xf5\x93\x7f\x08\xb2\nr\xdf\xdf\x068\xd5n\x01\xa9N*9:\x0cd\xa6~#\xfez(Wl\x80\xd3k\xd9\xbffg2U\xce\xcfZ\x1d5\xdc\t\x08;=\x02\x87\x87s\xca\xdeF\xd2\x9a\xff!\xeb\x8f\xaa\xca\xa1\x18N;]8\xbec2\xde\x17@\x97\xcbW\x8c\x90\xcb\xa0]\xce\xe6P\x10\x82\xb1\x14\x9f\xaaf4s\x00\x1b\xe0\xc1E\xd3k\xcd\xb7\x86\x92\xf3\x10\xcd\xbf\xe3o\xcc\xb4#\xcd\xf1\xdb\xf9\xe8"\xae\x05\x15TXo\xc5\x84\xf2\xfe\xb8\xc5\xda\xb4\xf9]\xd6-\xc5\x8f\xe8\x92\xaa\xa2\x87\x81\xc6\x8c\xb9\xd6\xbfV\xb1\x1d\xb7Nt\xcbV1\xc4\xc3m2%W$4\x1aS\xf8\xca\xe2i\xb0t9\xe8\xa8\xc3&\x12\x8f\xa4\xf0$_\x06\xc7\x19q\x98\xc7\xfe\xf4\x9d&ol\x9cR\xd6\xfe\x8e\xc2$4\xf5dU\xb9\xe2\xa0\xc7!\x13G_o\x9b\xc5~F\xdc_Xn|\xc7\nm\xaf\xc5%,\x1c\x90rg\xfc\xab7\xf5]\xee\xfe\xed\xef\xfc\xd3W\xf5\xc5\xb5*=t\xac\xf1\xf6\x1b\xd4k(\xa4e}Y\x1c\\\x83Q\xcf\x99(t-\xee\x1a\xadw\x1d\xda\x96\x05r\x02\n\xaf\x13C\x8269\x19\xc9\xaf\x97um\xd0\xe2\x80seb\xb4\x0e\x1d\xde\x06\xa8y\x89Lx\xa8\x99/\x9f\xad/\x9dm^\xbe\x03KY\xabY\x16\x93\xae\x91\x08g\x1f)|]\xf9\xb7B\xc2\xfa\xc0\xfar\xf7\xce4\xae\xbc\x0f]zI\x8a\x1fz[\x12`}`\x87\xd5D\xea\xc0\xfaM\x1fm\xd8\xb0\xf9\xe0\xda\\QTdO)J^l\xb04\xce\xed\x14\xba`Y\xa5Q\xc4\x87GW\xddD\x8e\xf5\xc7\xb7\xbd\x10(>\x1c\xd1>\x1a3\xe5\xf1c\xdd\xfe\x82~\xa5\xdfI\xad\xa1\xeeN%\x10\x141W\x10\x89\x9b\xa2\x05v\x10-\x8e[\xb80\x98\xf3\x94\x17\xc0\x89\r\xb4F\xd8p\xce\xe5f\xb7\xf8\x1a\xe8\x16W\xbdF\x82\xed\xc6\xc6\x9b\x99\x8ad\x12\xa8\x0b\xe3\xc4\xb9S,p\xe88H\x07\xcb\xd9\x9c\x0b\xa3\xf3{[\xf4\x92\xb9y\x7f\xe1\xb6\xff\xcdi\x0e%c`\x93\xdb*h{R\xdd\xd1,1\xac\xd8b\xc1v\xd0\n4[\xac\x03yNu@\x0b\x1fW\xcb*0\xda\x94\xd5\x90y\xa5r!\xe0\xd9\x81\x83T\xbc\x90?\xb0\xe1\xa0\xe9\xc0*\x16\xfah\xd4\xc4\xc1\xb5\x86\x11\xc8\x02\xe7L&@\x97\'e\x94*\xef\x95|1\xd8\xbb\xd0l-\xd1F{\xbb1\x8bw\xbdrE\x9eJ\x06eM\x8e\xf2\x99\xce\xd6.\xfc\x12\x97\x84\xb5\x16\'\x0c!\xea}\x9dp\xd5\xf2\x80-\x04K\x03\x04:)w\'\x80%b%Z[\xcbZ\xf3Z\xee3\x80\xb9\xdf\xb4\xac\xc0\xaal\xbb\xdb\xab\xc1\x15\xc3\xbd\xd0\x94\xe45\x1eX\xdf\x89\x17\xc4\xa7\x82\x8c\xb3\xb5\xc5\xf3\xd8\x12d\'\x02\xc7\xef\x14CT\xcc)\xc3Ul\xde*\xc4\xce\x91\n]>B\xc7U\xb5\x11A\x9b\xb0\x06\xe4\xffl\xb6v\xed\x06g~\xacu\x03\xfb\x81\xf5\x1ej\xd3\x15\x9c\x95d\xbas\xb8BrD;\x01\xb0\xea\x8d\xd2\xc2\x94\nBi\x16\x1cXE\t\x84v\x0e\xf9\x03\x8cP\x0b\x9c\xd6Yj\t\xf1\xc35\x9d\xe9+6Y8\xc9B\x9c\xe2\x941R\xfb\xe5R\xfdT\xdb\x0b\x19\xb4\xc2\x87I\xab\x03\xee\x91>\xdc+\xe0\xb9\n\xaa\x0f\xad\x8f7lNw\x0e\x8cZ\xfe\xd4\xc9\xe6\xec\xf7k\xd4\x00\xf7\xf2;r\xd3\x87T\xc0s\xc5\xe0z\x92J\xe1\x12\xb1\xa9\x17xQ\xfd\xb9\x1bzUR\x07\x06\x0f\x7f\x80\xc3\xf6csr\x06[\x1c\x8c?\x06\x1e\xb4\xeaJ\x04\xb6d\x1b\xb08w\xac\x82W\x19j\xae/\n\x8dQ\xcc\x11$S\xb8\xa1<\xed\x00f#<\xa4\x92\x94\xb1\xc8e~\x1a%\x1f\x11O\xa0\xb2\x8d\x9f\xf3\x1f\x9fl\x15\x06\xb19\x15k\xc8\xa7\xf8Wj[\xb1m\xcf~&\x18N\x051\xfa\xf2\x95\xab\x05}k\x11\xeb3Q\x81\xfd\xdd\xca\xeb%T#.\xcfs\xe2:G\xe7\x11\xddD/+o\xce\xa3\x93Qg"\xc2T\xef\xcc\xae\xc0\x84\x1c/\xe78\xa5dg*&\xf9\xddtv\x8b;\x02%+\xb7\xfb1\xce4\xf5[\xa9\xe6r$\x80\xe9\x1c\x07\xad\xe9\xf8\xf4\x02\x8fE>e\xe40\x8b\xe4Z>\x8b\xb5\xfcO&u\xde\xa46\x8f\x82t\xc2P\xd6\xb92\xf4\x02\x8b.R3\x81\x9d\xb6\x19r\x92S\x98\xb3\xaf=R\xba\xb2W\xdcl\x94]\xc7\x97X\xaf2]\x8b]\xc5\xef\xb69RAUyp\x94X\x1d\t\x15\x08\x88\xf3Gr\xc4\xd8\x800\xd2(\x91,\xf1j\x9f7ARk,\\\x91\xe3\x9fl\x97c\x95Z+GP3\xe8X\x11\xc3\xae\xf9\x1fy\x94\xa5\xa4D\t"T\x17W)\x14\x89\x9ep\x1a1\x17\xcd\xb4\xdb\x9a\x1b\xb6\x8fR\xaa\'\xcb\xc6p\xc9\xc7\xb0\x9b\x1dt\xe4B\x02\xe3\xa0+\xfa\x833Hi\xc2T\xc6w\xe2\x16\x9c\x0eF&S\x19\xe6\xf2(\xc9\x0f\x86\'\'\x82k\x9e\x85\xb7\x13\xb4K7h)\n!\xc5\x93"B-\xf7\xcdG`\xd5\xa5\x1f\xa0SJ\xc9\xb2y\xb5\xbc\xa4\x08\x10\nc\x11\x16\x9b|\xcey\xf9a\xbeG\xba\x90+\x1dKT3\x823wMf\xf7*h\xe1)\x89\x16\x0b\xc7W^\xddeD`\xfcei\xf6\x1f\x91|\xcfu\x19RtU\xed\xd7\'&\x9a\x83\xcf\x95\x19f0h\xed\xa9\xf8\xf8\xfc\xcbbF\xf9\xb0\x84\x83\x02?\xe2\x8e\xf7\xc0A#\x97\xb7\xabn\xa0.U\xa4-_\x8f\xdc\xc4\xae{cr\xd1j\xc8"\x92\x0b!\xa4\xad\x04^\xb1D\x8d\xb5\xb9\xff\n\x05\xdct+P\x9e,\x91J\x97\x07s\xc5\xfed\xe2\xf7\xdc\xc5\xa9\x14\xe9:]\xa2^\x8di:\x92\x1e\xaa\x11Iu\xa1gJ\x92\xe1)\xef\x8ex+\xb2[\xd2=xe|\xdd\xa7G\xb7x\xab\xcba\x1e\xd7\xbe}\x04\xf8\xd4=\xf8\xb9\xe9\xf4j\x10\xf33\x82l\xd5\xde\xf8\xd4\x13_5\xc9O\xe9\x0e\x8d\x03\xd8\x85\xb1Rd\x13E\xa9l\xd2\x8f\xb4(\x80\xe2Mt\x9e}\x18\xb9-\xa7\x8e\xb5\xf6BC\xef\xd0\xfd;\xfe\xd4\x9d\x10\xe35\xfa\xbc\x11\x98\xd4^\xdb\x10\x19\xfc\xe6\xca\x9b\x93\xb5\xb9\xfb\xb8[SO\xd4\xe5\x19WF\xb4y\xf2;\xde\t\xefwd\x9f\xa8\xab\xe7\xf7\x0b\x85\x1a\xce\x1e\xc009\xdc\xd1\x11B\xf7%\xe2\x81\x16T\xbb\xa84`\xd6\xc5\xcc\xc5\xfa\xc3{\x98\xc1Ld\xc1\x14\xc35\xbel(\xa3\xa6\x13\xa2\x12\xee\xf9\xc2\x85\xdf\xe2\x8e\x94\xb5\x1cr\x8en\xc9\xd9\xf9\xfe\xacM}\x006\xd3\xbf\xc6\xadYyd\xdd\xaa\x02\xca>\x1c4\x80\\\x8f\xa5\xd5\xd4\xcb\x8d\t\x1a6\xeaO\xf6 \x16-`H\x8d\x83*\xc52\x05\xea1C\x17d\xc7\xf59N\xafnL\xfe\x1c}5\xe8s\x8b\xf3\xec\xde\xbfo\xcf\xfe}}\xdb{\xf6\xe2\x15p\xbbw\xefCR\x1e\xc2\xb4q\x8c\xd4\xe1-\xd9\x82\xcd\xd6\xbe\x7f\x91\xd3uT\xb3\x1f\xbe`5r\xb8\xc2\xcd\x90\x82i\xd7\xd4f6T\xcd`\x04\x9a\x18\xc9\x92\x02E\x97\xc2\xe2\x05\xde\xc4o\xf8\x80\x98\x84\x17\xb9\t\xd0?\xf5\x98\xef\xc4R0\xf1z\xfa\xd0\xd5\r\xb80\x10C\xdd}\xa9\xe5\xb2<\xc4\x9b\xd0q\xc7\x9c\xfch\x9f\xd4&\x8f\x98\xdb.\x07K\x91\xdeN\xe1PF5\x95\xdcf\xa73\xb31\xf5o\xb8\xecV\xb8\xea\xcb\x85\xb3*h\x07l\x8a\xa3\xe2\xbe\x86\x9c\xc1\xd6zj\xe3\xde\xc9\xc6wg\xb0\x04\xec\xd4t\x98\x14e\x87\xb6X\xc6\xd2\x18\xa4]|Ie\xe0\xd6T\xcd\xf3=\xac_r\xcaxK:\xba\xccPJSk\x1e\x7fi\x06\xaf\xa4!\xff%\xcf\x1f\xbe\xbe\'\xa6\xcd@W\x87\xd5\t\xa6b\xb4\xca\x93\xe4\'9l\x00\x1dA%e\xd0\xf8\x1f=x\xd5B6h$\x80\x1d\xa2\x1fGV\xaf\x9d5\xb1\xedU*\xb2\xe24\x93d\xdb\xc6T!\x9aR=\x7f\xa4F\\\x90Gg4\xd2\xa1X\xd2~\xc8z\x90\x92|\x90\x1b\x9a\x98\xe2f2L\xcd@\xdd\xc9\x05R\xac\x80)}\xbaWG\xf8f7n8\xba\xb2x\x86m}\xb9Y54\x9d\x1a3n\xc7\xd3(eH\xc2\xa0~h~!\x14L\xfb\xb6y\xc3\xc6\xech\xf8[!bU/\xa5\xfan\x07\xd1]>~\x12@\x92N\xb5Q\xae@,I\xeaFT\xe7m}!\xa4F^\x98)\xc5tq\xe6\x83\xd5!\xc3\x87\xab*9|\xe5Us\x95\x8e\x90M\xde\xc1\xf1H]L\x9c\xb13\x18\xf3r\x86\xec\xc3.\x96(\x8a\xf6*WR\x8c\xb3\x9e\r\xba \'@\xfbo\x1f \xf3\x0f\x97\xdfHw\x03\x96Ct\xa3\x03\xc13\xfa\xeb\x9b9\xfe\xb9\xf5^\nx\x94\xd9cK\r\xafb\x16e\xf1\xbat\xe7Q\xa5\x8aSr\xe7\x87\xd8\x1a\x99\x92\x0e3N.\xc7\xbc\x9f\xa4\x14\xeb\xbcR\xf6\xa45\x03\xae=|v\xcf\xecO\xc9\x99\x9dX\xec\xbd\x88\x19\xdfj\x9d\xd4\xaa\x9f\x99\x9d\xa6\xa5h\xb7Dio+J\x97\xc7\x0e\xfc\xe0\xa2!\xa0\xe4lV_\xaeijjT`F\x00\x8b\x0b\x87\xb93v\x9bFR\xa0\xca2\x84\xdc\xda\x06/Y\xb0\xa2\xc6\xd2\x03]\xd4\xca\x00\x87\xf7#\xf1\xbf\xe7\xbe\xb3\xea\x0f\x1fZ\x9d\x9d\x98u\xa77G\x0f\xcc\xe3q\xef \xf1j\xe3\xdb]\x94\xa3\xd7\x82\x01\xd8\x95\xff\x03\xfc\x00\xec\xb4').decode('utf-8')
REMOTE_ALLOWED_TASKS = ['t2v', 'i2v', 'l2v', 'flf2v', 'r2v']
REMOTE_MAX_IMAGES = 9

'''
Prepare remote-only defaults for tasks not selected manually in the Notebook.
遠端 API 為尚未手動執行的 Task 建立預設參數，不影響 Notebook 既有設定。
'''
TASK_CONFIGS = globals().setdefault('TASK_CONFIGS', {})
for task_name in REMOTE_ALLOWED_TASKS:
    if task_name not in TASK_CONFIGS:
        defaults = dict(prompt='A cinematic landscape with ambient sound.',
                        width=384, height=224, seconds=5, steps=6 if REMOTE_SERVICE_ID.startswith('06_HyperFlow_') else 20,
                        seed=42, quant=QUANT, output_name=f'{task_name}.mp4',
                        first_image='', last_image='', ref_images=[], ref_videos=[], ref_audios=[],
                        video_audio=False, ref_image_size='match')
        if REMOTE_SERVICE_ID.startswith('06_MiniMax_'):
            defaults['audio_shift'] = 6.0
        else:
            defaults['encoder_variant'] = ENCODER_VARIANT
        TASK_CONFIGS[task_name] = defaults

REMOTE_ROOT.mkdir(parents=True, exist_ok=True)
REMOTE_JOBS = globals().setdefault('REMOTE_JOBS', {})
REMOTE_JOB_LOCK = globals().setdefault('REMOTE_JOB_LOCK', threading.Lock())
REMOTE_ENGINE_LOCK = globals().setdefault('GENERATION_LOCK', threading.Lock())
REMOTE_QUEUE = globals().setdefault('REMOTE_QUEUE', queue.Queue(maxsize=REMOTE_QUEUE_CAPACITY))
assert REMOTE_QUEUE.maxsize == REMOTE_QUEUE_CAPACITY, 'FIFO 容量已變更，請先重新啟動 Kernel 及遠端服務。'
REMOTE_UPLOAD_SLOTS = globals().setdefault('REMOTE_UPLOAD_SLOTS', threading.BoundedSemaphore(REMOTE_UPLOAD_CONCURRENCY))
REMOTE_RESULT_TTL = REMOTE_RESULT_TTL_HOURS * 3600
REMOTE_RESULT_BYTES = REMOTE_RESULT_LIMIT_GIB * 1024**3
REMOTE_MIN_FREE_BYTES = REMOTE_MIN_FREE_GIB * 1024**3

'''
Do not start a second GPU job when a long-running backend appears stalled.
當底層生成疑似卡住時，停止接收新工作，絕不啟動第二筆 GPU 推論。
'''
REMOTE_STALL_SECONDS = REMOTE_STALL_HOURS * 3600

'''
Verify the CUDA backend and existing model without loading another model.
確認 CUDA 與已啟動的模型後端，不另外載入第二份模型。
'''
def remote_gpu_status():
    try:
        import torch
        if not torch.cuda.is_available():
            return False, 'Kaggle CUDA GPU 不可用；請啟用 GPU 並重新執行模型初始化 Cell。', []
        qwen = REMOTE_KIND == 'image_multi'
        yue = REMOTE_SERVICE_ID.startswith('06_YuE2_')
        if yue:
            selected_gpus = [globals().get('GPU_ID', 0)]
        elif qwen and globals().get('QWEN21_STATE', {}).get('mode') != 'bf16_dual':
            selected_gpus = [0]
        else:
            selected_gpus = [0, 1]
        available = torch.cuda.device_count()
        if any(not isinstance(index, int) or index < 0 or index >= available for index in selected_gpus):
            return False, f'需要 cuda:{selected_gpus}，但實際只有 {available} 張 CUDA GPU。', []
        devices = [f'cuda:{index} {torch.cuda.get_device_name(index)}' for index in selected_gpus]
        if qwen:
            state = globals().get('QWEN21_STATE', {})
            if state.get('pipeline') is None or state.get('manager') is None:
                return False, 'Qwen Pipeline／GPU 記憶體管理器尚未初始化，請先執行模型初始化 Cell。', devices
            if state.get('device') != 'cuda:0':
                return False, 'Qwen Transformer 沒有設定為 cuda:0。', devices
        elif REMOTE_SERVICE_ID.startswith('06_HeartMuLa_'):
            pipeline = globals().get('pipe')
            if pipeline is None or pipeline.lazy_load or pipeline._mula is None or pipeline._codec is None:
                return False, 'HeartMuLa 雙 GPU Pipeline 尚未完整載入，請先執行模型載入 Cell。', devices
            for name, module, index in (('HeartMuLa', pipeline.mula, 0), ('HeartCodec', pipeline.codec, 1)):
                parameter = next(module.parameters(), None)
                if parameter is None or parameter.device != torch.device(f'cuda:{index}'):
                    return False, f'{name} 未位於指定的 cuda:{index}，請重新載入模型。', devices
        elif yue:
            gpu_id = globals().get('GPU_ID', 0)
            process = globals().get('SERVER_PROCS', {}).get(gpu_id)
            if process is None or process.poll() is not None:
                return False, f'YuE2 CUDA{gpu_id} 常駐服務尚未啟動。', devices
            if f'CUDA{gpu_id}' not in [str(value) for value in process.args]:
                return False, 'YuE2 常駐服務沒有使用預期的 CUDA 啟動參數。', devices
        else:
            base = globals().get('BASE')
            if not base and REMOTE_IS_LTX:
                base = f"http://127.0.0.1:{globals().get('COMFY_PORT', 8188)}"
            if not isinstance(base, str) or not base:
                return False, 'ComfyUI 尚未啟動，請先執行模型服務 Cell。', devices
            with urllib.request.urlopen(base.rstrip('/') + '/system_stats', timeout=4) as response:
                stats = json.load(response)
            comfy_devices = stats.get('devices', [])
            if not any(str(item.get('type', '')).lower().startswith('cuda') or
                       str(item.get('name', '')).lower().startswith('cuda:')
                       for item in comfy_devices if isinstance(item, dict)):
                return False, 'ComfyUI 沒有回報 CUDA 裝置，已拒絕 CPU-only 推論。', devices
        return True, 'CUDA 後端已就緒（不代表所有權重常駐 GPU 記憶體）。', devices
    except Exception as error:
        return False, f'GPU 檢查失敗：{type(error).__name__}: {error}', []

'''
Verify every global required by the selected Notebook generation engine.
驗證所選模型的生成程式所需變數與設定已完成初始化。
'''
def remote_inference_ready(task):
    namespace = globals()
    common = {
        '06_HeartMuLa_': ('ROOT', 'pipe'),
        '06_YuE2_': ('ROOT', 'GPU_ID', 'PORTS', 'SERVER_PROCS'),
        '06_Z-Image-Turbo_': ('ROOT', 'BASE', 'LOG_PATH'),
        '06_Qwen_Image_': ('MODEL_MODE', 'GPU0_MODEL_GIB', 'GPU_ID', 'QWEN21_STATE'),
        '06_LTX-2.5_': ('ROOT', 'ENGINE_PATH', 'COMFY_PORT', 'COMFY_LOG', 'COMFY_PROC',
                          'ltx_available_ram_gib', 'LTX_MIN_AVAILABLE_RAM_GIB', 'importlib'),
        '06_MiniMax_H3_': ('ROOT', 'BASE', 'COMFY', 'LOG', 'PORT', 'TASK_CONFIGS',
                              'H3_DESIRED_CONFIG', 'H3_ACTIVE_CONFIG', 'H3_MODEL_CHOICE',
                              'H3_ACTIVE_MODEL', 'DIT_NAME', 'TEXT_MODE', 'TEXT_GPU',
                              'QUANT_SIZES_GIB', 'ready', 'choices', 'check_choice'),
        '06_HyperFlow_H3_': ('ROOT', 'BASE', 'COMFY', 'LOG', 'PORT', 'TASK_CONFIGS',
                                'H3_DESIRED_CONFIG', 'H3_ACTIVE_CONFIG', 'H3_MODEL_CHOICE',
                                'H3_ACTIVE_MODEL', 'DIT_NAME', 'TEXT_MODE', 'TEXT_GPU',
                                'QUANT_SIZES_GB', 'ready', 'choices', 'ENCODERS',
                                'ENCODER_FILE', 'MODEL_FAMILY', 'WEIGHTS_IN',
                                'TEXT_LOADER_NODE', 'PROJECTION_FILE'),
    }
    required = next((names for prefix, names in common.items()
                     if REMOTE_SERVICE_ID.startswith(prefix)), ())
    missing = [name for name in required if name not in namespace or namespace[name] is None]
    if missing:
        return False, 'Notebook 尚未完成初始化，請先執行對應的模型／服務 Cells；缺少：' + ', '.join(missing)
    if task not in REMOTE_ALLOWED_TASKS:
        return False, '此模型不支援生成模式：' + task
    if REMOTE_SERVICE_ID.startswith(('06_MiniMax_H3_', '06_HyperFlow_H3_')):
        config = namespace['TASK_CONFIGS'].get(task)
        if not isinstance(config, dict):
            return False, f'{task.upper()} 尚未設定；請先執行 Notebook 對應的模式設定 Cell。'
        if namespace['H3_ACTIVE_CONFIG'] != namespace['H3_DESIRED_CONFIG']:
            return False, 'GPU／Offload 設定與已啟動 ComfyUI 不一致，請重新啟動模型服務 Cell。'
        quant = config.get('quant')
        if REMOTE_SERVICE_ID.startswith('06_MiniMax_H3_'):
            variant = 'ref2va_pruned' if task == 'r2v' else 'fl2va_pruned'
            expected = (variant, quant)
            expected_file = f'minimax_h3_{variant}-{quant}.gguf'
        else:
            variant = 'ref2va' if task == 'r2v' else 'fl2va'
            encoder = config.get('encoder_variant')
            expected = (variant, quant, encoder)
            expected_file = f'minimax_h3_{variant}_hyperflow_flash_14b-{quant}.gguf'
            if encoder not in namespace['ENCODERS']:
                return False, f'{task.upper()} 的 Encoder 設定無效，請重新執行模式設定 Cell。'
            if Path(namespace['ENCODER_FILE']).name != namespace['ENCODERS'][encoder][1]:
                return False, 'GPU 1 Encoder 與此模式不一致，請重新執行模型下載／啟動 Cells。'
        if namespace['H3_MODEL_CHOICE'] != expected or namespace['H3_ACTIVE_MODEL'] != expected:
            return False, f'{task.upper()} 尚未載入相應模型權重；請設定模式並重新執行模型下載／啟動 Cells。'
        if namespace['DIT_NAME'] != expected_file:
            return False, 'ComfyUI 使用的模型檔名與任務不一致，請重新載入正確模型。'
    elif REMOTE_SERVICE_ID.startswith('06_Qwen_Image_'):
        state = namespace['QWEN21_STATE']
        if state.get('mode') != namespace['MODEL_MODE']:
            return False, 'Qwen MODEL_MODE 與已初始化的模型模式不同，請重新執行模型初始化 Cell。'
        if state.get('gpu0_model_gib') != namespace['GPU0_MODEL_GIB']:
            return False, 'Qwen GPU0_MODEL_GIB 與目前載入設定不同，請重新初始化 GPU 記憶體管理器。'
    elif REMOTE_IS_LTX:
        if not Path(namespace['ENGINE_PATH']).is_file():
            return False, 'LTX 模型引擎檔案不存在；請先執行模型與 Workflow Cell。'
        if namespace['COMFY_PROC'].poll() is not None:
            return False, 'LTX ComfyUI 行程已停止，請重新執行啟動 ComfyUI Cell。'
    return True, '模型初始化與所選模式已就緒。'

'''
Validate client parameters before they occupy a FIFO slot.
在進入 FIFO 前檢查 client 參數，避免排隊後才因設定錯誤而失敗。
'''
def remote_validate_fields(task, fields):
    required = {'task'}
    if REMOTE_SERVICE_ID.startswith('06_HeartMuLa_'):
        required.update(('lyrics', 'tags', 'duration', 'seed'))
    elif REMOTE_SERVICE_ID.startswith('06_YuE2_'):
        required.update(('style', 'lyrics', 'duration', 'steps', 'lm_seed', 'seed', 'cot'))
    elif REMOTE_KIND == 'image_text':
        required.update(('prompt', 'width', 'height', 'steps', 'seed', 'cfg'))
    elif REMOTE_KIND == 'image_multi':
        required.update(('prompt', 'negative_prompt', 'sampler', 'reference_size',
                         'width', 'height', 'steps', 'seed', 'cfg'))
    elif REMOTE_IS_LTX:
        required.update(('prompt', 'width', 'height', 'seconds', 'seed', 'image_strength'))
    else:
        required.update(('prompt', 'width', 'height', 'seconds', 'steps', 'seed', 'ref_image_size'))
        if REMOTE_SERVICE_ID.startswith('06_MiniMax_H3_'):
            required.add('audio_shift')
    missing = required - fields.keys()
    if missing:
        raise ValueError('請求缺少必要參數：' + ', '.join(sorted(missing)))
    integer_fields = ('seed', 'lm_seed', 'width', 'height', 'steps', 'duration', 'reference_size')
    float_fields = ('seconds', 'cfg', 'image_strength', 'audio_shift')
    numbers = {}
    for key in integer_fields:
        if key in fields:
            raw = fields[key]
            if isinstance(raw, bool) or str(raw).strip() != str(int(raw)):
                raise ValueError(f'{key} 必須是整數')
            numbers[key] = int(raw)
    for key in float_fields:
        if key in fields:
            numbers[key] = float(fields[key])
            if not math.isfinite(numbers[key]):
                raise ValueError(f'{key} 必須是有限數值')
    for key in ('seed', 'lm_seed'):
        if key in numbers and not -1 <= numbers[key] < 2**32:
            raise ValueError(f'{key} 必須介於 -1 至 4294967295')
    for key in ('width', 'height'):
        if key in numbers:
            size = numbers[key]
            divisor = 64 if REMOTE_KIND == 'image_text' else 32
            if size < 1 or size > 4096 or size % divisor:
                raise ValueError(f'{key} 必須介於 1～4096 且為 {divisor} 的倍數')
    for key in ('seconds', 'duration'):
        if key in numbers and not 0 < numbers[key] <= 3600:
            raise ValueError(f'{key} 必須大於 0 且不超過 3600')
    if 'steps' in numbers and not 1 <= numbers['steps'] <= 200:
        raise ValueError('steps 必須介於 1～200')
    if REMOTE_SERVICE_ID.startswith('06_HyperFlow_H3_') and numbers.get('steps') not in (4, 6, 8):
        raise ValueError('HyperFlow steps 只支援 4、6、8')
    if 'image_strength' in numbers and not 0 <= numbers['image_strength'] <= 1:
        raise ValueError('image_strength 必須介於 0～1')
    if 'reference_size' in numbers and not 64 <= numbers['reference_size'] <= 2048:
        raise ValueError('reference_size 必須介於 64～2048')
    if 'audio_shift' in numbers and not 0.01 <= numbers['audio_shift'] <= 100:
        raise ValueError('audio_shift 必須介於 0.01～100')
    if fields.get('ref_image_size', 'match') not in ('match', 'max'):
        raise ValueError('ref_image_size 只支援 match 或 max')
    if REMOTE_KIND in ('video', 'image_text', 'image_multi') and not str(fields.get('prompt', '')).strip():
        raise ValueError('PROMPT 不可為空白')
    if task == 'r2v':
        image_count = int(fields.get('_image_count', 0))
        video_count = int(fields.get('_video_count', 0))
        audio_count = int(fields.get('_audio_count', 0))
        if not (image_count or video_count):
            raise ValueError('R2V 至少需要參考圖片或影片，不能只提供音訊。')
        if image_count > 9 or video_count > 3 or audio_count > 3 or image_count + video_count + audio_count > 12:
            raise ValueError('R2V 最多 9 張圖片、3 段影片、3 段音訊，合計最多 12 個素材。')
        count_by_type = {'Picture': image_count, 'Video': video_count,
                         'Audio': audio_count + (video_count if str(fields.get('video_audio', 'false')).lower() == 'true' else 0)}
        for media, count in count_by_type.items():
            matches = [int(number) for number in re.findall(r'<%s\s+(\d+)>' % media, str(fields['prompt']), re.I)]
            if any(number < 1 or number > count for number in matches):
                raise ValueError(f'R2V Prompt 的 <{media} N> 編號超出已上傳的參考素材數量。')

'''
Report whether the model is ready for CUDA inference.
回報模型是否已具備 CUDA 推論條件。
'''
def remote_backend_ready():
    return remote_gpu_status()[0]

'''
A Python thread cannot safely terminate an already running CUDA inference.
Python thread 無法安全強制終止正在執行的 CUDA 推論。
'''
def remote_stalled():
    with REMOTE_JOB_LOCK:
        return any(record.get('status') == 'running' and
                   time.monotonic() - record.get('_start_clock', time.monotonic()) > REMOTE_STALL_SECONDS
                   for record in REMOTE_JOBS.values())

'''
Remove obsolete input files once the model no longer needs them.
模型不再需要輸入檔案時立即移除，避免參考圖片長期佔用空間。
'''
def remote_remove_inputs(job_id):
    folder = REMOTE_ROOT / job_id
    for file in folder.glob('*'):
        if file.name.startswith('input.') or (file.stem.isdecimal() and file.suffix.lower() in {'.png', '.jpg', '.jpeg', '.webp'}):
            file.unlink(missing_ok=True)

'''
Keep only finished results; never delete active jobs or in-progress downloads.
僅清理已結束工作，絕不刪除排隊、推論或正在下載的檔案。
'''
def remote_cleanup():
    now = time.time()
    with REMOTE_JOB_LOCK:
        finished = [(job_id, record) for job_id, record in REMOTE_JOBS.items()
                    if record.get('status') in ('done', 'error', 'expired') and
                    not record.get('_downloads')]
        finished.sort(key=lambda item: item[1].get('finished_at', now))
        sizes = {}
        for job_id, record in finished:
            folder = REMOTE_ROOT / job_id
            sizes[job_id] = sum(path.stat().st_size for path in folder.glob('**/*') if path.is_file()) if folder.exists() else 0
        total = sum(sizes.values())
        for job_id, record in finished:
            old = now - record.get('finished_at', now) > REMOTE_RESULT_TTL
            if old or total > REMOTE_RESULT_BYTES:
                shutil.rmtree(REMOTE_ROOT / job_id, ignore_errors=True)
                total -= sizes[job_id]
                record.update(status='expired', error='結果已過期或因儲存空間不足而清除，請重新提交。')
                record.pop('_path', None)
                record.pop('_last_frame', None)
            if record['status'] == 'expired' and now - record.get('finished_at', now) > 2 * REMOTE_RESULT_TTL:
                REMOTE_JOBS.pop(job_id, None)

'''
Sweep completed records periodically without interfering with inference.
定期整理已完成資料，不干擾單一 GPU worker 的推論。
'''
def remote_maintenance():
    while True:
        try:
            remote_cleanup()
        except Exception as error:
            print('API cleanup warning:', type(error).__name__, str(error)[:200])
        time.sleep(300)

'''
Serve only the generation API without exposing notebook controls.
只提供生成 API，不公開 Notebook 的管理功能。
'''
class RemoteHandler(BaseHTTPRequestHandler):
    def log_message(self, fmt, *args):
        return

    def send_json(self, code, payload, retry_after=None):
        data = json.dumps(payload, ensure_ascii=False).encode('utf-8')
        self.send_response(code)
        self.send_header('Content-Type', 'application/json; charset=utf-8')
        self.send_header('Content-Length', str(len(data)))
        if retry_after is not None:
            self.send_header('Retry-After', str(retry_after))
        self.end_headers()
        self.wfile.write(data)

    def do_GET(self):
        path = urlsplit(self.path).path
        if path == '/health':
            with REMOTE_JOB_LOCK:
                running = sum(record.get('status') == 'running' for record in REMOTE_JOBS.values())
            gpu_ready, gpu_message, gpu_devices = remote_gpu_status()
            requested_task = parse_qs(urlsplit(self.path).query).get('task', [REMOTE_ALLOWED_TASKS[0]])[-1]
            inference_ready, inference_message = remote_inference_ready(requested_task)
            if gpu_ready and not inference_ready:
                gpu_message = inference_message
            self.send_json(200, {'ok': True, 'api': 'kaggle-generation-bridge-v3',
                                 'service_id': REMOTE_SERVICE_ID, 'task': requested_task,
                                 'backend_ready': gpu_ready and inference_ready, 'gpu_ready': gpu_ready,
                                 'gpu_message': gpu_message, 'gpu_devices': gpu_devices,
                                 'stalled': remote_stalled(), 'running': running,
                                 'waiting': REMOTE_QUEUE.qsize(), 'queue_capacity': REMOTE_QUEUE.maxsize})
            return
        if path == '/schema':
            self.send_json(200, {'kind': REMOTE_KIND, 'tasks': REMOTE_ALLOWED_TASKS,
                                 'fields': REMOTE_FIELDS, 'image_limit': REMOTE_MAX_IMAGES, 'video_limit': 3, 'audio_limit': 3,
                                 'upload_field': 'images' if REMOTE_MAX_IMAGES else None,
                                 'queue_capacity': REMOTE_QUEUE.maxsize,
                                 'model_switching': 'configure in notebook first'})
            return
        pieces = path.strip('/').split('/')
        if pieces[0] != 'jobs' or len(pieces) not in (2, 3):
            self.send_json(404, {'error': 'Not found'})
            return
        job_id = pieces[1]
        with REMOTE_JOB_LOCK:
            record = REMOTE_JOBS.get(job_id)
            item = dict(record) if record else None
            if item and item['status'] == 'running':
                item['stalled'] = time.monotonic() - item.get('_start_clock', time.monotonic()) > REMOTE_STALL_SECONDS
            if item and item['status'] == 'queued':
                waiting = [key for key, value in REMOTE_JOBS.items() if value.get('status') == 'queued']
                if job_id in waiting:
                    item['queue_position'] = waiting.index(job_id) + 1
            if item and len(pieces) == 3 and pieces[2] in ('file', 'last-frame') and item['status'] == 'done':
                record['_downloads'] = record.get('_downloads', 0) + 1
                downloading = True
            else:
                downloading = False
        if item is None:
            self.send_json(404, {'error': 'Unknown job'})
            return
        if len(pieces) == 2:
            for field in tuple(item):
                if field.startswith('_'):
                    item.pop(field)
            self.send_json(200, item)
            return
        if not downloading:
            self.send_json(404, {'error': 'Result not available for this job'})
            return
        try:
            file_path = item.get('_path') if pieces[2] == 'file' else item.get('_last_frame')
            if not file_path or not Path(file_path).is_file():
                self.send_json(404, {'error': 'Output is no longer available'})
                return
            result = Path(file_path)
            self.send_response(200)
            self.send_header('Content-Type', 'application/octet-stream')
            self.send_header('Content-Disposition', f'attachment; filename="{result.name}"')
            self.send_header('Content-Length', str(result.stat().st_size))
            self.end_headers()
            with result.open('rb') as stream:
                shutil.copyfileobj(stream, self.wfile, length=1024 * 1024)
        except (BrokenPipeError, ConnectionResetError):
            pass
        finally:
            with REMOTE_JOB_LOCK:
                record['_downloads'] -= 1

    def do_POST(self):
        if urlsplit(self.path).path != '/generate':
            self.send_json(404, {'error': 'Not found'})
            return
        supplied_id = self.headers.get('X-Request-ID', '').strip()
        try:
            job_id = uuid.UUID(supplied_id).hex if supplied_id else uuid.uuid4().hex
        except ValueError:
            self.send_json(400, {'error': 'X-Request-ID must be a UUID'})
            return
        with REMOTE_JOB_LOCK:
            existing = REMOTE_JOBS.get(job_id)
        if existing:
            self.send_json(202, {'job_id': job_id, 'status_url': f'/jobs/{job_id}',
                                 'download_url': f'/jobs/{job_id}/file', 'reused': True})
            return
        if remote_stalled():
            self.send_json(503, {'error': 'GPU worker appears stalled; inspect the Notebook before accepting more jobs'}, 60)
            return
        gpu_ready, gpu_message, _ = remote_gpu_status()
        if not gpu_ready:
            self.send_json(409, {'error': gpu_message})
            return
        if shutil.disk_usage(REMOTE_ROOT).free < REMOTE_MIN_FREE_BYTES:
            self.send_json(507, {'error': 'Kaggle disk space is low; clean old outputs before retrying'})
            return
        if not REMOTE_UPLOAD_SLOTS.acquire(timeout=1):
            self.send_json(503, {'error': 'Upload slots busy; retry later'}, 5)
            return
        staging = None
        try:
            self.connection.settimeout(120)
            length = int(self.headers.get('Content-Length', '0'))
            if not 0 < length <= 120 * 1024 * 1024:
                raise ValueError('Request must be between 1 byte and 120 MiB')
            body = self.rfile.read(length)
            if len(body) != length:
                raise ValueError('Incomplete request body')
            content_type = self.headers.get('Content-Type', '')
            fields, files, videos, audios = {}, [], [], []
            if content_type.startswith('application/json'):
                fields = json.loads(body)
            elif content_type.startswith('application/x-www-form-urlencoded'):
                fields = {key: values[-1] for key, values in parse_qs(body.decode('utf-8')).items()}
            elif content_type.startswith('multipart/form-data'):
                message = BytesParser(policy=policy.default).parsebytes(
                    b'Content-Type: ' + content_type.encode() + b'\r\nMIME-Version: 1.0\r\n\r\n' + body)
                for part in message.iter_parts():
                    name = part.get_param('name', header='content-disposition')
                    data = part.get_payload(decode=True)
                    if name in ('images', 'videos', 'audios'):
                        collection = {'images': files, 'videos': videos, 'audios': audios}[name]
                        collection.append((part.get_filename() or 'upload', data))
                    elif name:
                        fields[name] = data.decode('utf-8')
            else:
                raise ValueError('Use application/json, form data, or multipart/form-data')
            if not isinstance(fields, dict):
                raise ValueError('Parameters must be a JSON object')
            if 'abc' in fields and len(str(fields['abc']).encode('utf-8')) > 1024 * 1024:
                raise ValueError('ABC score must be 1 MiB or smaller')
            task = str(fields.get('task', REMOTE_ALLOWED_TASKS[0])).lower()
            if task not in REMOTE_ALLOWED_TASKS:
                raise ValueError('Unsupported task: ' + task)
            if len(files) > REMOTE_MAX_IMAGES or len(videos) > 3 or len(audios) > 3:
                raise ValueError(f'Only {REMOTE_MAX_IMAGES} input images are supported')
            if REMOTE_KIND in ('music', 'image_text') and (files or videos or audios):
                raise ValueError('This model does not accept reference images')
            if REMOTE_KIND == 'video':
                minimum, maximum = {'t2v': (0, 0), 'i2v': (1, 1), 'l2v': (1, 1), 'flf2v': (2, 2),
                                    'r2v': (0, REMOTE_MAX_IMAGES)}[task] if task == 'r2v' else {
                                    't2v': (0, 0), 'i2v': (1, 1), 'l2v': (1, 1), 'flf2v': (2, 2)}[task]
                if not minimum <= len(files) <= maximum:
                    raise ValueError('Input image count does not match the selected video task')
            if task != 'r2v' and (videos or audios):
                raise ValueError('Reference videos/audio are supported only for R2V')
            if task == 'r2v' and not (files or videos):
                raise ValueError('R2V requires an image or a video; audio alone is not supported')
            if REMOTE_KIND == 'image_multi':
                minimum, maximum = {'text': (0, 0), 'edit': (1, 1), 'multi': (2, REMOTE_MAX_IMAGES)}[task]
                if not minimum <= len(files) <= maximum:
                    raise ValueError(f'Task {task} requires {minimum}–{maximum} images')
            for key in fields:
                if key not in REMOTE_FIELDS:
                    raise ValueError('Unsupported parameter: ' + key)
            remote_validate_fields(task, dict(fields, _image_count=len(files), _video_count=len(videos), _audio_count=len(audios)))
            if 'quant' in fields and fields['quant']:
                active_model = globals().get('H3_ACTIVE_MODEL', ())
                if fields['quant'] not in active_model:
                    self.send_json(409, {'error': '指定的量化尚未載入；請先在 Notebook 下載並預檢該 GGUF。'})
                    return
            inference_ready, inference_message = remote_inference_ready(task)
            if not inference_ready:
                self.send_json(409, {'error': inference_message})
                return
            for filename, data in files:
                if not isinstance(data, bytes) or len(data) > 20 * 1024 * 1024:
                    raise ValueError('Each image must be 20 MiB or smaller')
                if Path(filename).suffix.lower() not in {'.png', '.jpg', '.jpeg', '.webp'}:
                    raise ValueError('Only PNG/JPEG/WebP images are supported')
                from PIL import Image
                with Image.open(io.BytesIO(data)) as photo:
                    photo.verify()
            if str(fields.get('video_audio', 'false')).lower() not in ('true', 'false'):
                raise ValueError('video_audio must be true or false')
            for filename, data in videos:
                if not data or len(data) > 80 * 1024 * 1024 or Path(filename).suffix.lower() not in ('.mp4', '.mov', '.webm'):
                    raise ValueError('Reference video must be MP4/MOV/WebM and at most 80 MiB')
            for filename, data in audios:
                if not data or len(data) > 30 * 1024 * 1024 or Path(filename).suffix.lower() not in ('.mp3', '.wav', '.flac', '.ogg', '.m4a'):
                    raise ValueError('Reference audio must be an audio file at most 30 MiB')
            staging = Path(tempfile.mkdtemp(prefix='pending_', dir=REMOTE_ROOT))
            image_names, video_names, audio_names = [], [], []
            for index, (filename, data) in enumerate(files, 1):
                target = staging / f'{index:03d}{Path(filename).suffix.lower()}'
                target.write_bytes(data)
                image_names.append(target.name)
            for kind, uploaded, names in (('video', videos, video_names), ('audio', audios, audio_names)):
                for index, (filename, data) in enumerate(uploaded, 1):
                    target = staging / f'{kind}_{index:03d}{Path(filename).suffix.lower()}'
                    target.write_bytes(data)
                    names.append(target.name)

            '''
            Commit the validated upload and queue entry in one critical section.
            在同一個鎖定區段確認請求、保存資料並加入工作佇列。
            '''
            with REMOTE_JOB_LOCK:
                existing = REMOTE_JOBS.get(job_id)
                full = REMOTE_QUEUE.full()
                if not existing and not full:
                    folder = REMOTE_ROOT / job_id
                    staging.rename(folder)
                    staging = None
                    image_paths = [str(folder / name) for name in image_names]
                    fields['_ref_videos'] = [str(folder / name) for name in video_names]
                    fields['_ref_audios'] = [str(folder / name) for name in audio_names]
                    REMOTE_JOBS[job_id] = {'id': job_id, 'task': task, 'status': 'queued',
                                           'created_at': time.time()}
                    REMOTE_QUEUE.put_nowait((job_id, fields, image_paths))
                    position = REMOTE_QUEUE.qsize()
            if existing:
                self.send_json(202, {'job_id': job_id, 'status_url': f'/jobs/{job_id}',
                                     'download_url': f'/jobs/{job_id}/file', 'reused': True})
            elif full:
                self.send_json(429, {'error': 'Queue is full; retry later'}, 15)
            else:
                self.send_json(202, {'job_id': job_id, 'status_url': f'/jobs/{job_id}',
                                     'download_url': f'/jobs/{job_id}/file', 'queue_position': position})
        except (ValueError, KeyError, TypeError, json.JSONDecodeError) as error:
            self.send_json(400, {'error': str(error)})
        except (TimeoutError, ConnectionError, OSError) as error:
            self.send_json(503, {'error': f'Upload temporarily failed: {type(error).__name__}'}, 10)
        except Exception as error:
            self.send_json(500, {'error': type(error).__name__ + ': ' + str(error)[:250]})
        finally:
            if staging is not None:
                shutil.rmtree(staging, ignore_errors=True)
            REMOTE_UPLOAD_SLOTS.release()

'''
Convert numeric HTTP form fields into explicitly supported generator settings.
只轉換白名單中的 HTTP 參數，不執行客戶端提供的任何程式碼。
'''
def remote_generate(job_id, fields, image_paths):
    with REMOTE_ENGINE_LOCK:
        with REMOTE_JOB_LOCK:
            REMOTE_JOBS[job_id].update(status='running', _start_clock=time.monotonic())
        namespace = globals()
        keys = ['TASK', 'PROMPT', 'WIDTH', 'HEIGHT', 'SEED', 'LM_SEED', 'SECONDS', 'STEPS', 'CFG',
                'LYRICS', 'TAGS', 'STYLE', 'DURATION', 'GPU_ID', 'COT', 'ABC_PATH',
                'OUTPUT_NAME', 'REFERENCE_PATHS', 'ACTIVE_JOB', 'TASK_CONFIGS',
                'R2V_REF_DIR', 'NEGATIVE_PROMPT', 'SAMPLER', 'REFERENCE_SIZE',
                'OUTPUTS', 'output', 'OUTPUT_PATH', 'OUTPUT_MP4', 'LAST_OUTPUT', 'RUN_SEED',
                'QUANT', 'ENCODER_VARIANT']
        before = {key: copy.deepcopy(namespace.get(key)) for key in keys if key in namespace}
        missing = set(keys) - set(before)
        try:
            task = fields.get('task', REMOTE_ALLOWED_TASKS[0]).lower()
            numerics = {'seed': int, 'lm_seed': int, 'width': int, 'height': int, 'steps': int,
                        'seconds': float, 'cfg': float, 'duration': int,
                        'image_strength': float, 'reference_size': int, 'audio_shift': float}
            inputs = {}
            for key, value in fields.items():
                inputs[key] = numerics[key](value) if key in numerics else value
            for seed_key in ('seed', 'lm_seed'):
                if seed_key in inputs:
                    if not -1 <= inputs[seed_key] < 2**32:
                        raise ValueError(f'{seed_key} must be -1 or between 0 and 4294967295')
                    if inputs[seed_key] == -1:
                        inputs[seed_key] = secrets.randbelow(2**32)
            with REMOTE_JOB_LOCK:
                for seed_key in ('seed', 'lm_seed'):
                    if seed_key in inputs:
                        REMOTE_JOBS[job_id][seed_key + '_used'] = inputs[seed_key]
            for dimension in ('width', 'height'):
                if dimension in inputs and not 0 < inputs[dimension] <= 4096:
                    raise ValueError('Width and height must be between 1 and 4096')
            for key in ('seconds', 'duration'):
                if key in inputs and not 0 < inputs[key] <= 3600:
                    raise ValueError('Duration must be between 0 and 3600 seconds')
            if 'reference_size' in inputs and not 64 <= inputs['reference_size'] <= 2048:
                raise ValueError('Reference size must be between 64 and 2048')
            if 'ref_image_size' in inputs and inputs['ref_image_size'] not in ('match', 'max'):
                raise ValueError('Reference image mode must be match or max')
            if 'image_strength' in inputs and not 0 <= inputs['image_strength'] <= 1:
                raise ValueError('Image strength must be between 0 and 1')
            if 'steps' in inputs and not 1 <= inputs['steps'] <= 200:
                raise ValueError('Steps must be between 1 and 200')
            if REMOTE_KIND == 'video':
                if 'TASK_CONFIGS' in namespace:
                    if task not in namespace['TASK_CONFIGS']:
                        raise ValueError('Run this task configuration cell before API generation')
                    updated = copy.deepcopy(namespace['TASK_CONFIGS'][task])
                    for field in ('prompt', 'width', 'height', 'seconds', 'steps', 'seed',
                                  'audio_shift', 'ref_image_size', 'quant'):
                        if field in inputs:
                            updated[field] = inputs[field]
                    updated['output_name'] = f'remote_{job_id[:12]}.mp4'
                    updated['quant'] = inputs.get('quant') or namespace.get('QUANT', updated.get('quant'))
                    if task == 'r2v':
                        updated['video_audio'] = str(inputs.get('video_audio', 'false')).lower() == 'true'
                        updated['ref_videos'] = inputs.get('_ref_videos', [])
                        updated['ref_audios'] = inputs.get('_ref_audios', [])
                    if task == 'i2v':
                        updated['first_image'] = image_paths[0]
                    elif task == 'l2v':
                        updated['last_image'] = image_paths[0]
                    elif task == 'flf2v':
                        updated['first_image'], updated['last_image'] = image_paths
                    elif task == 'r2v':
                        updated['ref_images'] = image_paths
                        namespace['R2V_REF_DIR'] = REMOTE_ROOT / job_id
                    namespace['TASK_CONFIGS'][task] = updated
                    namespace['TASK'] = task
                else:
                    namespace['ACTIVE_JOB'] = {'mode': task, 'prompt': inputs.get('prompt', namespace.get('PROMPT', '')),
                                               'run_name': f'remote_{job_id[:12]}',
                                               'first_path': image_paths[0] if image_paths else None,
                                               'last_path': image_paths[1] if len(image_paths) > 1 else None}
                    for key in ('width', 'height', 'seconds', 'seed', 'image_strength'):
                        if key in inputs:
                            namespace[key.upper()] = inputs[key]
            else:
                for key in ('prompt', 'width', 'height', 'seed', 'lm_seed', 'steps', 'cfg',
                            'lyrics', 'tags', 'style', 'duration', 'negative_prompt',
                            'sampler', 'reference_size', 'cot'):
                    if key in inputs:
                        namespace[key.upper()] = inputs[key]
                if REMOTE_KIND == 'image_multi':
                    namespace['TASK'] = task
                    namespace['REFERENCE_PATHS'] = image_paths
                if REMOTE_KIND == 'music':
                    namespace['OUTPUT_NAME'] = f'remote_{job_id[:12]}.mp3'
                    if 'ABC_PATH' in namespace:
                        namespace['ABC_PATH'] = ''
                    if 'abc' in inputs:
                        score = (REMOTE_ROOT / job_id / 'input.abc')
                        score.write_text(str(inputs['abc']), encoding='utf-8')
                        namespace['ABC_PATH'] = str(score)
                if REMOTE_KIND in ('image_multi', 'image_text'):
                    namespace['OUTPUT_NAME'] = f'remote_{job_id[:12]}.png'
            for output_key in ('OUTPUTS', 'output', 'OUTPUT_PATH', 'OUTPUT_MP4', 'LAST_OUTPUT'):
                namespace.pop(output_key, None)
            '''
            Recheck the CUDA backend after queuing to prevent CPU-only generation.
            工作排入佇列後再次確認 CUDA，避免後端異常時改走純 CPU 生成。
            '''
            gpu_ready, gpu_message, _ = remote_gpu_status()
            if not gpu_ready:
                raise RuntimeError(gpu_message)
            inference_ready, inference_message = remote_inference_ready(task)
            if not inference_ready:
                raise RuntimeError(inference_message)
            exec(compile(REMOTE_SCRIPT, '<notebook-generation-cell>', 'exec'), namespace)
            paths = []
            if REMOTE_KIND == 'video':
                paths = [namespace.get('LAST_OUTPUT') or namespace.get('OUTPUT_MP4')]
            elif REMOTE_KIND == 'image_text':
                paths = [namespace.get('OUTPUT_PATH')]
            elif REMOTE_KIND == 'image_multi':
                paths = [namespace.get('output')]
            elif REMOTE_KIND == 'music':
                paths = namespace.get('OUTPUTS', []) if 'OUTPUTS' in namespace else [namespace.get('output')]
            results = [Path(path) for path in paths if path is not None]
            if REMOTE_KIND == 'music' and REMOTE_SERVICE_ID.startswith('06_YuE2_'):
                results += [sidecar for path in list(results)
                            for ext in ('.json', '.abc')
                            if (sidecar := path.with_suffix(ext)).is_file()]
            if not results or any(not path.is_file() for path in results):
                raise FileNotFoundError('Generation finished without a saved result file')
            folder = REMOTE_ROOT / job_id
            if len(results) == 1:
                target = folder / f'result{results[0].suffix.lower()}'
                shutil.copy2(results[0], target)
            else:
                target = folder / 'results.zip'
                with zipfile.ZipFile(target, 'w', compression=zipfile.ZIP_STORED) as archive:
                    for result in results:
                        archive.write(result, arcname=result.name)
            last_frame = None
            if REMOTE_KIND == 'video' and shutil.which('ffprobe') and shutil.which('ffmpeg'):
                try:
                    probe = subprocess.run(['ffprobe', '-v', 'error', '-count_frames',
                                            '-select_streams', 'v:0', '-show_entries',
                                            'stream=nb_read_frames', '-of', 'json', str(target)],
                                           check=True, text=True, capture_output=True, timeout=120)
                    index = int(json.loads(probe.stdout)['streams'][0]['nb_read_frames']) - 1
                    if index >= 0:
                        last_frame = folder / 'last_frame.png'
                        subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', str(target),
                                        '-vf', rf'select=eq(n\,{index})', '-frames:v', '1',
                                        str(last_frame)], check=True)
                except (OSError, ValueError, KeyError, subprocess.CalledProcessError, subprocess.TimeoutExpired):
                    last_frame = None
            for original in results:
                if original.name.startswith(f'remote_{job_id[:12]}') and original != target:
                    try:
                        original.unlink(missing_ok=True)
                    except OSError:
                        pass
            comfy = globals().get('COMFY')
            if comfy is not None and (Path(comfy) / 'output').exists():
                for artifact in (Path(comfy) / 'output').rglob(f'remote_{job_id[:12]}*'):
                    if artifact.is_file():
                        try:
                            artifact.unlink()
                        except OSError:
                            pass
            with REMOTE_JOB_LOCK:
                REMOTE_JOBS[job_id].update(status='done', filename=target.name,
                                           bytes=target.stat().st_size, _path=str(target),
                                           last_frame_available=bool(last_frame),
                                           _last_frame=str(last_frame) if last_frame else None)
        except Exception as error:
            with REMOTE_JOB_LOCK:
                REMOTE_JOBS[job_id].update(status='error', error=f'{type(error).__name__}: {error}'[:2000])
        finally:
            for key in missing:
                namespace.pop(key, None)
            namespace.update(before)

'''
Use one GPU worker, regardless of the number of HTTP request threads.
不論有多少 HTTP 連線，都只有一個 GPU worker 消費佇列。
'''
def remote_worker():
    while True:
        job_id, fields, image_paths = REMOTE_QUEUE.get()
        try:
            remote_generate(job_id, fields, image_paths)
        except Exception as error:
            with REMOTE_JOB_LOCK:
                if job_id in REMOTE_JOBS:
                    REMOTE_JOBS[job_id].update(status='error', error=f'{type(error).__name__}: {error}'[:2000])
        finally:
            with REMOTE_JOB_LOCK:
                if job_id in REMOTE_JOBS:
                    REMOTE_JOBS[job_id]['finished_at'] = time.time()
            try:
                remote_remove_inputs(job_id)
            except OSError as error:
                print('API input cleanup warning:', str(error)[:180])
            finally:
                REMOTE_QUEUE.task_done()

if not (globals().get('REMOTE_WORKER') and REMOTE_WORKER.is_alive()):
    REMOTE_WORKER = threading.Thread(target=remote_worker, daemon=True)
    REMOTE_WORKER.start()

if not (globals().get('REMOTE_MAINTENANCE') and REMOTE_MAINTENANCE.is_alive()):
    REMOTE_MAINTENANCE = threading.Thread(target=remote_maintenance, daemon=True)
    REMOTE_MAINTENANCE.start()

old_tunnel = globals().get('REMOTE_NGROK_PROC')
if old_tunnel is not None and old_tunnel.poll() is None:
    old_tunnel.terminate()
    try:
        old_tunnel.wait(timeout=5)
    except subprocess.TimeoutExpired:
        old_tunnel.kill()
        old_tunnel.wait(timeout=5)

NGROK_URL = ''

'''
Install the official ngrok agent only if no usable binary exists.
系統尚無 ngrok 才下載官方執行檔，不重複安裝。
'''
ngrok_bin = shutil.which('ngrok') or str(ROOT / 'ngrok')
if not Path(ngrok_bin).is_file():
    url = 'https://bin.ngrok.com/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz'
    with urllib.request.urlopen(url, timeout=90) as response:
        blob = response.read()
    with tarfile.open(fileobj=io.BytesIO(blob), mode='r:gz') as archive:
        entry = next((member for member in archive.getmembers() if Path(member.name).name == 'ngrok' and member.isfile()), None)
        if entry is None:
            raise RuntimeError('ngrok 官方下載檔案缺少執行檔')
        Path(ngrok_bin).write_bytes(archive.extractfile(entry).read())
    Path(ngrok_bin).chmod(0o755)

ngrok_token = getpass('輸入 ngrok Authtoken：').strip()
if not ngrok_token:
    raise ValueError('未輸入 ngrok Authtoken')

result = subprocess.run([ngrok_bin, 'config', 'add-authtoken', ngrok_token], capture_output=True, text=True)
if result.returncode:
    raise RuntimeError('ngrok token 設定失敗（請確認 Token 是否有效）')

del ngrok_token
result = subprocess.run([ngrok_bin, 'config', 'check'], capture_output=True, text=True)
if result.returncode:
    raise RuntimeError('ngrok 設定格式檢查失敗：' + result.stderr[-500:])

server = globals().get('REMOTE_SERVER')
server_thread = globals().get('REMOTE_SERVER_THREAD')
server_is_current = (
    server is not None
    and server_thread is not None
    and server_thread.is_alive()
    and getattr(server, 'notebook_id', None) == REMOTE_SERVICE_ID
)
if server is not None and not server_is_current:
    if server_thread is not None and server_thread.is_alive():
        server.shutdown()
    server.server_close()

if not server_is_current:
    try:
        REMOTE_SERVER = ThreadingHTTPServer(('127.0.0.1', REMOTE_PORT), RemoteHandler)
    except OSError as error:
        raise RuntimeError(f'本機 API Port {REMOTE_PORT} 已被其他服務占用') from error
    REMOTE_SERVER.notebook_id = REMOTE_SERVICE_ID
    REMOTE_SERVER.daemon_threads = True
    REMOTE_SERVER_THREAD = threading.Thread(target=REMOTE_SERVER.serve_forever, daemon=True)
    REMOTE_SERVER_THREAD.start()
    print('已啟動新的本機 API Server。')
else:
    REMOTE_SERVER.RequestHandlerClass = RemoteHandler
    print('已沿用本機 API Server；現有生成工作與結果保持不變。')

with urllib.request.urlopen(f'http://127.0.0.1:{REMOTE_PORT}/health', timeout=4) as response:
    if json.load(response).get('ok') is not True:
        raise RuntimeError('本機 API Server 尚未就緒')

'''
Use the agent inspection API to obtain the actual public URL.
透過 ngrok 本機檢查 API 取得實際對外 HTTPS 網址。
'''
NGROK_LOG = REMOTE_ROOT / 'ngrok.log'
with NGROK_LOG.open('w') as log:
    REMOTE_NGROK_PROC = subprocess.Popen([ngrok_bin, 'http', str(REMOTE_PORT)], stdout=log,
                                         stderr=subprocess.STDOUT, start_new_session=True)

NGROK_URL = ''
for attempt in range(25):
    if REMOTE_NGROK_PROC.poll() is not None:
        break
    try:
        with urllib.request.urlopen('http://127.0.0.1:4040/api/tunnels', timeout=2) as response:
            tunnels = json.load(response).get('tunnels', [])
        NGROK_URL = next((item['public_url'] for item in tunnels
                          if item.get('public_url', '').startswith('https://') and
                          str(REMOTE_PORT) in item.get('config', {}).get('addr', '')), '')
        if NGROK_URL:
            break
    except (OSError, ValueError, KeyError):
        pass
    time.sleep(1)

if not NGROK_URL:
    raise RuntimeError('ngrok URL 建立失敗：\n' + NGROK_LOG.read_text(errors='replace')[-2500:])

print('ngrok HTTPS URL:', NGROK_URL)
print('提醒：公開的 ngrok API URL 不需要額外驗證，請只分享給授課學生。')
print('GET /health | GET /schema | POST /generate | GET /jobs/{id} | GET /jobs/{id}/file')
print('影片輸出另提供：GET /jobs/{id}/last-frame')
print('✓ 第 14 節完成：選用 FIFO／ngrok API 已啟動。')


15. （選用）遠端 API 與 client 使用說明


In [ ]:
'''
Query the active bridge for supported parameters and task names.
從正在執行的橋接服務讀取實際支援的參數與模式。
'''
import json
import urllib.request
request = urllib.request.Request(f'http://127.0.0.1:{REMOTE_PORT}/schema')
with urllib.request.urlopen(request, timeout=5) as response:
    schema = json.load(response)

print(json.dumps(schema, ensure_ascii=False, indent=2))
print('用法：POST NGROK_URL + "/generate"（JSON 或 multipart/images），取得 job_id；')
print('輪詢 GET /jobs/{job_id}；完成後下載 GET /jobs/{job_id}/file。')
print('本機 requests 範例：06_HyperFlow_H3_Flash_影片生成_client.py（與本 Notebook 同名對應）。')
print('使用前只需在對應的 client.py 填入 API_URL。')
print('本服務使用單一 FIFO worker，遠端請求不會同時啟動多筆 GPU 推論。')
print('✓ 第 15 節完成：API Schema 和 client 說明已顯示。')
